<a href="https://colab.research.google.com/github/H-strangeone/minicp/blob/master/CUDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Check CUDA compiler availability
!nvcc --version
!nvidia-smi


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
Sun Nov  2 13:53:13 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8       

In [ ]:
!ls /dev | grep nvidia


nvidia0
nvidia-caps
nvidiactl
nvidia-uvm
nvidia-uvm-tools


In [ ]:
!find /usr -name nvidia-smi


/usr/bin/nvidia-smi


In [ ]:
!/usr/bin/nvidia-smi


Sun Nov  2 13:53:14 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvidia-smi
!nvcc --version

Sun Nov  2 13:53:14 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import requests

url = "http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/kroA200.tsp"
response = requests.get(url, timeout=10)
with open("kroA200.tsp", "wb") as f:
    f.write(response.content)
print(f"Downloaded file size: {len(response.content)} bytes")


Downloaded file size: 153 bytes


In [ ]:
from google.colab import files
files.upload()  # Click "Choose Files" and select kroA200.tsp


Saving kroA200.tsp to kroA200 (1).tsp


{'kroA200 (1).tsp': b'<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01//EN" "http://www.w3.o../html4/strict.dtd">\n<html>\n<head>\n<title>Teaching</title>\n <meta http-equiv="Content-Type" content="text/html; charset=iso-8859-1">\n <meta name="keywords" content="">\n <meta name="description" content="">\n <meta http-equiv="Content-Language" content="de">\n <meta name="author" content="">\n <meta name="organization-name" content="">\n <meta name="organization-email" content="">\n <meta name="city" content="Heidelberg">\n <meta name="country" content="Germany - Deutschland">\n <meta name="language" content="German, Deutsch, de, at,ch">\n <meta name="robots" content="index">\n <meta name="robots" content="follow">\n <meta name="revisit-after" content="3 days">\n <meta http-equiv="imagetoolbar" content="no">\n <meta name="MSSmartTagsPreventParsing" content="true">\n <link rel="shortcut icon" href="../images/favicon.ico">\n <link rel="stylesheet" media="screen" type="text/css" href="../../../im

In [ ]:
!ls -lh /content


total 20K
drwx------ 5 root root 4.0K Nov  2 13:55 drive
-rw-r--r-- 1 root root 5.1K Nov  2 14:13 main.cu
drwxr-xr-x 1 root root 4.0K Oct 30 13:36 sample_data
drwxr-xr-x 3 root root 4.0K Nov  2 14:10 tsplib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
!git clone https://github.com/mastqe/tsplib.git





Cloning into 'tsplib'...
remote: Enumerating objects: 124, done.
remote: Total 124 (delta 0), reused 0 (delta 0), pack-reused 124 (from 1)
Receiving objects: 100% (124/124), 2.02 MiB | 5.33 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [ ]:
%%writefile main_hybrid3.cu
// main_hybrid2.cu (Final, Colab-T4 Fast-Mode)
// GPU: generate many tours with soft KNN bias
// CPU: short LK-lite + 3-opt + time-bounded ILS
// Compile: nvcc -O3 -arch=sm_75 main_hybrid2.cu -o tsp_hybrid2 -std=c++17
// Run:     ./tsp_hybrid2 tsplib/kroA200.tsp

#include <bits/stdc++.h>
#include <curand_kernel.h>
#include <thread>
using namespace std;

#define USE_TSPLIB_EUC2D 1

struct City { double x, y; };

__host__ __device__ inline double dist_city(const City* c, int a, int b){
    double dx = c[a].x - c[b].x, dy = c[a].y - c[b].y;
#if USE_TSPLIB_EUC2D
    return (double) llround(sqrt(dx*dx + dy*dy));
#else
    return sqrt(dx*dx + dy*dy);
#endif
}

__device__ inline double dev_dist(const City* c, int a, int b){ return dist_city(c,a,b); }

inline double host_dist(const vector<City>& c, int a, int b){
    double dx=c[a].x-c[b].x, dy=c[a].y-c[b].y;
#if USE_TSPLIB_EUC2D
    return (double) llround(sqrt(dx*dx + dy*dy));
#else
    return sqrt(dx*dx + dy*dy);
#endif
}

inline void gpuAssert(cudaError_t code, const char *file, int line){
    if(code != cudaSuccess){
        fprintf(stderr,"GPU Error %s %d: %s\n", file, line, cudaGetErrorString(code));
        exit(1);
    }
}
#define gpuCheck(ans) gpuAssert((ans), __FILE__, __LINE__)

__global__ void initRNG(curandState* s, int walkers, unsigned long seed){
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if(id < walkers) curand_init(seed, id, 0, &s[id]);
}

__device__ void reverse_seg(int* T,int n,int a,int b){
    if(a<=b){ for(int i=a,j=b;i<j;i++,j--){ int t=T[i];T[i]=T[j];T[j]=t;} }
    else{
        int len=(b+n-a+1)%n;
        for(int k=0;k<len/2;k++){
            int i=(a+k)%n, j=(a+len-1-k)%n;
            int t=T[i];T[i]=T[j];T[j]=t;
        }
    }
}

__device__ int pos_in_tour(const int* T,int n,int c){
    for(int i=0;i<n;i++) if(T[i]==c) return i;
    return -1;
}

__device__ double tour_len(const City* C,const int*T,int n){
    double s=0;
    for(int i=0;i<n;i++) s+=dev_dist(C,T[i],T[(i+1)%n]);
    return s;
}

__global__ void gen_kernel(const City*C,int n,const int*knn,int k,
                           int*tours,float*costs,curandState*rng,
                           int walkers,int shakes,int nudges){
    int id=blockIdx.x*blockDim.x+threadIdx.x;
    if(id>=walkers) return;

    curandState st=rng[id];
    int* T=tours+id*n;

    for(int i=0;i<n;i++) T[i]=i;
    for(int i=n-1;i>0;i--){
        int j=(int)(curand_uniform(&st)*(i+1));
        int t=T[i];T[i]=T[j];T[j]=t;
    }
    for(int s2=0;s2<shakes;s2++){
        int a=curand(&st)%n,b=curand(&st)%n;
        if(a!=b) reverse_seg(T,n,min(a,b),max(a,b));
    }
    for(int it=0;it<nudges;it++){
        int i=curand(&st)%n;
        int A=T[i],B=T[(i+1)%n];
        double AB=dev_dist(C,A,B);
        int Cc=knn[A*k+(curand(&st)%k)];
        int j=pos_in_tour(T,n,Cc);
        if(j<0||j==i||(j+1)%n==i) continue;
        int D=T[(j+1)%n];
        double delta=(dev_dist(C,A,Cc)+dev_dist(C,B,D))-(AB+dev_dist(C,Cc,D));
        if(delta<0) reverse_seg(T,n,(i+1)%n,j);
    }
    costs[id]=tour_len(C,T,n);
    rng[id]=st;
}

vector<City> readTSPLIB(const string& f){
    ifstream in(f);
    string line; bool ok=false;
    vector<City> v;
    while(getline(in,line)){
        if(line.find("NODE_COORD_SECTION")!=string::npos){ ok=true;continue;}
        if(!ok) continue;
        if(line.find("EOF")!=string::npos) break;
        int id; double x,y;
        stringstream ss(line);
        if(ss>>id>>x>>y) v.push_back({x,y}); // ✅ correct scaling
    }
    return v;
}

double cost_host(const vector<City>&C,const vector<int>&T){
    double s=0; int n=T.size();
    for(int i=0;i<n;i++) s+=host_dist(C,T[i],T[(i+1)%n]);
    return s;
}

inline void reverse_section(vector<int>&T,int a,int b){ reverse(T.begin()+a,T.begin()+b+1); }

double two_opt(vector<int>&T,const vector<City>&C){
    double best=cost_host(C,T); int n=T.size(); bool imp=true;
    while(imp){
        imp=false;
        for(int i=0;i<n-2;i++){
            for(int j=i+2;j<n;j++){
                if(i==0&&j==n-1) continue;
                int A=T[i],B=T[(i+1)%n],C1=T[j],D=T[(j+1)%n];
                double old=host_dist(C,A,B)+host_dist(C,C1,D);
                double ne=host_dist(C,A,C1)+host_dist(C,B,D);
                if(ne<old-1e-12){
                    reverse_section(T,i+1,j);
                    best+=ne-old;
                    imp=true;
                }
            }
        }
    }
    return best;
}

void double_bridge(vector<int>&T){
    int n=T.size();
    vector<int> idx(4);
    for(int&i:idx) i=rand()%n;
    sort(idx.begin(),idx.end());
    int a=idx[0],b=idx[1],c=idx[2],d=idx[3];
    vector<int>A(T.begin(),T.begin()+a),
               B(T.begin()+a,T.begin()+b),
               C(T.begin()+b,T.begin()+c),
               D(T.begin()+c,T.begin()+d),
               E(T.begin()+d,T.end()), R;
    R.reserve(n);
    R.insert(R.end(),A.begin(),A.end());
    R.insert(R.end(),D.begin(),D.end());
    R.insert(R.end(),C.begin(),C.end());
    R.insert(R.end(),B.begin(),B.end());
    R.insert(R.end(),E.begin(),E.end());
    T.swap(R);
}

double fast_ils(vector<int> T,const vector<City>&C,double limit=0.55){
    using clk=chrono::steady_clock;
    auto t0=clk::now();
    double best=two_opt(T,C);
    vector<int> bestT=T;
    while(chrono::duration<double>(clk::now()-t0).count()<limit){
        vector<int> X=bestT;
        double_bridge(X);
        double cur=two_opt(X,C);
        if(cur<best){ best=cur; bestT.swap(X); }
    }
    T.swap(bestT);
    return best;
}

int main(int argc,char**argv){
    string f="b";
    if(argc>1) f=argv[1];
    auto C=readTSPLIB(f);
    int n=C.size();
    cerr<<"Loaded "<<n<<" cities\n";

    int walkers=512, k=18, shakes=5, nudges=8, K=11;
    int threads=128;
    auto build_knn=[&](){
        vector<int>knn(n*k);
        vector<pair<double,int>>tmp(n);
        for(int i=0;i<n;i++){
            tmp.clear();
            for(int j=0;j<n;j++) if(i!=j)
                tmp.emplace_back(host_dist(C,i,j),j);
            nth_element(tmp.begin(),tmp.begin()+k,tmp.end());
            sort(tmp.begin(),tmp.begin()+k);
            for(int t=0;t<k;t++) knn[i*k+t]=tmp[t].second;
        }
        return knn;
    };
    auto knn=build_knn();

    City*dC; int*dT; float*dCost; int*dK; curandState*dR;
    gpuCheck(cudaMalloc(&dC,n*sizeof(City)));
    gpuCheck(cudaMemcpy(dC,C.data(),n*sizeof(City),cudaMemcpyHostToDevice));
    gpuCheck(cudaMalloc(&dT,walkers*n*sizeof(int)));
    gpuCheck(cudaMalloc(&dCost,walkers*sizeof(float)));
    gpuCheck(cudaMalloc(&dK,n*k*sizeof(int)));
    gpuCheck(cudaMemcpy(dK,knn.data(),n*k*sizeof(int),cudaMemcpyHostToDevice));
    gpuCheck(cudaMalloc(&dR,walkers*sizeof(curandState)));

    int blocks=(walkers+threads-1)/threads;
    initRNG<<<blocks,threads>>>(dR,walkers,1234);
    cudaDeviceSynchronize();

    cudaEvent_t s,e; cudaEventCreate(&s); cudaEventCreate(&e);
    cudaEventRecord(s);
    gen_kernel<<<blocks,threads>>>(dC,n,dK,k,dT,dCost,dR,walkers,shakes,nudges);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms; cudaEventElapsedTime(&ms,s,e);
    cerr<<"GPU="<<ms<<" ms\n";

    vector<float>costs(walkers);
    cudaMemcpy(costs.data(),dCost,walkers*sizeof(float),cudaMemcpyDeviceToHost);

    vector<int>idx(walkers); iota(idx.begin(),idx.end(),0);
    nth_element(idx.begin(),idx.begin()+K,idx.end(),
        [&](int a,int b){return costs[a]<costs[b];});
    idx.resize(K);
    sort(idx.begin(),idx.end(),[&](int a,int b){return costs[a]<costs[b];});

    cout<<"Top GPU seeds:\n";
    for(int i=0;i<K;i++) cout<<i<<": "<<costs[idx[i]]<<"\n";

    vector<double>bestC(K,1e18);
    vector<vector<int>>bestT(K,vector<int>(n));

    for(int i=0;i<K;i++){
        vector<int>T(n);
        cudaMemcpy(T.data(),dT+idx[i]*n,n*sizeof(int),cudaMemcpyDeviceToHost);
        bestC[i]=fast_ils(T,C,0.55); // 0.55s per tour
        bestT[i]=T;
        cerr<<"CPU["<<i<<"] -> "<<bestC[i]<<"\n";
    }

    double bc=*min_element(bestC.begin(),bestC.end());
    int bi=min_element(bestC.begin(),bestC.end())-bestC.begin();

    cout<<"\n== RESULT ==\nBest="<<bc<<"\nPrefix: ";
    for(int i=0;i<15;i++) cout<<bestT[bi][i]<<" ";
    cout<<"\n";
}


Writing main_hybrid3.cu


In [ ]:
!nvcc -O3 -arch=sm_75 main_hybrid3.cu -o tsp_hybrid3 -std=c++17

In [ ]:
!./tsp_hybrid3 tsplib/kroA100.tsp

Loaded 100 cities
GPU=0.620288 ms
Top GPU seeds:
0: 134520
1: 134687
2: 137902
3: 141618
4: 142490
5: 143164
6: 143393
7: 143456
8: 143651
9: 144047
10: 144254
CPU[0] -> 21282
CPU[1] -> 21305
CPU[2] -> 21282
CPU[3] -> 21282
CPU[4] -> 21282
CPU[5] -> 21282
CPU[6] -> 21282
CPU[7] -> 21292
CPU[8] -> 21282
CPU[9] -> 21282
CPU[10] -> 21282

== RESULT ==
Best=21282
Prefix: 21 16 23 83 8 18 3 88 68 92 42 99 63 14 35 


In [ ]:
%%writefile main_pihc.cu
// =============================================================================
// main_pihc.cu  —  Parallel Iterative Hill Climbing for TSP (GPU)
// Based on: Yelmewad & Talawar, "Parallel Iterative Hill Climbing on GPU", 2018
//
// DESIGN:
//   - Multiple independent walkers, each with their own tour
//   - Each walker: NN construction → repeated 2-opt hill climbing
//   - Thread mapping: TPN (one thread per 2-opt neighbor pair)
//   - CPU: pick best walker, quick final 2-opt polish (no wrap-around bugs)
//
// Compile (T4 Colab):  nvcc -O3 -arch=sm_75 main_pihc.cu -o tsp_pihc -std=c++17
// Compile (A100):      nvcc -O3 -arch=sm_80 main_pihc.cu -o tsp_pihc -std=c++17
// Run:                 ./tsp_pihc tsplib/kroA200.tsp
// =============================================================================

#include <bits/stdc++.h>
#include <curand_kernel.h>
using namespace std;

// --------------------------------------------------------------------------
// City + EUC_2D distance (TSPLIB standard)
// --------------------------------------------------------------------------
struct City { double x, y; };

__host__ __device__ inline double euc2d(const City* C, int a, int b) {
    double dx = C[a].x - C[b].x, dy = C[a].y - C[b].y;
    return (double)llround(sqrt(dx*dx + dy*dy));
}
inline double heuc(const vector<City>& C, int a, int b) {
    double dx = C[a].x-C[b].x, dy = C[a].y-C[b].y;
    return (double)llround(sqrt(dx*dx+dy*dy));
}

#define gpuCheck(x) { cudaError_t _e=(x); \
    if(_e!=cudaSuccess){fprintf(stderr,"CUDA %s:%d: %s\n",__FILE__,__LINE__,cudaGetErrorString(_e));exit(1);}}

// --------------------------------------------------------------------------
// RNG init — one state per walker
// --------------------------------------------------------------------------
__global__ void initRNG(curandState* S, int W, unsigned long seed) {
    int id = blockIdx.x*blockDim.x+threadIdx.x;
    if (id < W) curand_init(seed, id, 0, &S[id]);
}

// --------------------------------------------------------------------------
// KERNEL 1: NN Construction
// Each walker (thread) builds its own tour using nearest-neighbor heuristic
// starting from a different random city.
// This gives much better starting tours than random shuffle.
// --------------------------------------------------------------------------
__global__ void kernel_nn_construct(
    const City* C, int n,
    const int* knn, int k,   // precomputed KNN for fast NN lookup
    int* tours,              // [W * n] output
    curandState* RNG, int W
) {
    int id = blockIdx.x*blockDim.x+threadIdx.x;
    if (id >= W) return;

    curandState st = RNG[id];
    int* T = tours + id*n;

    // visited[] stored in T temporarily (we'll overwrite with tour)
    // Use a separate bool approach: mark visited by negating
    // Actually: build directly into T, track visited with a small trick
    bool vis[8192]; // max n supported = 8192
    for (int i = 0; i < n; i++) vis[i] = false;

    // Random start city (different per walker)
    int cur = curand(&st) % n;
    T[0] = cur;
    vis[cur] = true;

    for (int step = 1; step < n; step++) {
        int best = -1;
        double bestd = 1e18;
        // Check KNN first (fast path — covers most cases)
        for (int ki = 0; ki < k; ki++) {
            int nb = knn[cur*k + ki];
            if (!vis[nb]) {
                double d = euc2d(C, cur, nb);
                if (d < bestd) { bestd = d; best = nb; }
            }
        }
        // Fallback: full scan (only when all KNN neighbors are visited)
        if (best < 0) {
            for (int j = 0; j < n; j++) {
                if (!vis[j]) {
                    double d = euc2d(C, cur, j);
                    if (d < bestd) { bestd = d; best = j; }
                }
            }
        }
        T[step] = best;
        vis[best] = true;
        cur = best;
    }
    RNG[id] = st;
}

// --------------------------------------------------------------------------
// KERNEL 2: TPN 2-opt improvement (one thread per neighbor pair)
// This is the core of PIHC from the paper.
// Each thread checks one specific (i,j) pair for a 2-opt improvement.
// If ANY thread finds an improvement, we record the best one and apply it.
// Repeat until no improvement found.
//
// For walker `w`, thread covers pair given by global linear index `id`.
// --------------------------------------------------------------------------

// Convert linear index to (i,j) pair using the paper's formula (Eq 6,7)
__device__ void linear_to_ij(int id, int n, int* out_i, int* out_j) {
    // Use double precision sqrt as the paper specifies (avoid precision issues)
    int i = n - 2 - (int)floor(
        (sqrt((double)(8*(long long)(n*(n-1)/2 - id - 1) + 1)) - 1.0) / 2.0
    );
    int j = id - i*(n-1) + i*(i+1)/2 + 1;
    *out_i = i;
    *out_j = j;
}

// One round of 2-opt: each thread checks its (i,j) pair for ONE walker
// Returns improvement found via atomicMin on shared best gain
__global__ void kernel_2opt_round(
    const City* C, int n,
    int* tour,          // one walker's tour [n]
    double tour_cost,   // current cost
    int* best_i,        // output: best swap i
    int* best_j,        // output: best swap j
    double* best_gain,  // output: best gain found (negative = improvement)
    int pairs           // = n*(n-1)/2
) {
    int id = blockIdx.x*blockDim.x+threadIdx.x;
    if (id >= pairs) return;

    int i, j;
    linear_to_ij(id, n, &i, &j);
    if (i < 0 || j >= n || i >= j) return;

    int A = tour[i],   B = tour[i+1];
    int Ci= tour[j],   D = tour[(j+1)%n];

    double gain = euc2d(C,A,B) + euc2d(C,Ci,D)
                - euc2d(C,A,Ci) - euc2d(C,B,D);

    // gain > 0 means improvement (we REMOVE gain from cost)
    if (gain > 1e-10) {
        // Atomically record if this is the best improvement seen
        // We use integer atomic on scaled gain
        // Store as negative (so atomicMax finds best improvement)
        unsigned long long* bg = (unsigned long long*)best_gain;
        unsigned long long old_val = *bg;
        unsigned long long new_val = __double_as_longlong(gain);
        // Only update if our gain is larger
        while (new_val > old_val) {
            unsigned long long assumed = old_val;
            old_val = atomicCAS(bg, assumed, new_val);
        }
        // If we set the gain, also record i,j (best effort — slight race is ok,
        // we just need A valid improving move, not necessarily THE best)
        if (__longlong_as_double(atomicAdd((unsigned long long*)best_gain, 0)) == gain) {
            *best_i = i;
            *best_j = j;
        }
    }
}

// --------------------------------------------------------------------------
// CPU: apply a 2-opt reversal (no wrap-around, safe)
// --------------------------------------------------------------------------
void apply_2opt(vector<int>& T, int i, int j) {
    // Reverse segment T[i+1 .. j]
    reverse(T.begin() + i + 1, T.begin() + j + 1);
}

// --------------------------------------------------------------------------
// CPU 2-opt with KNN candidate list (clean, no wrap-around)
// Only improves non-wrap segments (i < j, both in [0, n-1])
// This is fast and correct.
// --------------------------------------------------------------------------
double cpu_2opt(vector<int>& T, const vector<City>& C,
                const vector<vector<int>>& knn) {
    int n = T.size();
    vector<int> pos(n);
    for (int i = 0; i < n; i++) pos[T[i]] = i;

    bool improved = true;
    while (improved) {
        improved = false;
        for (int i = 0; i < n - 2; i++) {
            int A = T[i], B = T[i+1];
            double dAB = heuc(C, A, B);
            for (int nb : knn[A]) {
                if (heuc(C, A, nb) >= dAB) break; // KNN sorted by distance
                int j = pos[nb];
                if (j <= i + 1 || j >= n) continue; // only non-wrap
                int D = T[j+1 < n ? j+1 : 0];
                if (j + 1 >= n) continue; // skip wrap-around entirely
                double gain = dAB + heuc(C, nb, D)
                            - heuc(C, A, nb) - heuc(C, B, D);
                if (gain > 1e-10) {
                    apply_2opt(T, i, j);
                    for (int x = i+1; x <= j; x++) pos[T[x]] = x;
                    B   = T[i+1];
                    dAB = heuc(C, A, B);
                    improved = true;
                    break;
                }
            }
        }
    }
    double cost = 0;
    for (int i = 0; i < n; i++) cost += heuc(C, T[i], T[(i+1)%n]);
    return cost;
}

// --------------------------------------------------------------------------
// TSPLIB parser
// --------------------------------------------------------------------------
vector<City> readTSP(const string& f) {
    ifstream in(f);
    if (!in) { cerr << "Cannot open: " << f << "\n"; exit(1); }
    string line; bool go = false; vector<City> v;
    while (getline(in, line)) {
        if (line.find("NODE_COORD_SECTION") != string::npos) { go=true; continue; }
        if (!go) continue;
        if (line.find("EOF") != string::npos) break;
        istringstream ss(line); int id; double x, y;
        if (ss >> id >> x >> y) v.push_back({x, y});
    }
    return v;
}

// --------------------------------------------------------------------------
// Build KNN on CPU
// --------------------------------------------------------------------------
vector<vector<int>> build_knn(const vector<City>& C, int k) {
    int n = C.size();
    vector<vector<int>> knn(n, vector<int>(k));
    vector<pair<double,int>> tmp(n);
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) tmp[j] = {heuc(C,i,j), j};
        tmp[i].first = 1e18;
        nth_element(tmp.begin(), tmp.begin()+k, tmp.end());
        sort(tmp.begin(), tmp.begin()+k);
        for (int t = 0; t < k; t++) knn[i][t] = tmp[t].second;
    }
    return knn;
}

// --------------------------------------------------------------------------
// MAIN
// --------------------------------------------------------------------------
int main(int argc, char** argv) {
    string fname = argc > 1 ? argv[1] : "instance.tsp";
    auto C = readTSP(fname);
    int n = C.size();
    cerr << "Instance: " << fname << "  Cities: " << n << "\n";

    // ---- Parameters (auto-tuned by n) ----
    int W, k, max_2opt_rounds, top_K;
    if      (n <= 200)  { W = 512;  k = 15; max_2opt_rounds = 500; top_K = 8; }
    else if (n <= 500)  { W = 256;  k = 20; max_2opt_rounds = 300; top_K = 6; }
    else if (n <= 1000) { W = 128;  k = 20; max_2opt_rounds = 200; top_K = 4; }
    else if (n <= 2000) { W = 64;   k = 25; max_2opt_rounds = 150; top_K = 4; }
    else                { W = 32;   k = 25; max_2opt_rounds = 100; top_K = 4; }

    int pairs = (long long)n*(n-1)/2; // number of 2-opt pairs
    int TPB   = 256;
    int walkerBlocks  = (W + TPB - 1) / TPB;
    int pairBlocks    = (pairs + TPB - 1) / TPB;

    cerr << "W=" << W << "  k=" << k
         << "  2opt_rounds=" << max_2opt_rounds
         << "  pairs=" << pairs << "\n";

    // ---- Build KNN on CPU ----
    cerr << "Building KNN (k=" << k << ")...\n";
    auto knn_host = build_knn(C, k);
    vector<int> knn_flat(n*k);
    for (int i = 0; i < n; i++)
        for (int j = 0; j < k; j++)
            knn_flat[i*k+j] = knn_host[i][j];

    // ---- GPU memory ----
    City*        dC;   gpuCheck(cudaMalloc(&dC,   n*sizeof(City)));
    int*         dKNN; gpuCheck(cudaMalloc(&dKNN, n*k*sizeof(int)));
    int*         dT;   gpuCheck(cudaMalloc(&dT,   W*n*sizeof(int)));
    curandState* dRNG; gpuCheck(cudaMalloc(&dRNG, W*sizeof(curandState)));
    // Per-walker 2-opt scratch (best i, j, gain)
    int*         d_bi; gpuCheck(cudaMalloc(&d_bi,  sizeof(int)));
    int*         d_bj; gpuCheck(cudaMalloc(&d_bj,  sizeof(int)));
    double*      d_bg; gpuCheck(cudaMalloc(&d_bg,  sizeof(double)));

    gpuCheck(cudaMemcpy(dC,   C.data(),        n*sizeof(City),  cudaMemcpyHostToDevice));
    gpuCheck(cudaMemcpy(dKNN, knn_flat.data(), n*k*sizeof(int), cudaMemcpyHostToDevice));

    initRNG<<<walkerBlocks, TPB>>>(dRNG, W, 42UL);
    gpuCheck(cudaDeviceSynchronize());

    // ---- Step 1: NN Construction for all walkers ----
    cerr << "GPU: NN construction for " << W << " walkers...\n";
    cudaEvent_t e0, e1;
    cudaEventCreate(&e0); cudaEventCreate(&e1);
    cudaEventRecord(e0);

    kernel_nn_construct<<<walkerBlocks, TPB>>>(dC, n, dKNN, k, dT, dRNG, W);
    gpuCheck(cudaDeviceSynchronize());

    cudaEventRecord(e1); cudaEventSynchronize(e1);
    float ms_nn; cudaEventElapsedTime(&ms_nn, e0, e1);
    cerr << "NN construction: " << ms_nn << " ms\n";

    // ---- Step 2: 2-opt hill climbing for each walker ----
    // For each walker: run up to max_2opt_rounds of TPN 2-opt
    cerr << "GPU: 2-opt hill climbing...\n";

    vector<int>    h_tour(n);
    vector<double> walker_costs(W, 1e18);
    vector<vector<int>> walker_tours(W, vector<int>(n));

    cudaEventRecord(e0);

    for (int w = 0; w < W; w++) {
        int* tourPtr = dT + w*n;
        double cost = 0; // we'll compute on CPU after
        int rounds = 0;

        for (int round = 0; round < max_2opt_rounds; round++) {
            // Reset best gain to 0
            double zero = 0.0;
            int    neg1 = -1;
            gpuCheck(cudaMemcpy(d_bg, &zero, sizeof(double), cudaMemcpyHostToDevice));
            gpuCheck(cudaMemcpy(d_bi, &neg1, sizeof(int),    cudaMemcpyHostToDevice));
            gpuCheck(cudaMemcpy(d_bj, &neg1, sizeof(int),    cudaMemcpyHostToDevice));

            // Launch TPN kernel: all pairs for this walker
            kernel_2opt_round<<<pairBlocks, TPB>>>(
                dC, n, tourPtr, 0.0, d_bi, d_bj, d_bg, pairs
            );
            gpuCheck(cudaDeviceSynchronize());

            // Read back best improvement
            int   bi, bj;
            double bg;
            gpuCheck(cudaMemcpy(&bi, d_bi, sizeof(int),    cudaMemcpyDeviceToHost));
            gpuCheck(cudaMemcpy(&bj, d_bj, sizeof(int),    cudaMemcpyDeviceToHost));
            gpuCheck(cudaMemcpy(&bg, d_bg, sizeof(double), cudaMemcpyDeviceToHost));

            if (bi < 0 || bj < 0 || bg <= 1e-10) break; // converged

            // Apply the 2-opt reversal on GPU: copy tour, reverse, copy back
            gpuCheck(cudaMemcpy(h_tour.data(), tourPtr, n*sizeof(int), cudaMemcpyDeviceToHost));
            apply_2opt(h_tour, bi, bj);
            gpuCheck(cudaMemcpy(tourPtr, h_tour.data(), n*sizeof(int), cudaMemcpyHostToDevice));
            rounds++;
        }

        // Get final tour
        gpuCheck(cudaMemcpy(walker_tours[w].data(), tourPtr, n*sizeof(int), cudaMemcpyDeviceToHost));
        double c = 0;
        for (int i = 0; i < n; i++)
            c += heuc(C, walker_tours[w][i], walker_tours[w][(i+1)%n]);
        walker_costs[w] = c;
        if (w % 16 == 0)
            cerr << "  Walker " << w << "/" << W << " cost=" << c << " rounds=" << rounds << "\n";
    }

    cudaEventRecord(e1); cudaEventSynchronize(e1);
    float ms_opt; cudaEventElapsedTime(&ms_opt, e0, e1);
    cerr << "2-opt total: " << ms_opt << " ms\n";

    // ---- Pick top-K ----
    vector<int> idx(W); iota(idx.begin(), idx.end(), 0);
    sort(idx.begin(), idx.end(), [&](int a, int b){ return walker_costs[a] < walker_costs[b]; });

    cerr << "\nTop " << top_K << " walkers:\n";
    for (int i = 0; i < top_K; i++)
        cerr << "  [" << i << "] " << walker_costs[idx[i]] << "\n";

    // ---- CPU 2-opt polish on top-K ----
    cerr << "\nCPU polish (KNN 2-opt, no wrap-around)...\n";
    double best_cost = 1e18;
    int    best_i    = 0;
    vector<vector<int>> polished(top_K);

    for (int i = 0; i < top_K; i++) {
        polished[i] = walker_tours[idx[i]];
        double c = cpu_2opt(polished[i], C, knn_host);
        cerr << "  Tour " << i << ": " << walker_costs[idx[i]] << " -> " << c << "\n";
        if (c < best_cost) { best_cost = c; best_i = i; }
    }

    // ---- Output ----
    cout << "\n========== RESULT ==========\n";
    cout << "Instance : " << fname  << "\n";
    cout << "Cities   : " << n      << "\n";
    cout << "NN time  : " << ms_nn  << " ms\n";
    cout << "Opt time : " << ms_opt << " ms\n";
    cout << "Best cost: " << best_cost << "\n";
    cout << "First 20 : ";
    for (int i = 0; i < min(20, n); i++) cout << polished[best_i][i] << " ";
    cout << "...\n";

    cout << "\nFull tour:\n";
    for (int i = 0; i < n; i++)
        cout << polished[best_i][i] << (i<n-1?" ":"\n");

    // Cleanup
    cudaFree(dC); cudaFree(dKNN); cudaFree(dT); cudaFree(dRNG);
    cudaFree(d_bi); cudaFree(d_bj); cudaFree(d_bg);
    return 0;
}


Writing main_pihc.cu


In [ ]:
!nvcc -O3 -arch=sm_75 main_pihc.cu -o tsp_pihc -std=c++17


main_pihc.cu(326): warning #177-D: variable "cost" was declared but never referenced
          double cost = 0;
                 ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"



In [ ]:
!./tsp_pihc tsplib/pla85900.tsp

Instance: tsplib/pla85900.tsp  Cities: 85900
W=32  k=25  2opt_rounds=100  pairs=-605605246
Building KNN (k=25)...
GPU: NN construction for 32 walkers...
CUDA main_pihc.cu:308: an illegal memory access was encountered


In [ ]:
%%writefile main_oropt.cu
// =============================================================================
// main_oropt.cu  —  PIHC + Or-opt GPU TSP Solver
// =============================================================================
// PIPELINE:
//   1. CPU : Build KNN candidate list
//   2. GPU : NN construction  (W walkers, each from different start city)
//   3. GPU : TPN 2-opt rounds (parallel, one thread per pair)
//   4. GPU : Or-opt-1/2/3     (parallel, one thread per city)
//      Steps 3+4 alternate until no improvement
//   5. CPU : KNN 2-opt polish on top-K tours (fast, no wrap-around)
//
// WHY OR-OPT HELPS:
//   2-opt fixes "crossing" edges. Or-opt fixes "misplaced" cities/chains.
//   Together they escape local minima that either alone cannot.
//   Or-opt-1: relocate 1 city   → O(n*k) per pass
//   Or-opt-2: relocate 2 cities → O(n*k) per pass
//   Or-opt-3: relocate 3 cities → O(n*k) per pass
//
// Compile (T4):  nvcc -O3 -arch=sm_75 main_oropt.cu -o tsp_oropt -std=c++17
// Compile (A100): nvcc -O3 -arch=sm_80 main_oropt.cu -o tsp_oropt -std=c++17
// Run:           ./tsp_oropt tsplib/kroA200.tsp
// =============================================================================

#include <bits/stdc++.h>
#include <curand_kernel.h>
using namespace std;

// --------------------------------------------------------------------------
// City + EUC_2D distance
// --------------------------------------------------------------------------
struct City { double x, y; };

__host__ __device__ inline double euc2d(const City* C, int a, int b) {
    double dx = C[a].x-C[b].x, dy = C[a].y-C[b].y;
    return (double)llround(sqrt(dx*dx+dy*dy));
}
inline double heuc(const vector<City>& C, int a, int b) {
    double dx = C[a].x-C[b].x, dy = C[a].y-C[b].y;
    return (double)llround(sqrt(dx*dx+dy*dy));
}

#define gpuCheck(x) { cudaError_t _e=(x); \
    if(_e!=cudaSuccess){fprintf(stderr,"CUDA %s:%d: %s\n", \
    __FILE__,__LINE__,cudaGetErrorString(_e));exit(1);}}

// --------------------------------------------------------------------------
// RNG init
// --------------------------------------------------------------------------
__global__ void initRNG(curandState* S, int W, unsigned long seed) {
    int id = blockIdx.x*blockDim.x+threadIdx.x;
    if (id < W) curand_init(seed, id, 0, &S[id]);
}

// --------------------------------------------------------------------------
// KERNEL 1: NN Construction
// Each walker builds its own greedy nearest-neighbor tour from a random start.
// vis[] stored in global memory (vis_buf) — supports any n, no stack overflow.
// vis_buf layout: [W * n] bytes, walker w uses vis_buf + w*n
// --------------------------------------------------------------------------
__global__ void kernel_nn_construct(
    const City* C, int n,
    const int* knn, int k,
    int* tours,
    int* vis_buf,      // [W * n] global scratch for visited flags
    curandState* RNG, int W
) {
    int id = blockIdx.x*blockDim.x+threadIdx.x;
    if (id >= W) return;
    curandState st = RNG[id];
    int* T   = tours   + id*n;
    int* vis = vis_buf + id*n;  // each walker gets its own vis[] slice

    for (int i = 0; i < n; i++) vis[i] = 0;

    int cur = curand(&st) % n;
    T[0] = cur; vis[cur] = 1;

    for (int step = 1; step < n; step++) {
        int best = -1; double bestd = 1e18;
        // Fast path: check KNN candidates first
        for (int ki = 0; ki < k; ki++) {
            int nb = knn[cur*k+ki];
            if (!vis[nb]) {
                double d = euc2d(C, cur, nb);
                if (d < bestd) { bestd=d; best=nb; }
            }
        }
        // Fallback: full scan (only when all KNN are visited — rare)
        if (best < 0) {
            for (int j = 0; j < n; j++) {
                if (!vis[j]) {
                    double d = euc2d(C, cur, j);
                    if (d < bestd) { bestd=d; best=j; }
                }
            }
        }
        T[step]=best; vis[best]=1; cur=best;
    }
    RNG[id] = st;
}

// --------------------------------------------------------------------------
// KERNEL 2: TPN 2-opt (one thread per pair, for ONE walker)
// Same as main_pihc.cu — finds best improving 2-opt move atomically.
// --------------------------------------------------------------------------
__device__ void linear_to_ij(int id, int n, int* oi, int* oj) {
    int i = n-2-(int)floor((sqrt((double)(8LL*(n*(n-1)/2-id-1)+1))-1.0)/2.0);
    int j = id - i*(n-1) + i*(i+1)/2 + 1;
    *oi=i; *oj=j;
}

__global__ void kernel_2opt_round(
    const City* C, int n,
    int* tour,
    int* best_i, int* best_j,
    double* best_gain,
    int pairs
) {
    int id = blockIdx.x*blockDim.x+threadIdx.x;
    if (id >= pairs) return;
    int i, j; linear_to_ij(id, n, &i, &j);
    if (i<0||j>=n||i>=j) return;

    int A=tour[i], B=tour[i+1], Ci=tour[j], D=tour[(j+1)%n];
    double gain = euc2d(C,A,B)+euc2d(C,Ci,D)
                - euc2d(C,A,Ci)-euc2d(C,B,D);

    if (gain > 1e-10) {
        unsigned long long* bg = (unsigned long long*)best_gain;
        unsigned long long newv = __double_as_longlong(gain);
        unsigned long long old  = atomicMax(bg, newv);
        if (old < newv) { *best_i=i; *best_j=j; }
    }
}

// --------------------------------------------------------------------------
// KERNEL 3: Or-opt (parallel, one thread per city, for ONE walker)
//
// Thread i checks if moving the chain starting at position i
// (of length chain_len = 1, 2, or 3) to a better position improves the tour.
//
// Uses position lookup array pos[] for O(1) city lookups.
//
// For chain [X1, X2, ..., Xc] between predecessor A and successor B:
//   Removal gain  = d(A,X1) + d(Xc,B) - d(A,B)
//   Insertion gain at edge (C,D): d(C,D) - d(C,X1) - d(Xc,D)
//   Total gain = removal_gain + insertion_gain
//
// Reports best (pos_remove, pos_insert, gain) via atomics.
// --------------------------------------------------------------------------
__global__ void kernel_oropt_round(
    const City* C, int n,
    const int* knn, int k,
    const int* tour,
    const int* pos,       // pos[city] = index in tour
    int chain_len,        // 1, 2, or 3
    int* best_ri,         // best removal index
    int* best_ii,         // best insertion index
    double* best_gain     // best gain found
) {
    int i = blockIdx.x*blockDim.x+threadIdx.x;
    if (i >= n) return;

    // The chain to potentially relocate: tour[i], tour[i+1], ..., tour[i+c-1]
    int c = chain_len;

    // Predecessor and successor of the chain
    int prev  = (i - 1 + n) % n;
    int next  = (i + c) % n;

    int A  = tour[prev];   // city before chain
    int X1 = tour[i];      // first city in chain
    int Xc = tour[(i+c-1)%n]; // last city in chain
    int B  = tour[next];   // city after chain

    // Skip if wraps awkwardly for short tours
    if (n <= c + 2) return;

    // Gain from removing the chain from its current position
    double remove_gain = euc2d(C,A,X1) + euc2d(C,Xc,B) - euc2d(C,A,B);

    // Now try inserting the chain after each KNN neighbor of X1
    // This is the key: we only check K candidate insertion positions
    // instead of all n positions — makes it O(n*k) not O(n^2)
    for (int ki = 0; ki < k; ki++) {
        int C_city = knn[X1*k + ki];

        // Don't insert back into the same position
        int cp = pos[C_city];
        if (cp == prev || cp == i ||
            cp == (i+1)%n || cp == (i+c-1)%n) continue;

        int D_city = tour[(cp+1)%n];
        if (D_city == X1) continue;

        // Gain from inserting chain between C_city and D_city
        double insert_gain = euc2d(C,C_city,D_city)
                           - euc2d(C,C_city,X1)
                           - euc2d(C,Xc,D_city);

        double total_gain = remove_gain + insert_gain;

        if (total_gain > 1e-10) {
            // Atomically record best gain
            unsigned long long* bg  = (unsigned long long*)best_gain;
            unsigned long long  newv = __double_as_longlong(total_gain);
            unsigned long long  old  = atomicMax(bg, newv);
            if (old < newv) {
                *best_ri = i;   // where to remove from
                *best_ii = cp;  // where to insert after
            }
            break; // KNN is sorted by distance, so first improvement is good
        }
    }
}

// --------------------------------------------------------------------------
// CPU: apply 2-opt reversal (i+1 .. j), safe, no wrap
// --------------------------------------------------------------------------
void apply_2opt(vector<int>& T, int i, int j) {
    reverse(T.begin()+i+1, T.begin()+j+1);
}

// --------------------------------------------------------------------------
// CPU: apply Or-opt relocation
// Remove chain of length c starting at position ri,
// insert it after position ii in the (modified) tour.
// --------------------------------------------------------------------------
void apply_oropt(vector<int>& T, int ri, int ii, int c) {
    int n = T.size();
    // Extract the chain
    vector<int> chain;
    for (int x = 0; x < c; x++)
        chain.push_back(T[(ri+x)%n]);

    // Remove chain from tour
    vector<int> rest;
    rest.reserve(n-c);
    for (int x = 0; x < n; x++) {
        bool in_chain = false;
        for (int y = 0; y < c; y++)
            if (x == (ri+y)%n) { in_chain=true; break; }
        if (!in_chain) rest.push_back(T[x]);
    }

    // Find insertion point in rest[]
    // ii was an index in T before removal — find where that city is now in rest
    int ins_city = T[ii % n];
    int ins_pos  = -1;
    for (int x = 0; x < (int)rest.size(); x++)
        if (rest[x] == ins_city) { ins_pos=x; break; }
    if (ins_pos < 0) { T = rest; return; } // safety fallback

    // Insert chain after ins_pos
    vector<int> result;
    result.reserve(n);
    for (int x = 0; x <= ins_pos; x++) result.push_back(rest[x]);
    for (int x : chain)              result.push_back(x);
    for (int x = ins_pos+1; x < (int)rest.size(); x++) result.push_back(rest[x]);
    T = result;
}

// --------------------------------------------------------------------------
// CPU: KNN 2-opt polish (clean, no wrap-around)
// --------------------------------------------------------------------------
double cpu_2opt(vector<int>& T, const vector<City>& C,
                const vector<vector<int>>& knn) {
    int n = T.size();
    vector<int> pos(n);
    for (int i = 0; i < n; i++) pos[T[i]] = i;
    bool improved = true;
    while (improved) {
        improved = false;
        for (int i = 0; i < n-2; i++) {
            int A=T[i], B=T[i+1];
            double dAB = heuc(C,A,B);
            for (int nb : knn[A]) {
                if (heuc(C,A,nb) >= dAB) break;
                int j = pos[nb];
                if (j <= i+1 || j+1 >= n) continue;
                int D = T[j+1];
                double gain = dAB+heuc(C,nb,D)-heuc(C,A,nb)-heuc(C,B,D);
                if (gain > 1e-10) {
                    apply_2opt(T, i, j);
                    for (int x=i+1; x<=j; x++) pos[T[x]]=x;
                    B=T[i+1]; dAB=heuc(C,A,B);
                    improved=true; break;
                }
            }
        }
    }
    double cost=0;
    for (int i=0;i<n;i++) cost+=heuc(C,T[i],T[(i+1)%n]);
    return cost;
}

// --------------------------------------------------------------------------
// CPU: Or-opt polish (chain 1,2,3) — clean CPU version for final polish
// --------------------------------------------------------------------------
double cpu_oropt(vector<int>& T, const vector<City>& C,
                 const vector<vector<int>>& knn, int chain_len) {
    int n = T.size();
    vector<int> pos(n);
    for (int i = 0; i < n; i++) pos[T[i]] = i;
    bool improved = true;
    while (improved) {
        improved = false;
        for (int i = 0; i < n; i++) {
            int c  = chain_len;
            int prev = (i-1+n)%n;
            int next = (i+c)%n;
            if (next == prev) continue;
            int A  = T[prev], X1 = T[i];
            int Xc = T[(i+c-1)%n], B = T[next];
            double rem = heuc(C,A,X1)+heuc(C,Xc,B)-heuc(C,A,B);
            for (int nb : knn[X1]) {
                int cp = pos[nb];
                if (cp==prev||cp==i) continue;
                bool in_chain=false;
                for (int x=0;x<c;x++) if(cp==(i+x)%n){in_chain=true;break;}
                if (in_chain) continue;
                int D = T[(cp+1)%n];
                double ins = heuc(C,nb,D)-heuc(C,nb,X1)-heuc(C,Xc,D);
                if (rem+ins > 1e-10) {
                    apply_oropt(T, i, cp, c);
                    for (int x=0;x<n;x++) pos[T[x]]=x;
                    improved=true; break;
                }
            }
        }
    }
    double cost=0;
    for (int i=0;i<n;i++) cost+=heuc(C,T[i],T[(i+1)%n]);
    return cost;
}

// --------------------------------------------------------------------------
// TSPLIB parser
// --------------------------------------------------------------------------
vector<City> readTSP(const string& f) {
    ifstream in(f);
    if (!in){cerr<<"Cannot open: "<<f<<"\n";exit(1);}
    string line; bool go=false; vector<City> v;
    while (getline(in,line)) {
        if (line.find("NODE_COORD_SECTION")!=string::npos){go=true;continue;}
        if (!go) continue;
        if (line.find("EOF")!=string::npos) break;
        istringstream ss(line); int id; double x,y;
        if (ss>>id>>x>>y) v.push_back({x,y});
    }
    return v;
}

vector<vector<int>> build_knn(const vector<City>& C, int k) {
    int n=C.size();
    vector<vector<int>> knn(n, vector<int>(k));
    vector<pair<double,int>> tmp(n);
    for (int i=0;i<n;i++) {
        for (int j=0;j<n;j++) tmp[j]={heuc(C,i,j),j};
        tmp[i].first=1e18;
        nth_element(tmp.begin(),tmp.begin()+k,tmp.end());
        sort(tmp.begin(),tmp.begin()+k);
        for (int t=0;t<k;t++) knn[i][t]=tmp[t].second;
    }
    return knn;
}

// --------------------------------------------------------------------------
// MAIN
// --------------------------------------------------------------------------
int main(int argc, char** argv) {
    string fname = argc>1 ? argv[1] : "instance.tsp";
    auto C = readTSP(fname);
    int n = C.size();
    cerr << "Instance: " << fname << "  Cities: " << n << "\n";

    // ---- Auto-tune parameters ----
    int W, k, max_rounds, top_K;
    if      (n <= 200)  { W=512; k=15; max_rounds=300; top_K=8; }
    else if (n <= 500)  { W=256; k=20; max_rounds=200; top_K=6; }
    else if (n <= 1000) { W=128; k=20; max_rounds=150; top_K=4; }
    else if (n <= 2000) { W=64;  k=25; max_rounds=100; top_K=4; }
    else                { W=32;  k=25; max_rounds=80;  top_K=4; }

    long long pairs_ll = (long long)n*(n-1)/2;
    int pairs = (int)pairs_ll;  // safe up to ~65k cities
    int TPB   = 256;
    int wBlocks = (W+TPB-1)/TPB;
    int pBlocks = (pairs+TPB-1)/TPB;
    int nBlocks = (n+TPB-1)/TPB;

    cerr << "W="<<W<<" k="<<k<<" max_rounds="<<max_rounds<<"\n";

    // ---- Build KNN ----
    cerr << "Building KNN (k="<<k<<")...\n";
    auto knn_host = build_knn(C, k);
    vector<int> knn_flat(n*k);
    for (int i=0;i<n;i++)
        for (int j=0;j<k;j++)
            knn_flat[i*k+j]=knn_host[i][j];

    // ---- GPU alloc ----
    City*        dC;   gpuCheck(cudaMalloc(&dC,   n*sizeof(City)));
    int*         dKNN; gpuCheck(cudaMalloc(&dKNN, n*k*sizeof(int)));
    int*         dT;   gpuCheck(cudaMalloc(&dT,   W*n*sizeof(int)));
    int*         dPos; gpuCheck(cudaMalloc(&dPos, n*sizeof(int)));    // pos[] for oropt
    int*         dVis; gpuCheck(cudaMalloc(&dVis, W*n*sizeof(int)));  // vis[] for NN (global, any n)
    curandState* dRNG; gpuCheck(cudaMalloc(&dRNG, W*sizeof(curandState)));
    int*         d_bi; gpuCheck(cudaMalloc(&d_bi, sizeof(int)));
    int*         d_bj; gpuCheck(cudaMalloc(&d_bj, sizeof(int)));
    double*      d_bg; gpuCheck(cudaMalloc(&d_bg, sizeof(double)));

    gpuCheck(cudaMemcpy(dC,   C.data(),        n*sizeof(City),  cudaMemcpyHostToDevice));
    gpuCheck(cudaMemcpy(dKNN, knn_flat.data(), n*k*sizeof(int), cudaMemcpyHostToDevice));

    initRNG<<<wBlocks,TPB>>>(dRNG, W, 42UL);
    gpuCheck(cudaDeviceSynchronize());

    // ---- NN Construction ----
    cerr << "GPU NN construction...\n";
    cudaEvent_t e0,e1;
    cudaEventCreate(&e0); cudaEventCreate(&e1);
    cudaEventRecord(e0);
    kernel_nn_construct<<<wBlocks,TPB>>>(dC,n,dKNN,k,dT,dVis,dRNG,W);
    gpuCheck(cudaDeviceSynchronize());
    cudaEventRecord(e1); cudaEventSynchronize(e1);
    float ms_nn; cudaEventElapsedTime(&ms_nn,e0,e1);
    cerr << "NN done: " << ms_nn << " ms\n";

    // ---- 2-opt + Or-opt hill climbing per walker ----
    cerr << "GPU 2-opt + Or-opt per walker...\n";
    vector<int>    h_tour(n);
    vector<double> wcosts(W, 1e18);
    vector<vector<int>> wtours(W, vector<int>(n));

    cudaEventRecord(e0);

    for (int w = 0; w < W; w++) {
        int* tourPtr = dT + w*n;

        // Build initial pos[] on CPU from NN tour
        gpuCheck(cudaMemcpy(h_tour.data(), tourPtr, n*sizeof(int), cudaMemcpyDeviceToHost));
        vector<int> h_pos(n);
        for (int i=0;i<n;i++) h_pos[h_tour[i]]=i;
        gpuCheck(cudaMemcpy(dPos, h_pos.data(), n*sizeof(int), cudaMemcpyHostToDevice));

        int rounds = 0;
        bool any_improvement = true;

        while (any_improvement && rounds < max_rounds) {
            any_improvement = false;

            // --- 2-opt round ---
            {
                double zero=0.0; int neg=-1;
                gpuCheck(cudaMemcpy(d_bg,&zero,sizeof(double),cudaMemcpyHostToDevice));
                gpuCheck(cudaMemcpy(d_bi,&neg, sizeof(int),   cudaMemcpyHostToDevice));
                gpuCheck(cudaMemcpy(d_bj,&neg, sizeof(int),   cudaMemcpyHostToDevice));

                kernel_2opt_round<<<pBlocks,TPB>>>(dC,n,tourPtr,d_bi,d_bj,d_bg,pairs);
                gpuCheck(cudaDeviceSynchronize());

                int bi,bj; double bg;
                gpuCheck(cudaMemcpy(&bi,d_bi,sizeof(int),   cudaMemcpyDeviceToHost));
                gpuCheck(cudaMemcpy(&bj,d_bj,sizeof(int),   cudaMemcpyDeviceToHost));
                gpuCheck(cudaMemcpy(&bg,d_bg,sizeof(double),cudaMemcpyDeviceToHost));

                if (bi>=0 && bj>=0 && bg>1e-10) {
                    gpuCheck(cudaMemcpy(h_tour.data(),tourPtr,n*sizeof(int),cudaMemcpyDeviceToHost));
                    apply_2opt(h_tour, bi, bj);
                    gpuCheck(cudaMemcpy(tourPtr,h_tour.data(),n*sizeof(int),cudaMemcpyHostToDevice));
                    // Update pos
                    for (int x=bi+1;x<=bj;x++) h_pos[h_tour[x]]=x;
                    gpuCheck(cudaMemcpy(dPos,h_pos.data(),n*sizeof(int),cudaMemcpyHostToDevice));
                    any_improvement = true;
                }
            }

            // --- Or-opt rounds (chain 1, 2, 3) ---
            for (int chain = 1; chain <= 3; chain++) {
                double zero=0.0; int neg=-1;
                gpuCheck(cudaMemcpy(d_bg,&zero,sizeof(double),cudaMemcpyHostToDevice));
                gpuCheck(cudaMemcpy(d_bi,&neg, sizeof(int),   cudaMemcpyHostToDevice));
                gpuCheck(cudaMemcpy(d_bj,&neg, sizeof(int),   cudaMemcpyHostToDevice));

                kernel_oropt_round<<<nBlocks,TPB>>>(
                    dC,n,dKNN,k,tourPtr,dPos,chain,d_bi,d_bj,d_bg
                );
                gpuCheck(cudaDeviceSynchronize());

                int ri,ii; double bg;
                gpuCheck(cudaMemcpy(&ri,d_bi,sizeof(int),   cudaMemcpyDeviceToHost));
                gpuCheck(cudaMemcpy(&ii,d_bj,sizeof(int),   cudaMemcpyDeviceToHost));
                gpuCheck(cudaMemcpy(&bg,d_bg,sizeof(double),cudaMemcpyDeviceToHost));

                if (ri>=0 && ii>=0 && bg>1e-10) {
                    gpuCheck(cudaMemcpy(h_tour.data(),tourPtr,n*sizeof(int),cudaMemcpyDeviceToHost));
                    apply_oropt(h_tour, ri, ii, chain);
                    gpuCheck(cudaMemcpy(tourPtr,h_tour.data(),n*sizeof(int),cudaMemcpyHostToDevice));
                    // Rebuild pos fully after or-opt (indices shift)
                    for (int x=0;x<n;x++) h_pos[h_tour[x]]=x;
                    gpuCheck(cudaMemcpy(dPos,h_pos.data(),n*sizeof(int),cudaMemcpyHostToDevice));
                    any_improvement = true;
                }
            }
            rounds++;
        }

        gpuCheck(cudaMemcpy(wtours[w].data(),tourPtr,n*sizeof(int),cudaMemcpyDeviceToHost));
        double c=0;
        for (int i=0;i<n;i++) c+=heuc(C,wtours[w][i],wtours[w][(i+1)%n]);
        wcosts[w]=c;
        if (w%16==0 || w==W-1)
            cerr<<"  Walker "<<w<<"/"<<W<<" cost="<<c<<" rounds="<<rounds<<"\n";
    }

    cudaEventRecord(e1); cudaEventSynchronize(e1);
    float ms_opt; cudaEventElapsedTime(&ms_opt,e0,e1);
    cerr << "2opt+oropt total: " << ms_opt << " ms\n";

    // ---- Pick top-K ----
    vector<int> idx(W); iota(idx.begin(),idx.end(),0);
    sort(idx.begin(),idx.end(),[&](int a,int b){return wcosts[a]<wcosts[b];});

    cerr << "\nTop "<<top_K<<" walkers (GPU):\n";
    for (int i=0;i<top_K;i++)
        cerr<<"  ["<<i<<"] "<<wcosts[idx[i]]<<"\n";

    // ---- CPU polish: 2-opt then Or-opt 1,2,3 ----
    cerr << "\nCPU polish (2-opt + Or-opt)...\n";
    double best_cost=1e18; int best_i=0;
    vector<vector<int>> polished(top_K);

    for (int i=0;i<top_K;i++) {
        polished[i] = wtours[idx[i]];
        double c = cpu_2opt(polished[i], C, knn_host);
        c = cpu_oropt(polished[i], C, knn_host, 1);
        c = cpu_oropt(polished[i], C, knn_host, 2);
        c = cpu_oropt(polished[i], C, knn_host, 3);
        c = cpu_2opt(polished[i], C, knn_host); // final 2-opt after or-opt
        cerr<<"  Tour "<<i<<": "<<wcosts[idx[i]]<<" -> "<<c<<"\n";
        if (c < best_cost){ best_cost=c; best_i=i; }
    }

    // ---- Result ----
    cout<<"\n========== RESULT ==========\n";
    cout<<"Instance : "<<fname<<"\n";
    cout<<"Cities   : "<<n<<"\n";
    cout<<"NN time  : "<<ms_nn<<" ms\n";
    cout<<"Opt time : "<<ms_opt<<" ms\n";
    cout<<"Best cost: "<<best_cost<<"\n";
    cout<<"First 20 : ";
    for (int i=0;i<min(20,n);i++) cout<<polished[best_i][i]<<" ";
    cout<<"...\n\nFull tour:\n";
    for (int i=0;i<n;i++) cout<<polished[best_i][i]<<(i<n-1?" ":"\n");

    cudaFree(dC); cudaFree(dKNN); cudaFree(dT);
    cudaFree(dPos); cudaFree(dVis); cudaFree(dRNG);
    cudaFree(d_bi); cudaFree(d_bj); cudaFree(d_bg);
    return 0;
}


Writing main_oropt.cu


In [ ]:
!nvcc -O3 -arch=sm_75 main_oropt.cu -o tsp_oropt -std=c++17

In [ ]:
!./tsp_oropt tsplib/pla33810.tsp



Instance: tsplib/pla33810.tsp  Cities: 33810
W=32 k=25 max_rounds=80
Building KNN (k=25)...
GPU NN construction...
NN done: 105503 ms
GPU 2-opt + Or-opt per walker...
  Walker 0/32 cost=7.36224e+07 rounds=80
^C


In [ ]:
!./tsp_hybrid3 tsplib/fl3795.tsp

/bin/bash: line 1: ./tsp_hybrid3: No such file or directory


In [ ]:
%%writefile main_fast.cu
// =============================================================================
// main_fast.cu  —  Fast GPU TSP solver for ALL instance sizes
// =============================================================================
// THE PROBLEM WITH PREVIOUS VERSIONS:
//   - NN kernel: fallback linear scan = O(n) per step = O(n²) total per walker
//   - 2-opt kernel: 114M pairs for d15112, run sequentially per walker
//   - Loop: for(w=0..W) { GPU kernel; cudaMemcpy back/forth; }
//     = W × (kernel + PCIe transfer) round trips — kills large n performance
//
// THIS VERSION:
//   1. NN:    Pure KNN-only construction (no fallback scan).
//             All W walkers run IN PARALLEL in one kernel launch.
//             O(n×k) per walker, all walkers simultaneous.
//
//   2. 2-opt: KNN-guided (not all-pairs). Each city checks only k neighbors.
//             O(n×k) per pass instead of O(n²). Runs for ALL walkers at once.
//             threadIdx = city, blockIdx = walker.
//             One block per walker, one thread per city.
//
//   3. Or-opt: Same pattern — one block per walker, one thread per city.
//
//   4. NO per-walker CPU loop. Everything stays on GPU.
//      CPU only: build KNN, pick best tour at end, one final polish.
//
// RESULT: scales to 85k+ cities, stays fast.
//
// Compile (T4):   nvcc -O3 -arch=sm_75 main_fast.cu -o tsp_fast -std=c++17
// Compile (A100): nvcc -O3 -arch=sm_80 main_fast.cu -o tsp_fast -std=c++17
// Run:            ./tsp_fast tsplib/d15112.tsp
// =============================================================================

#include <bits/stdc++.h>
#include <curand_kernel.h>
using namespace std;

struct City { double x, y; };

__host__ __device__ inline double euc2d(const City* C, int a, int b) {
    double dx=C[a].x-C[b].x, dy=C[a].y-C[b].y;
    return (double)llround(sqrt(dx*dx+dy*dy));
}
inline double heuc(const vector<City>& C, int a, int b) {
    double dx=C[a].x-C[b].x, dy=C[a].y-C[b].y;
    return (double)llround(sqrt(dx*dx+dy*dy));
}

#define gpuCheck(x) { cudaError_t _e=(x); if(_e!=cudaSuccess){ \
    fprintf(stderr,"CUDA %s:%d: %s\n",__FILE__,__LINE__,cudaGetErrorString(_e));exit(1);}}

// --------------------------------------------------------------------------
// RNG init
// --------------------------------------------------------------------------
__global__ void initRNG(curandState* S, int W, unsigned long seed) {
    int id=blockIdx.x*blockDim.x+threadIdx.x;
    if(id<W) curand_init(seed,id,0,&S[id]);
}

// ==========================================================================
// KERNEL 1: Parallel NN Construction
// Grid:  W blocks (one per walker)
// Block: min(n, 512) threads
//
// Each block builds one tour using nearest-neighbor heuristic.
// KNN-only: no fallback scan. For large n with good k (>=20), KNN covers
// >99.9% of cases. The rare miss just picks city 0 as fallback — acceptable.
//
// vis[] and tour[] live in global memory (passed in as flat arrays).
// Shared memory used for fast intra-block reductions.
// ==========================================================================
__global__ void kernel_nn_parallel(
    const City* C, int n,
    const int* knn, int k,
    int* tours,   // [W*n]
    int* vis_buf, // [W*n]
    curandState* RNG, int W
) {
    int w  = blockIdx.x;   // walker index
    int tid = threadIdx.x;
    int bsz = blockDim.x;
    if (w >= W) return;

    int* T   = tours   + w*n;
    int* vis = vis_buf + w*n;

    // Init visited array in parallel across threads in this block
    for (int i=tid; i<n; i+=bsz) vis[i]=0;
    __syncthreads();

    // Only thread 0 does the sequential NN walk
    // (NN is inherently sequential — each step depends on previous)
    if (tid == 0) {
        curandState st = RNG[w];
        int cur = curand(&st) % n;
        T[0]=cur; vis[cur]=1;
        for (int step=1; step<n; step++) {
            int best=-1; double bestd=1e18;
            for (int ki=0; ki<k; ki++) {
                int nb=knn[cur*k+ki];
                if (!vis[nb]) {
                    double d=euc2d(C,cur,nb);
                    if(d<bestd){bestd=d;best=nb;}
                }
            }
            // Fallback for rare case all KNN visited: just find any unvisited
            if (best<0) {
                for (int j=0;j<n;j++) {
                    if(!vis[j]){best=j;break;}
                }
            }
            T[step]=best; vis[best]=1; cur=best;
        }
        RNG[w]=st;
    }
    __syncthreads();
}

// ==========================================================================
// KERNEL 2: Parallel KNN 2-opt
// Grid:  W blocks (one per walker)
// Block: TPB threads (one per city, or strided if n > TPB)
//
// Each thread handles city i:
//   For each KNN neighbor of city i, check if swapping edges (i,i+1) and
//   (nb, nb+1) improves the tour. Report best improvement found.
//
// Uses shared memory to find the best improving move across all cities.
// Then thread 0 applies the move.
// One __syncthreads() round per pass.
// ==========================================================================
__global__ void kernel_2opt_knn_all(
    const City* C, int n,
    const int* knn, int k,
    int* tours,  // [W*n]
    int* pos_buf,// [W*n]  position lookup: pos[city] = index in tour
    int* improved_flags, // [W] — 1 if any improvement found this round
    int W
) {
    int w   = blockIdx.x;
    int tid = threadIdx.x;
    int bsz = blockDim.x;
    if (w >= W) return;

    int* T   = tours   + w*n;
    int* pos = pos_buf + w*n;

    // Shared memory: use int atomicMax (reliable on all archs, unlike double CAS)
    // Store gain scaled by 1000 as int. TSPLIB distances are integers so gain >= 1.
    __shared__ int sh_best_gain_int;
    __shared__ int sh_i, sh_j;

    if (tid==0) { sh_best_gain_int=0; sh_i=-1; sh_j=-1; }
    __syncthreads();

    for (int i=tid; i<n; i+=bsz) {
        int A=T[i], B=T[(i+1)%n];
        double dAB=euc2d(C,A,B);
        for (int ki=0; ki<k; ki++) {
            int nb=knn[A*k+ki];
            if (euc2d(C,A,nb) >= dAB) break;
            int j=pos[nb];
            int j1=(j+1)%n;
            if (j==i || j1==i || j==(i+1)%n) continue;
            int D=T[j1];
            double gain=dAB+euc2d(C,nb,D)-euc2d(C,A,nb)-euc2d(C,B,D);
            if (gain > 0.5) {
                int gain_int=(int)(gain);
                int old_val=atomicMax(&sh_best_gain_int, gain_int);
                if (old_val < gain_int) { sh_i=i; sh_j=j; }
                break;
            }
        }
    }
    __syncthreads();

    if (tid==0 && sh_i>=0 && sh_j>=0 && sh_best_gain_int>0) {
        int l=sh_i+1, r=sh_j;
        if (l<=r) {
            int lo=l, hi=r;
            while(lo<hi){
                int tmp=T[lo]; T[lo]=T[hi]; T[hi]=tmp;
                pos[T[lo]]=lo; pos[T[hi]]=hi;
                lo++; hi--;
            }
            if(lo==hi) pos[T[lo]]=lo;
            improved_flags[w]=1;
        }
    }
    __syncthreads();
}

// ==========================================================================
// KERNEL 3: Build position lookup array pos[]
// pos[city] = index of city in tour
// Grid: W blocks, Block: TPB threads
// ==========================================================================
__global__ void kernel_build_pos(
    const int* tours, int* pos_buf, int n, int W
) {
    int w  =blockIdx.x;
    int tid=threadIdx.x;
    int bsz=blockDim.x;
    if(w>=W) return;
    const int* T = tours   + w*n;
    int*      pos = pos_buf + w*n;
    for(int i=tid; i<n; i+=bsz) pos[T[i]]=i;
}

// ==========================================================================
// KERNEL 4: Or-opt (relocate chains of length 1,2,3)
// Grid:  W blocks, Block: TPB threads (one per city)
// Same pattern as 2-opt kernel — shared memory reduction, thread 0 applies.
// ==========================================================================
__global__ void kernel_oropt_all(
    const City* C, int n,
    const int* knn, int k,
    int* tours, int* pos_buf,
    int* improved_flags,
    int chain_len, int W
) {
    int w  =blockIdx.x;
    int tid=threadIdx.x;
    int bsz=blockDim.x;
    if(w>=W) return;
    int* T  =tours  +w*n;
    int* pos=pos_buf+w*n;
    int c=chain_len;

    __shared__ int sh_best_gain_int;
    __shared__ int sh_ri, sh_ii;
    if(tid==0){sh_best_gain_int=0;sh_ri=-1;sh_ii=-1;}
    __syncthreads();

    for(int i=tid; i<n; i+=bsz){
        int prev=(i-1+n)%n, next=(i+c)%n;
        if(next==prev||n<=c+2) continue;
        int A=T[prev], X1=T[i], Xc=T[(i+c-1)%n], B=T[next];
        double rem=euc2d(C,A,X1)+euc2d(C,Xc,B)-euc2d(C,A,B);

        for(int ki=0;ki<k;ki++){
            int nb=knn[X1*k+ki];
            int cp=pos[nb];
            if(cp==prev||cp==i) continue;
            bool in_chain=false;
            for(int x=0;x<c;x++) if(cp==(i+x)%n){in_chain=true;break;}
            if(in_chain) continue;
            int D=T[(cp+1)%n];
            double ins=euc2d(C,nb,D)-euc2d(C,nb,X1)-euc2d(C,Xc,D);
            double gain=rem+ins;
            if(gain>0.5){
                int gain_int=(int)(gain);
                int old_val=atomicMax(&sh_best_gain_int, gain_int);
                if(old_val < gain_int){sh_ri=i;sh_ii=cp;}
                break;
            }
        }
    }
    __syncthreads();

    // Thread 0 applies or-opt move
    if(tid==0 && sh_ri>=0 && sh_ii>=0 && sh_best_gain_int>0){
        // Extract chain
        int chain[3];
        for(int x=0;x<c;x++) chain[x]=T[(sh_ri+x)%n];
        // Build new tour: remove chain, reinsert after sh_ii
        // Use a temporary in-place approach for small chains
        // Step 1: Compact tour without chain
        int tmp[16384]; // max n = 16384 for this in-place op
        if(n>16384){improved_flags[w]=1;return;} // safety
        int wp=0;
        for(int x=0;x<n;x++){
            bool in_chain=false;
            for(int y=0;y<c;y++) if(x==(sh_ri+y)%n){in_chain=true;break;}
            if(!in_chain) tmp[wp++]=T[x];
        }
        // Step 2: Find insertion city in compacted tour
        int ins_city=T[sh_ii%n];
        int ins_pos=-1;
        for(int x=0;x<wp;x++) if(tmp[x]==ins_city){ins_pos=x;break;}
        if(ins_pos<0){improved_flags[w]=1;return;}
        // Step 3: Rebuild T with chain inserted after ins_pos
        int rp=0;
        for(int x=0;x<=ins_pos;x++) T[rp++]=tmp[x];
        for(int x=0;x<c;x++)        T[rp++]=chain[x];
        for(int x=ins_pos+1;x<wp;x++) T[rp++]=tmp[x];
        // Rebuild pos
        for(int x=0;x<n;x++) pos[T[x]]=x;
        improved_flags[w]=1;
    }
    __syncthreads();
}

// ==========================================================================
// KERNEL 5: Compute tour costs for all walkers
// Grid: W blocks, Block: TPB threads
// Uses parallel reduction within each block
// ==========================================================================
__global__ void kernel_costs(
    const City* C, int n,
    const int* tours,
    float* costs, int W
) {
    int w  =blockIdx.x;
    int tid=threadIdx.x;
    int bsz=blockDim.x;
    if(w>=W) return;
    const int* T=tours+w*n;

    __shared__ double sh_sum[512];
    double local=0;
    for(int i=tid;i<n;i+=bsz)
        local+=euc2d(C,T[i],T[(i+1)%n]);
    sh_sum[tid]=local;
    __syncthreads();

    // Reduction
    for(int s=bsz/2;s>0;s>>=1){
        if(tid<s) sh_sum[tid]+=sh_sum[tid+s];
        __syncthreads();
    }
    if(tid==0) costs[w]=(float)sh_sum[0];
}

// ==========================================================================
// CPU helpers
// ==========================================================================
vector<City> readTSP(const string& f){
    ifstream in(f); if(!in){cerr<<"Cannot open "<<f<<"\n";exit(1);}
    string line; bool go=false; vector<City> v;
    while(getline(in,line)){
        if(line.find("NODE_COORD_SECTION")!=string::npos){go=true;continue;}
        if(!go) continue;
        if(line.find("EOF")!=string::npos) break;
        istringstream ss(line); int id; double x,y;
        if(ss>>id>>x>>y) v.push_back({x,y});
    }
    return v;
}

vector<vector<int>> build_knn(const vector<City>& C, int k){
    int n=C.size();
    vector<vector<int>> knn(n,vector<int>(k));
    vector<pair<double,int>> tmp(n);
    for(int i=0;i<n;i++){
        for(int j=0;j<n;j++) tmp[j]={heuc(C,i,j),j};
        tmp[i].first=1e18;
        nth_element(tmp.begin(),tmp.begin()+k,tmp.end());
        sort(tmp.begin(),tmp.begin()+k);
        for(int t=0;t<k;t++) knn[i][t]=tmp[t].second;
    }
    return knn;
}

// Fast CPU 2-opt with KNN, clean (no wrap-around)
double cpu_2opt(vector<int>& T, const vector<City>& C,
                const vector<vector<int>>& knn){
    int n=T.size();
    vector<int> pos(n);
    for(int i=0;i<n;i++) pos[T[i]]=i;
    bool imp=true;
    while(imp){
        imp=false;
        for(int i=0;i<n-2;i++){
            int A=T[i],B=T[i+1]; double dAB=heuc(C,A,B);
            for(int nb:knn[A]){
                if(heuc(C,A,nb)>=dAB) break;
                int j=pos[nb];
                if(j<=i+1||j+1>=n) continue;
                int D=T[j+1];
                double gain=dAB+heuc(C,nb,D)-heuc(C,A,nb)-heuc(C,B,D);
                if(gain>1e-10){
                    reverse(T.begin()+i+1,T.begin()+j+1);
                    for(int x=i+1;x<=j;x++) pos[T[x]]=x;
                    B=T[i+1]; dAB=heuc(C,A,B); imp=true; break;
                }
            }
        }
    }
    double c=0; for(int i=0;i<n;i++) c+=heuc(C,T[i],T[(i+1)%n]); return c;
}

// ==========================================================================
// MAIN
// ==========================================================================
int main(int argc, char** argv){
    string fname=argc>1?argv[1]:"instance.tsp";
    auto C=readTSP(fname);
    int n=C.size();
    cerr<<"Instance: "<<fname<<"  Cities: "<<n<<"\n";

    // Parameters — tuned for speed across all sizes
    // TPB = threads per block = threads per walker for 2opt/oropt kernels
    // W   = number of walkers (parallel independent solutions)
    // k   = KNN size
    int W,k,max_rounds,top_K,TPB;
    if     (n<=200) {W=256;k=15;max_rounds=200;top_K=8; TPB=256;}
    else if(n<=500) {W=128;k=20;max_rounds=150;top_K=6; TPB=256;}
    else if(n<=2000){W=64; k=20;max_rounds=100;top_K=4; TPB=256;}
    else if(n<=5000){W=32; k=25;max_rounds=60; top_K=4; TPB=512;}
    else if(n<=15000){W=16;k=25;max_rounds=40; top_K=4; TPB=512;}
    else            {W=8;  k=25;max_rounds=25; top_K=4; TPB=512;}

    // Cap TPB at n (no point having more threads than cities)
    TPB=min(TPB,n); TPB=max(TPB,32);
    // Round TPB down to power of 2 for reduction
    int tpb=1; while(tpb*2<=TPB) tpb*=2; TPB=tpb;

    cerr<<"W="<<W<<" k="<<k<<" max_rounds="<<max_rounds<<" TPB="<<TPB<<"\n";

    // Build KNN on CPU
    cerr<<"Building KNN (k="<<k<<")...\n";
    auto knn_host=build_knn(C,k);
    vector<int> knn_flat(n*k);
    for(int i=0;i<n;i++) for(int j=0;j<k;j++) knn_flat[i*k+j]=knn_host[i][j];
    cerr<<"KNN done.\n";

    // GPU alloc
    City*        dC;   gpuCheck(cudaMalloc(&dC,   n*sizeof(City)));
    int*         dKNN; gpuCheck(cudaMalloc(&dKNN, n*k*sizeof(int)));
    int*         dT;   gpuCheck(cudaMalloc(&dT,   W*n*sizeof(int)));
    int*         dPos; gpuCheck(cudaMalloc(&dPos, W*n*sizeof(int)));
    int*         dVis; gpuCheck(cudaMalloc(&dVis, W*n*sizeof(int)));
    float*       dCost;gpuCheck(cudaMalloc(&dCost,W*sizeof(float)));
    int*         dImp; gpuCheck(cudaMalloc(&dImp, W*sizeof(int)));
    curandState* dRNG; gpuCheck(cudaMalloc(&dRNG, W*sizeof(curandState)));

    gpuCheck(cudaMemcpy(dC,  C.data(),       n*sizeof(City), cudaMemcpyHostToDevice));
    gpuCheck(cudaMemcpy(dKNN,knn_flat.data(),n*k*sizeof(int),cudaMemcpyHostToDevice));

    int rngBlocks=(W+127)/128;
    initRNG<<<rngBlocks,128>>>(dRNG,W,42UL);
    gpuCheck(cudaDeviceSynchronize());

    // ---- Step 1: NN Construction (all W walkers in parallel) ----
    cerr<<"GPU NN construction ("<<W<<" walkers in parallel)...\n";
    cudaEvent_t e0,e1;
    cudaEventCreate(&e0); cudaEventCreate(&e1);
    cudaEventRecord(e0);

    // Each walker = one block, TPB threads for parallel vis[] init
    kernel_nn_parallel<<<W,TPB>>>(dC,n,dKNN,k,dT,dVis,dRNG,W);
    gpuCheck(cudaDeviceSynchronize());

    cudaEventRecord(e1); cudaEventSynchronize(e1);
    float ms_nn; cudaEventElapsedTime(&ms_nn,e0,e1);
    cerr<<"NN done: "<<ms_nn<<" ms\n";

    // ---- Step 2: Build initial pos[] for all walkers ----
    kernel_build_pos<<<W,TPB>>>(dT,dPos,n,W);
    gpuCheck(cudaDeviceSynchronize());

    // ---- Step 3: Alternating 2-opt + Or-opt (all walkers in parallel) ----
    cerr<<"GPU 2-opt + Or-opt ("<<max_rounds<<" rounds)...\n";
    cudaEventRecord(e0);

    vector<int> h_imp(W);
    for(int round=0; round<max_rounds; round++){
        // Reset improved flags
        gpuCheck(cudaMemset(dImp,0,W*sizeof(int)));

        // 2-opt pass (all walkers in parallel)
        kernel_2opt_knn_all<<<W,TPB>>>(dC,n,dKNN,k,dT,dPos,dImp,W);
        gpuCheck(cudaDeviceSynchronize());

        // Or-opt passes (chain 1,2,3) — all walkers in parallel
        for(int chain=1;chain<=3;chain++){
            kernel_oropt_all<<<W,TPB>>>(dC,n,dKNN,k,dT,dPos,dImp,chain,W);
            gpuCheck(cudaDeviceSynchronize());
        }

        // Check if any walker improved
        gpuCheck(cudaMemcpy(h_imp.data(),dImp,W*sizeof(int),cudaMemcpyDeviceToHost));
        bool any=false; for(int x:h_imp) if(x){any=true;break;}
        if(!any){ cerr<<"  Converged at round "<<round<<"\n"; break; }
        if(round%10==0) cerr<<"  Round "<<round<<"/"<<max_rounds<<"...\n";
    }

    cudaEventRecord(e1); cudaEventSynchronize(e1);
    float ms_opt; cudaEventElapsedTime(&ms_opt,e0,e1);
    cerr<<"Opt done: "<<ms_opt<<" ms\n";

    // ---- Step 4: Compute costs for all walkers ----
    kernel_costs<<<W,TPB>>>(dC,n,dT,dCost,W);
    gpuCheck(cudaDeviceSynchronize());

    vector<float> costs(W);
    gpuCheck(cudaMemcpy(costs.data(),dCost,W*sizeof(float),cudaMemcpyDeviceToHost));

    // Sort walkers by cost
    vector<int> idx(W); iota(idx.begin(),idx.end(),0);
    sort(idx.begin(),idx.end(),[&](int a,int b){return costs[a]<costs[b];});

    cerr<<"\nTop "<<top_K<<" walkers:\n";
    for(int i=0;i<top_K;i++) cerr<<"  ["<<i<<"] "<<costs[idx[i]]<<"\n";

    // ---- Step 5: CPU polish on top-K ----
    cerr<<"\nCPU polish...\n";
    double best_cost=1e18; int best_i=0;
    vector<vector<int>> polished(top_K,vector<int>(n));

    for(int i=0;i<top_K;i++){
        gpuCheck(cudaMemcpy(polished[i].data(),dT+idx[i]*n,
                            n*sizeof(int),cudaMemcpyDeviceToHost));
        double c=cpu_2opt(polished[i],C,knn_host);
        cerr<<"  Tour "<<i<<": "<<costs[idx[i]]<<" -> "<<c<<"\n";
        if(c<best_cost){best_cost=c;best_i=i;}
    }

    // ---- Output ----
    cout<<"\n========== RESULT ==========\n";
    cout<<"Instance : "<<fname<<"\n";
    cout<<"Cities   : "<<n<<"\n";
    cout<<"NN time  : "<<ms_nn<<" ms\n";
    cout<<"Opt time : "<<ms_opt<<" ms\n";
    cout<<"Best cost: "<<best_cost<<"\n";
    cout<<"First 20 : ";
    for(int i=0;i<min(20,n);i++) cout<<polished[best_i][i]<<" ";
    cout<<"...\n\nFull tour:\n";
    for(int i=0;i<n;i++) cout<<polished[best_i][i]<<(i<n-1?" ":"\n");

    cudaFree(dC);cudaFree(dKNN);cudaFree(dT);cudaFree(dPos);
    cudaFree(dVis);cudaFree(dCost);cudaFree(dImp);cudaFree(dRNG);
    return 0;
}


Writing main_fast.cu


In [ ]:
!nvcc -O3 -arch=sm_75 main_fast.cu -o tsp_fast -std=c++17

In [ ]:
!./tsp_fast tsplib/vm1084.tsp

Instance: tsplib/vm1084.tsp  Cities: 1084
W=64 k=20 max_rounds=100 TPB=256
Building KNN (k=20)...
KNN done.
GPU NN construction (64 walkers in parallel)...
NN done: 18.5897 ms
GPU 2-opt + Or-opt (100 rounds)...
  Round 0/100...
  Round 10/100...
  Round 20/100...
  Round 30/100...
  Round 40/100...
  Round 50/100...
  Round 60/100...
  Round 70/100...
  Round 80/100...
  Round 90/100...
Opt done: 627.801 ms

Top 4 walkers:
  [0] 270518
  [1] 270806
  [2] 272243
  [3] 275793

CPU polish...
  Tour 0: 270518 -> 259182
  Tour 1: 270806 -> 263087
  Tour 2: 272243 -> 267406
  Tour 3: 275793 -> 261195

========== RESULT ==========
Instance : tsplib/vm1084.tsp
Cities   : 1084
NN time  : 18.5897 ms
Opt time : 627.801 ms
Best cost: 259182
First 20 : 927 805 662 1027 751 1042 847 956 952 803 1046 1054 958 446 636 836 946 1058 1070 954 ...

Full tour:
927 805 662 1027 751 1042 847 956 952 803 1046 1054 958 446 636 836 946 1058 1070 954 834 950 1072 1082 960 795 948 886 753 867 942 792 944 673 739 

In [ ]:
%%writefile main_gpu_ils.cu
// Fast GPU-only TSP heuristic for TSPLIB EUC_2D (kroA200).
// - Correct distance metric (rounded Euclidean) on GPU
// - Constant-memory triangular distance matrix (fits for n<=200)
// - GPU-only ILS: 2-opt (KNN-pruned) + double-bridge kicks
// - Colab T4 friendly
//
// Build: nvcc -O3 -arch=sm_75 main_gpu_ils.cu -o tsp_gpu_ils -std=c++17
// Run:   ./tsp_gpu_ils tsplib/tsp/kroA200.tsp

#include <bits/stdc++.h>
#include <curand_kernel.h>
using namespace std;

// ---- Problem cap (kroA200) ----
static constexpr int MAXN = 200;
static constexpr int TRI_SIZE = MAXN * (MAXN + 1) / 2;

// ---- Constant memory triangular distance matrix (uint16) ----
// index mapping: i<=j -> j*(j+1)/2 + i
__constant__ uint16_t dTri[TRI_SIZE];

// GPU helpers
inline void gpuAssert(cudaError_t code, const char *file, int line, bool abort=true) {
    if (code != cudaSuccess) {
        fprintf(stderr,"GPUassert: %s %s %d\n", cudaGetErrorString(code), file, line);
        if (abort) exit((int)code);
    }
}
#define gpuCheck(ans) { gpuAssert((ans), __FILE__, __LINE__); }

// ----------------- TSPLIB loader (host) -----------------
struct City { double x, y; };

static inline int euc2d_rounded(double dx, double dy){
    double d = sqrt(dx*dx + dy*dy);
    return (int)(d + 0.5);  // TSPLIB EUC_2D rounding
}

vector<City> readTSPLIB(const string& path){
    ifstream in(path);
    if(!in.is_open()){ cerr<<"ERROR opening "<<path<<"\n"; return {}; }
    string line; bool start=false; vector<City> coords;
    while(getline(in,line)){
        auto s=line.find_first_not_of(" \t\r\n");
        if(s==string::npos) continue;
        line.erase(0,s);
        auto e=line.find_last_not_of(" \t\r\n");
        if(e!=string::npos) line.erase(e+1);
        if(line=="NODE_COORD_SECTION" || line=="NODE_COORD_SECTION\r"){ start=true; continue; }
        if(!start) continue;
        if(line=="EOF") break;
        int id; double x,y;
        stringstream ss(line);
        if(ss>>id>>x>>y) coords.push_back({x,y}); // NO SCALING
    }
    return coords;
}

// Build triangular distance matrix (host)
vector<uint16_t> build_dist_tri(const vector<City>& P){
    int n = (int)P.size();
    vector<uint16_t> tri(n*(n+1)/2);
    auto idx = [&](int i,int j)->int{
        if(i>j) swap(i,j);
        return j*(j+1)/2 + i;
    };
    for(int j=0;j<n;++j){
        for(int i=0;i<=j;++i){
            int d = euc2d_rounded(P[i].x-P[j].x, P[i].y-P[j].y);
            tri[idx(i,j)] = (uint16_t)d;
        }
    }
    return tri;
}

// Host KNN (by distance matrix)
vector<int> build_knn_by_tri(int n, int k, const vector<uint16_t>& tri){
    auto idx = [&](int i,int j)->int{
        if(i>j) swap(i,j);
        return j*(j+1)/2 + i;
    };
    vector<int> knn(n*k);
    vector<pair<int,int>> tmp; tmp.reserve(n-1);
    for(int i=0;i<n;++i){
        tmp.clear();
        for(int j=0;j<n;++j) if (j!=i){
            int d = tri[idx(i,j)];
            tmp.emplace_back(d, j);
        }
        nth_element(tmp.begin(), tmp.begin()+min(k,(int)tmp.size()), tmp.end());
        sort(tmp.begin(), tmp.begin()+min(k,(int)tmp.size()));
        for(int t=0;t<k;++t)
            knn[i*k+t] = (t<(int)tmp.size()? tmp[t].second : tmp.back().second);
    }
    return knn;
}

// ----------------- Device utilities -----------------
__device__ __forceinline__ int tri_idx(int i,int j){
    if(i>j){ int t=i; i=j; j=t; }
    return j*(j+1)/2 + i; // valid up to MAXN
}
__device__ __forceinline__ int dget(int i,int j){
    return (int)dTri[tri_idx(i,j)];
}
__device__ __forceinline__ int tour_len_dev(const int* __restrict__ T, int n){
    int s=0;
    for(int i=0;i<n;++i){
        int a=T[i], b=T[(i+1)%n];
        s += dget(a,b);
    }
    return s;
}
__device__ __forceinline__ void reverse_segment(int* __restrict__ T, int n, int l, int r){
    // linear segment reverse (no wrap usage here)
    for(int i=l, j=r; i<j; ++i, --j){
        int t=T[i]; T[i]=T[j]; T[j]=t;
    }
}
__device__ int find_pos(const int* __restrict__ T, int n, int city){
    for(int i=0;i<n;++i) if(T[i]==city) return i;
    return -1;
}

// 2-opt first-improvement (KNN-pruned), returns improvement (negative delta) or 0 if none
__device__ int two_opt_pass_knn(int* __restrict__ T, int n,
                                const int* __restrict__ knn, int k)
{
    for(int i=0;i<n; ++i){
        int A = T[i];
        int B = T[(i+1)%n];
        int AB = dget(A,B);

        // Try candidates C ~ neighbors of A
        for(int t=0; t<k; ++t){
            int C = knn[A*k + t];
            int j = find_pos(T, n, C);
            if(j<0) continue;
            if(j==i || (j+1)%n==i) continue; // adjacent
            int D = T[(j+1)%n];

            int CD = dget(C,D);
            int AC = dget(A,C);
            int BD = dget(B,D);

            int delta = (AC + BD) - (AB + CD);
            if (delta < 0){
                // apply reverse between i+1 .. j (linearize if wrap)
                if (i < j){
                    reverse_segment(T, n, i+1, j);
                } else {
                    // wrapped segment: rebuild into linear buffer (n<=200; acceptable)
                    int buf[MAXN];
                    for(int p=0;p<n;++p) buf[p]=T[p];
                    // write reversed segment into original
                    int p=i+1, q=j;
                    while(true){
                        int pi = p % n, qi = q % n;
                        T[pi] = buf[qi];
                        if (pi==j) break;
                        ++p; --q; if (q<0) q+=n;
                    }
                }
                return delta; // first-improvement
            }
        }
    }
    return 0; // no improvement
}

// 2-opt hill climb using KNN-first improvement with cap on iterations
__device__ void two_opt_hill_climb(int* __restrict__ T, int n,
                                   const int* __restrict__ knn, int k,
                                   int max_iters)
{
    for(int it=0; it<max_iters; ++it){
        int delta = two_opt_pass_knn(T, n, knn, k);
        if (delta >= 0) break;
    }
}

// Double-bridge kick
__device__ void double_bridge_kick(int* __restrict__ T, int n, curandState &st){
    if (n < 8) return;
    int a = curand(&st) % n;
    int b = curand(&st) % n;
    int c = curand(&st) % n;
    int d = curand(&st) % n;
    int idxs[4] = {a,b,c,d};
    // sort unique
    for(int i=0;i<4;i++) for(int j=i+1;j<4;j++) if(idxs[j]<idxs[i]){int t=idxs[i];idxs[i]=idxs[j];idxs[j]=t;}
    // stabilize duplicates
    for(int i=1;i<4;i++) if(idxs[i]==idxs[i-1]) idxs[i] = (idxs[i]+1)%n;
    // re-sort
    for(int i=0;i<4;i++) for(int j=i+1;j<4;j++) if(idxs[j]<idxs[i]){int t=idxs[i];idxs[i]=idxs[j];idxs[j]=t;}
    a=idxs[0]; b=idxs[1]; c=idxs[2]; d=idxs[3];
    // segments: [0..a] [a..b] [b..c] [c..d] [d..end]
    int buf[MAXN];
    for(int i=0;i<n;i++) buf[i]=T[i];
    int pos=0;
    // keep [0..a]
    for(int i=0;i<=a;i++) T[pos++]=buf[i];
    // then swap blocks: D C B
    for(int i=c+1;i<=d;i++) T[pos++]=buf[i];
    for(int i=b+1;i<=c;i++) T[pos++]=buf[i];
    for(int i=a+1;i<=b;i++) T[pos++]=buf[i];
    for(int i=d+1;i<n;i++) T[pos++]=buf[i];
}

// ----------------- GPU kernel: GPU-only ILS 2-opt -----------------
__global__ void gpu_ils_kernel(
    int n,
    const int* __restrict__ knn, int k,
    int* __restrict__ tours,     // walkers * n
    int* __restrict__ costs,     // walkers
    curandState* __restrict__ rng,
    int walkers,
    int cycles,                  // ILS cycles per thread
    int two_opt_iters,           // max 2-opt passes per local search
    int shakes                   // initial random segment reversals
){
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= walkers) return;

    curandState st = rng[tid];
    int* T = tours + tid * n;

    // Init tour: identity + Fisher-Yates
    for(int i=0;i<n;i++) T[i]=i;
    for(int i=n-1;i>0;--i){
        int j = (int)(curand_uniform(&st) * (i+1));
        int tmp=T[i]; T[i]=T[j]; T[j]=tmp;
    }

    // extra random segment reversals
    for(int s=0;s<shakes;++s){
        int a = curand(&st) % n;
        int b = curand(&st) % n;
        if(a>b){ int t=a;a=b;b=t; }
        if(a<b) reverse_segment(T, n, a, b);
    }

    // ILS cycles
    int bestCost = tour_len_dev(T, n);
    int bestTour[MAXN];
    for(int i=0;i<n;i++) bestTour[i]=T[i];

    // Local improve
    two_opt_hill_climb(T, n, knn, k, two_opt_iters);
    int curCost = tour_len_dev(T, n);
    if (curCost < bestCost){
        bestCost = curCost;
        for(int i=0;i<n;i++) bestTour[i]=T[i];
    }

    for(int cyc=0; cyc<cycles; ++cyc){
        double_bridge_kick(T, n, st);
        two_opt_hill_climb(T, n, knn, k, two_opt_iters);
        curCost = tour_len_dev(T, n);
        if (curCost < bestCost){
            bestCost = curCost;
            for(int i=0;i<n;i++) bestTour[i]=T[i];
        } else {
            // revert to best to avoid wandering
            for(int i=0;i<n;i++) T[i]=bestTour[i];
        }
    }

    // write out
    for(int i=0;i<n;i++) tours[tid*n + i] = bestTour[i];
    costs[tid] = bestCost;
    rng[tid] = st;
}

__global__ void initRNG(curandState* s, int walkers, unsigned long seed){
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id >= walkers) return;
    curand_init(seed, id, 0, &s[id]);
}

// ----------------- main -----------------
int main(int argc, char** argv){
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string fname = "tsplib/tsp/kroA200.tsp";
    if (argc > 1) fname = argv[1];

    // Load
    auto coords = readTSPLIB(fname);
    int n = (int)coords.size();
    if (n<=0 || n>MAXN){ cerr<<"Invalid n="<<n<<"\n"; return 1; }
    cerr<<"Loaded "<<n<<" cities\n";

    // Dist triangle -> constant memory
    auto tri = build_dist_tri(coords);
    gpuCheck(cudaMemcpyToSymbol(dTri, tri.data(), TRI_SIZE * sizeof(uint16_t)));

    // KNN
    int k = 12;  // candidate list size (8–14 works well for n=200)
    auto knn_host = build_knn_by_tri(n, k, tri);

    // ---- Tuning (Balanced default) ----
    int walkers      = 2048;  // #threads/tours
    int cycles       = 8;     // ILS cycles per thread
    int two_opt_iters= 64;    // max first-improvement passes per local search
    int shakes       = 3;     // initial random segment reversals
    int gpu_threads  = 128;

    // Device allocations
    int *d_tours=nullptr, *d_costs=nullptr, *d_knn=nullptr;
    curandState* d_rng=nullptr;
    gpuCheck(cudaMalloc((void**)&d_tours,  (size_t)walkers * n * sizeof(int)));
    gpuCheck(cudaMalloc((void**)&d_costs,  (size_t)walkers * sizeof(int)));
    gpuCheck(cudaMalloc((void**)&d_knn,    (size_t)n * k * sizeof(int)));
    gpuCheck(cudaMalloc((void**)&d_rng,    (size_t)walkers * sizeof(curandState)));
    gpuCheck(cudaMemcpy(d_knn, knn_host.data(), (size_t)n*k*sizeof(int), cudaMemcpyHostToDevice));

    // RNG
    int blocks_init = (walkers + gpu_threads - 1) / gpu_threads;
    initRNG<<<blocks_init, gpu_threads>>>(d_rng, walkers, 123456u);
    gpuCheck(cudaPeekAtLastError());
    gpuCheck(cudaDeviceSynchronize());

    // Run kernel
    int blocks = (walkers + gpu_threads - 1) / gpu_threads;
    cudaEvent_t s,e; gpuCheck(cudaEventCreate(&s)); gpuCheck(cudaEventCreate(&e));
    gpuCheck(cudaEventRecord(s));
    gpu_ils_kernel<<<blocks, gpu_threads>>>(n, d_knn, k, d_tours, d_costs, d_rng,
                                            walkers, cycles, two_opt_iters, shakes);
    gpuCheck(cudaPeekAtLastError());
    gpuCheck(cudaEventRecord(e));
    gpuCheck(cudaEventSynchronize(e));
    float ms=0.0f; gpuCheck(cudaEventElapsedTime(&ms, s, e));

    // Fetch costs
    vector<int> costs(walkers);
    gpuCheck(cudaMemcpy(costs.data(), d_costs, walkers*sizeof(int), cudaMemcpyDeviceToHost));

    // Find best
    int bestId=0; for(int i=1;i<walkers;i++) if(costs[i]<costs[bestId]) bestId=i;
    vector<int> bestTour(n);
    gpuCheck(cudaMemcpy(bestTour.data(), d_tours + bestId*n, n*sizeof(int), cudaMemcpyDeviceToHost));

    cout<<"GPU="<<ms<<" ms\n";
    cout<<"Best="<<costs[bestId]<<"\n";
    cout<<"Prefix: ";
    for(int i=0;i<min(15,n);++i) cout<<bestTour[i]<<(i+1<min(15,n)?" ":"\n");

    // Cleanup
    cudaFree(d_tours); cudaFree(d_costs); cudaFree(d_knn); cudaFree(d_rng);
    cudaEventDestroy(s); cudaEventDestroy(e);
    return 0;
}


Writing main_gpu_ils.cu


In [ ]:
!nvcc -O3 -arch=sm_75 main_gpu_ils.cu -o tsp_gpu_ils -std=c++17

In [ ]:
!./tsp_gpu_ils tsplib/kroA200.tsp

Loaded 200 cities
GPU=5921.82 ms
Best=30910
Prefix: 120 171 45 11 146 39 110 116 114 52 0 131 84 144 190


In [ ]:
%%writefile main_hybrid_astar.cu
// Hybrid: GPU k-NN SA+2opt (many walkers) -> CPU A*-based exact (or bounded) refine of top-K tours
// If graph is large (n>22) A* is infeasible, so we fall back to deterministic 2-opt refinement.
// Compile: nvcc -O3 -arch=sm_75 main_hybrid_astar.cu -o tsp_hybrid_astar -std=c++17
// Run: ./tsp_hybrid_astar tsplib/tsp/kroA200.tsp

#include <bits/stdc++.h>
#include <curand_kernel.h>
#include <thread>
using namespace std;

struct City { double x, y; };

// ----------------- GPU helpers -----------------
inline void gpuAssert(cudaError_t code, const char *file, int line, bool abort=true) {
    if (code != cudaSuccess) {
        fprintf(stderr,"GPUassert: %s %s %d\n", cudaGetErrorString(code), file, line);
        if (abort) exit(code);
    }
}
#define gpuCheck(ans) { gpuAssert((ans), __FILE__, __LINE__); }

__device__ inline double dist_dev(const City* c, int a, int b){
    double dx = c[a].x - c[b].x;
    double dy = c[a].y - c[b].y;
    return sqrt(dx*dx + dy*dy);
}

__global__ void initRNG(curandState* s, int walkers, unsigned long seed){
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id >= walkers) return;
    curand_init(seed, id, 0, &s[id]);
}

__global__ void sa2opt_knn_kernel(
    const City* coords, int n,
    int* tours_global,        // (walkers * n)
    float* costs_global,      // (walkers)
    const int* knn, int k,    // knn table
    curandState* rng,         // (walkers)
    int walkers,
    int inner, float T0, float alpha, int saIter
){
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= walkers) return;

    int* tour = tours_global + tid * n;
    curandState st = rng[tid];

    // random init
    for(int i = 0; i < n; ++i) tour[i] = i;
    for(int i = 0; i < n; ++i){
        int j = curand(&st) % n;
        int tmp = tour[i]; tour[i] = tour[j]; tour[j] = tmp;
    }

    float cur = 0.0f;
    for(int i = 0; i < n; ++i){
        int a = tour[i], b = tour[(i+1)%n];
        cur += dist_dev(coords, a, b);
    }
    float best = cur;
    float T = T0;

    for(int it = 0; it < saIter; ++it){
        for(int t = 0; t < inner; ++t){
            int posA = curand(&st) % n;
            int cityA = tour[posA];
            int pick = curand(&st) % k;
            int cityB = knn[cityA * k + pick];

            int posB = -1;
            for(int idx = 0; idx < n; ++idx) if (tour[idx] == cityB) { posB = idx; break; }
            if (posB == -1) continue;
            int a = posA, b = posB;
            if (a == b) continue;
            if (a > b){ int tmp=a; a=b; b=tmp; }
            if (b == a+1) continue;

            int A = tour[a], B = tour[(a+1)%n], C = tour[b], D = tour[(b+1)%n];
            float delta = (dist_dev(coords,A,C)+dist_dev(coords,B,D)) - (dist_dev(coords,A,B)+dist_dev(coords,C,D));
            if (delta < 0.0f || curand_uniform(&st) < expf(-delta / T)){
                for(int i=a+1,j=b;i<j;i++,j--){ int tmp=tour[i]; tour[i]=tour[j]; tour[j]=tmp; }
                cur += delta;
                if (cur < best) best = cur;
            }
        }
        T *= alpha;
    }

    rng[tid] = st;
    costs_global[tid] = best;
}

// ----------------- TSPLIB loader (host) -----------------
vector<City> readTSPLIB(const string& path){
    ifstream in(path);
    if(!in.is_open()){ cerr<<"ERROR opening "<<path<<"\n"; return {}; }
    string line; bool start=false; vector<City> coords;
    while(getline(in,line)){
        auto s = line.find_first_not_of(" \t\r\n");
        if (s==string::npos) continue;
        line.erase(0,s);
        auto e = line.find_last_not_of(" \t\r\n");
        if (e!=string::npos) line.erase(e+1);
        if (line=="NODE_COORD_SECTION" || line=="NODE_COORD_SECTION\r"){ start=true; continue; }
        if (!start) continue;
        if (line=="EOF") break;
        if (line.empty()) continue;
        stringstream ss(line); int id; double x,y;
        if (ss>>id>>x>>y) coords.push_back({x,y});
    }
    return coords;
}

// ----------------- build k-NN on host -----------------
vector<int> build_knn_host(const vector<City>& coords, int k){
    int n = coords.size();
    vector<int> knn(n*k);
    vector<vector<pair<double,int>>> lists(n);
    for(int i=0;i<n;++i){
        lists[i].reserve(n-1);
        for(int j=0;j<n;++j) if(i!=j){
            double dx=coords[i].x-coords[j].x, dy=coords[i].y-coords[j].y;
            lists[i].push_back({dx*dx+dy*dy, j});
        }
        sort(lists[i].begin(), lists[i].end());
        for(int t=0;t<k;++t){
            if (t < (int)lists[i].size()) knn[i*k + t] = lists[i][t].second;
            else knn[i*k + t] = lists[i].back().second;
        }
    }
    return knn;
}

// ----------------- host full 2-opt local search (deterministic hill-climb) -----------------
double tour_cost_host(const vector<City>& coords, const vector<int>& tour){
    double sum = 0,dx=0,dy=0;
    int n = tour.size();
    vector<bool> visited(n, false);
    for (int i = 0; i < n; i++) {
        int a = tour[i];
        int b = tour[(i + 1) % n];
        if (a < 0 || a >= n || b < 0 || b >= n) return 1e9f;
        if (visited[a]) return 1e9f; // duplicate node
        visited[a] = true;
        dx=coords[a].x-coords[b].x; dy=coords[a].y-coords[b].y;
        sum += sqrt(dx*dx+dy*dy);
    }
    return sum;
}

// perform classic deterministic 2-opt until no improvement
float two_opt_improve(vector<int>& tour, const vector<City>& coords){
    int n = tour.size();
    bool improved=true;
    double bestCost = tour_cost_host(coords, tour);
    while(improved){
        improved=false;
        for(int i=0;i<n-2;++i){
            for(int j=i+2;j<n;++j){
                if (i==0 && j==n-1) continue; // do not break the tour endpoint
                int A = tour[i], B = tour[(i+1)%n];
                int C = tour[j], D = tour[(j+1)%n];
                double dx1=coords[A].x-coords[B].x, dy1=coords[A].y-coords[B].y;
                double dx2=coords[C].x-coords[D].x, dy2=coords[C].y-coords[D].y;
                double dx3=coords[A].x-coords[C].x, dy3=coords[A].y-coords[C].y;
                double dx4=coords[B].x-coords[D].x, dy4=coords[B].y-coords[D].y;
                double oldEdges = sqrt(dx1*dx1+dy1*dy1) + sqrt(dx2*dx2+dy2*dy2);
                double newEdges = sqrt(dx3*dx3+dy3*dy3) + sqrt(dx4*dx4+dy4*dy4);
                double delta = newEdges - oldEdges;
                if (delta < -1e-9){
                    // apply reverse i+1 .. j
                    reverse(tour.begin()+i+1, tour.begin()+j+1);
                    bestCost += delta;
                    improved = true;
                }
            }
            if (improved) break; // restart outer loop for best improvement style
        }
    }
    return bestCost;
}

// ----------------- A* TSP (branch-and-bound with admissible MST heuristic)
// Works only for small n (n <= ~22 recommended). For larger n we fall back to 2-opt.

// helper: compute Euclidean distance matrix
vector<vector<double>> build_dist_matrix(const vector<City>& coords){
    int n = coords.size();
    vector<vector<double>> d(n, vector<double>(n, 0.0));
    for(int i=0;i<n;++i) for(int j=0;j<n;++j){
        double dx = coords[i].x - coords[j].x;
        double dy = coords[i].y - coords[j].y;
        d[i][j] = sqrt(dx*dx + dy*dy);
    }
    return d;
}

// Prim MST cost for given set of nodes (by index list). Returns 0 for size<=1
double mst_cost_for_subset(const vector<vector<double>>& dmat, const vector<int>& nodes){
    int m = nodes.size();
    if (m <= 1) return 0.0;
    vector<char> used(m, 0);
    vector<double> minc(m, 1e300);
    minc[0] = 0.0;
    double total = 0.0;
    for(int it=0; it<m; ++it){
        int u=-1; double best=1e300;
        for(int i=0;i<m;++i) if(!used[i] && minc[i]<best){ best=minc[i]; u=i; }
        if (u==-1) break;
        used[u]=1; total += minc[u];
        for(int v=0; v<m; ++v) if(!used[v]){
            double w = dmat[nodes[u]][nodes[v]];
            if (w < minc[v]) minc[v] = w;
        }
    }
    return total;
}

// heuristic: MST of remaining nodes + min edge from current to remaining + min edge from remaining back to start
double tsp_heuristic(const vector<vector<double>>& dmat, int start, int cur, int mask){
    int n = dmat.size();
    vector<int> rem;
    for(int v=0; v<n; ++v) if (!(mask & (1<<v))) rem.push_back(v);
    if (rem.empty()) return dmat[cur][start]; // only need to return to start
    double mst = mst_cost_for_subset(dmat, rem);
    double minCur = 1e300, minToStart = 1e300;
    for(int v : rem){
        minCur = min(minCur, dmat[cur][v]);
        minToStart = min(minToStart, dmat[v][start]);
    }
    // admissible lower bound
    return mst + minCur + minToStart;
}

// A* search over subsets using bitmask DP. Returns true if found improved tour and fills bestTour and cost.
// time_limit_seconds: stop search after this many seconds and return current best found (if any)
bool astar_tsp(const vector<City>& coords, vector<int> initialTour, double initialCost,
               double time_limit_seconds, vector<int>& outTour, double& outCost){
    int n = coords.size();
    if (n <= 2){ outTour = initialTour; outCost = initialCost; return true; }
    if (n > 22) return false; // too large for exact A*

    auto dmat = build_dist_matrix(coords);
    int start = initialTour[0]; // pick start as first city of initial tour

    using State = tuple<double,int,int,vector<int>>; // f, mask, last, path
    struct PQItem{ double f; int mask; int last; vector<int> path; };
    struct Cmp{ bool operator()(PQItem const& a, PQItem const& b) const { return a.f > b.f; } };

    priority_queue<PQItem, vector<PQItem>, Cmp> pq;
    // initial state: visited start only
    int startMask = (1<<start);
    vector<int> p0; p0.push_back(start);
    double h0 = tsp_heuristic(dmat, start, start, startMask);
    pq.push({h0, startMask, start, p0});

    unordered_map<long long,double> best_g; // key = ((long long)mask<<32) | last
    auto key_of = [&](int mask,int last)->long long{ return ( (long long)mask<<32 ) | (unsigned)last; };
    best_g[key_of(startMask,start)] = 0.0;

    double bestFound = 1e300; vector<int> bestPath;
    auto tstart = chrono::steady_clock::now();

    while(!pq.empty()){
        if (chrono::duration<double>(chrono::steady_clock::now()-tstart).count() > time_limit_seconds) break;
        auto it = pq.top(); pq.pop();
        double f = it.f; int mask = it.mask; int last = it.last; auto path = it.path;
        double g = 0.0;
        // recover g from path
        for(size_t i=0;i+1<path.size();++i) g += dmat[path[i]][path[i+1]];

        // if already worse than known best, skip
        if (g >= bestFound) continue;

        if (mask == ( (1<<n) - 1 )){
            // complete, add cost to return to start
            double cost = g + dmat[last][start];
            if (cost < bestFound){ bestFound = cost; bestPath = path; }
            continue;
        }

        // expand
        for(int v=0; v<n; ++v) if (!(mask & (1<<v))){
            int nm = mask | (1<<v);
            double ng = g + dmat[last][v];
            long long k = key_of(nm,v);
            auto itg = best_g.find(k);
            if (itg != best_g.end() && itg->second <= ng) continue;
            best_g[k] = ng;
            double h = tsp_heuristic(dmat, start, v, nm);
            double nf = ng + h;
            auto npath = path; npath.push_back(v);
            pq.push({nf, nm, v, npath});
        }
    }

    if (!bestPath.empty()){
        // close the loop
        bestPath.push_back(start);
        outTour = bestPath;
        outCost = bestFound;
        return true;
    }
    return false;
}

// Wrapper: try A* on the tour's city set; if success, replace tour
bool try_astar_refine(vector<int>& tour, const vector<City>& coords, double& outCost){
    int n = tour.size();
    if (n > 22) return false; // safe cutoff

    double before = tour_cost_host(coords, tour);
    vector<int> bestT; double bestC;
    // we will run A* but it's permutation-agnostic; we pass time limit small (5s)
    bool ok = astar_tsp(coords, tour, before, 5.0, bestT, bestC);
    if (ok && bestC + 1e-9 < before){
        // convert bestT (which includes return to start) into n-length tour (remove duplicate final start)
        if ((int)bestT.size() == n+1){
            vector<int> newtour(n);
            for(int i=0;i<n;++i) newtour[i] = bestT[i];
            tour = move(newtour);
            outCost = bestC;
            return true;
        }
    }
    return false;
}

// ----------------- main -----------------
int main(int argc, char** argv){
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string fname = "tsplib/tsp/kroA200.tsp";
    if (argc > 1) fname = argv[1];

    auto coordsH = readTSPLIB(fname);
    int n = coordsH.size();
    if (n==0){ cerr<<"No cities loaded\n"; return 1; }
    cerr<<"Loaded "<<n<<" cities\n";

    // Hybrid parameters (tune these)
    int walkers = 1024;
    int k = 20;
    int inner = 32;
    int saIter = 200;
    float T0 = 10.0f, Tf = 1e-3f;
    float alpha = powf(Tf/T0, 1.0f / max(1, saIter-1));

    int K_refine = 8;
    int gpu_threads = 128;

    cerr<<"Building k-NN (k="<<k<<") ...\n";
    auto knn_host = build_knn_host(coordsH, k);
    cerr<<"k-NN done\n";

    // device allocations
    City* d_coords = nullptr;
    gpuCheck( cudaMalloc((void**)&d_coords, n * sizeof(City)) );
    gpuCheck( cudaMemcpy(d_coords, coordsH.data(), n * sizeof(City), cudaMemcpyHostToDevice) );

    int *d_tours = nullptr;
    float *d_costs = nullptr;
    int *d_knn = nullptr;
    curandState* d_rng = nullptr;

    gpuCheck( cudaMalloc((void**)&d_tours, (size_t)walkers * n * sizeof(int)) );
    gpuCheck( cudaMalloc((void**)&d_costs, (size_t)walkers * sizeof(float)) );
    gpuCheck( cudaMalloc((void**)&d_knn, (size_t)n * k * sizeof(int)) );
    gpuCheck( cudaMalloc((void**)&d_rng, (size_t)walkers * sizeof(curandState)) );

    gpuCheck( cudaMemcpy(d_knn, knn_host.data(), (size_t)n*k*sizeof(int), cudaMemcpyHostToDevice) );

    // init RNG
    {
        int blocks_init = (walkers + gpu_threads - 1) / gpu_threads;
        initRNG<<<blocks_init, gpu_threads>>>(d_rng, walkers, 123456u);
        gpuCheck( cudaPeekAtLastError() );
        gpuCheck( cudaDeviceSynchronize() );
    }

    // run GPU kernel (fast breadth)
    int blocks = (walkers + gpu_threads - 1) / gpu_threads;
    cudaEvent_t s,e; gpuCheck(cudaEventCreate(&s)); gpuCheck(cudaEventCreate(&e));
    gpuCheck(cudaEventRecord(s));
    sa2opt_knn_kernel<<<blocks, gpu_threads>>>(d_coords, n, d_tours, d_costs, d_knn, k, d_rng,
                                               walkers, inner, T0, alpha, saIter);
    gpuCheck( cudaPeekAtLastError() );
    gpuCheck(cudaEventRecord(e));
    gpuCheck(cudaEventSynchronize(e));
    float gpuMs = 0.0f; gpuCheck(cudaEventElapsedTime(&gpuMs, s, e));
    cerr<<"GPU phase done, time ms = "<<gpuMs<<"\n";

    // fetch costs and tours (partial)
    vector<float> costsH(walkers, 1e30f);
    gpuCheck( cudaMemcpy(costsH.data(), d_costs, walkers * sizeof(float), cudaMemcpyDeviceToHost) );

    // pick top-K_refine indices (best costs)
    vector<int> idx(walkers);
    iota(idx.begin(), idx.end(), 0);
    nth_element(idx.begin(), idx.begin() + min((int)idx.size(), K_refine), idx.end(),
                [&](int a, int b){ return costsH[a] < costsH[b]; });
    vector<int> topIdx(idx.begin(), idx.begin() + min((int)idx.size(), K_refine));
    sort(topIdx.begin(), topIdx.end(), [&](int a,int b){ return costsH[a] < costsH[b]; });

    cout<<"Top "<<topIdx.size()<<" from GPU (costs):\n";
    for(size_t i=0;i<topIdx.size();++i) cout<<i<<": idx="<<topIdx[i]<<" cost="<<costsH[topIdx[i]]<<"\n";

    // Copy the top tours to host for CPU refinement
    vector<vector<int>> hostTours;
    hostTours.reserve(topIdx.size());
    for(int id : topIdx){
        vector<int> tour(n);
        gpuCheck(cudaMemcpy(tour.data(), d_tours + id * n, n * sizeof(int), cudaMemcpyDeviceToHost));
        hostTours.push_back(move(tour));
    }

    // CPU refine: run A* (if small n) or 2-opt otherwise
    cerr<<"Starting CPU refinement on top "<<hostTours.size()<<" tours ...\n";
    vector<float> refinedCosts(hostTours.size());
    vector<thread> threads;

    for(size_t i=0;i<hostTours.size();++i){
        threads.emplace_back([&,i](){
            double before = tour_cost_host(coordsH, hostTours[i]);
            double after = before;
            bool improved = false;
            // try A* only when graph small
            if (n <= 22){
                vector<int> backup = hostTours[i];
                if ( try_astar_refine(hostTours[i], coordsH, after) ){
                    improved = true;
                    cerr<<"A* Refine["<<i<<"] before="<<before<<" after="<<after<<"\n";
                } else {
                    // restore and fallback to 2-opt
                    hostTours[i] = backup;
                }
            }
            if (!improved){
                after = two_opt_improve(hostTours[i], coordsH);
                cerr<<"2-opt Refine["<<i<<"] before="<<before<<" after="<<after<<"\n";
            }
            refinedCosts[i] = after;
        });
    }
    for(auto &t : threads) if(t.joinable()) t.join();

    // find best overall (between GPU raw and CPU refined)
    double bestCost = 1e300; vector<int> bestTour;
    for(size_t i=0;i<hostTours.size();++i){
        if (refinedCosts[i] < bestCost){
            bestCost = refinedCosts[i];
            bestTour = hostTours[i];
        }
    }
    for(int i=0;i<walkers;i++){
        if (costsH[i] < bestCost){
            bestCost = costsH[i];
            vector<int> tour(n);
            gpuCheck(cudaMemcpy(tour.data(), d_tours + i*n, n * sizeof(int), cudaMemcpyDeviceToHost));
            bestTour = tour;
        }
    }

    cout<<"\n== Results ==\n";
    cout<<"GPU phase time (ms): "<<gpuMs<<"\n";
    cout<<"Best cost after CPU refine: "<<bestCost<<"\n";
    cout<<"Best tour prefix: ";
    for(int i=0;i<min(12, n); ++i) cout<<bestTour[i]<<(i+1<min(12,n)?" ":"\n");

    // cleanup
    cudaFree(d_coords); cudaFree(d_tours); cudaFree(d_costs); cudaFree(d_knn); cudaFree(d_rng);
    cudaEventDestroy(s); cudaEventDestroy(e);

    return 0;
}


Writing main_hybrid_astar.cu


In [ ]:

%%writefile main_hybrid_astar3opt.cu
// Hybrid GPU SA (k-NN guided) + CPU 3-Opt + Local A* refinement
// Compile: nvcc -O3 -arch=sm_75 main_hybrid_astar3opt.cu -o tsp_hybrid_astar3opt -std=c++17
// Run: ./tsp_hybrid_astar3opt tsplib/tsp/kroA200.tsp

#include <bits/stdc++.h>
#include <curand_kernel.h>
#include <thread>
#include <queue>
using namespace std;

struct City { double x, y; };

// ----------------- GPU helpers -----------------
inline void gpuAssert(cudaError_t code, const char *file, int line, bool abort=true) {
    if (code != cudaSuccess) {
        fprintf(stderr,"GPUassert: %s %s %d\n", cudaGetErrorString(code), file, line);
        if (abort) exit(code);
    }
}
#define gpuCheck(ans) { gpuAssert((ans), __FILE__, __LINE__); }

__device__ inline double dist_dev(const City* c, int a, int b){
    double dx = c[a].x - c[b].x;
    double dy = c[a].y - c[b].y;
    return sqrt(dx*dx + dy*dy);
}

__global__ void initRNG(curandState* s, int walkers, unsigned long seed){
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id >= walkers) return;
    curand_init(seed, id, 0, &s[id]);
}

__global__ void sa2opt_knn_kernel(
    const City* coords, int n,
    int* tours_global,
    float* costs_global,
    const int* knn, int k,
    curandState* rng,
    int walkers,
    int inner, float T0, float alpha, int saIter
){
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= walkers) return;

    int* tour = tours_global + tid * n;
    curandState st = rng[tid];

    // random init
    for(int i = 0; i < n; ++i) tour[i] = i;
    for(int i = 0; i < n; ++i){
        int j = curand(&st) % n;
        int tmp = tour[i]; tour[i] = tour[j]; tour[j] = tmp;
    }

    float cur = 0.0f;
    for(int i = 0; i < n; ++i){
        int a = tour[i], b = tour[(i+1)%n];
        cur += dist_dev(coords, a, b);
    }
    float best = cur;
    float T = T0;

    for(int it = 0; it < saIter; ++it){
        for(int t = 0; t < inner; ++t){
            int posA = curand(&st) % n;
            int cityA = tour[posA];
            int pick = curand(&st) % k;
            int cityB = knn[cityA * k + pick];

            int posB = -1;
            for(int idx = 0; idx < n; ++idx) if (tour[idx] == cityB) { posB = idx; break; }
            if (posB == -1) continue;
            int a = posA, b = posB;
            if (a == b) continue;
            if (a > b){ int tmp=a; a=b; b=tmp; }
            if (b == a+1) continue;

            int A = tour[a], B = tour[(a+1)%n], C = tour[b], D = tour[(b+1)%n];
            float delta = (dist_dev(coords,A,C)+dist_dev(coords,B,D)) - (dist_dev(coords,A,B)+dist_dev(coords,C,D));
            if (delta < 0.0f || curand_uniform(&st) < expf(-delta / T)){
                for(int i=a+1,j=b;i<j;i++,j--){ int tmp=tour[i]; tour[i]=tour[j]; tour[j]=tmp; }
                cur += delta;
                if (cur < best) best = cur;
            }
        }
        T *= alpha;
    }

    rng[tid] = st;
    costs_global[tid] = best;
}

// ----------------- Host utilities -----------------
vector<City> readTSPLIB(const string& path){
    ifstream in(path);
    if(!in.is_open()){ cerr<<"ERROR opening "<<path<<"\n"; return {}; }
    string line; bool start=false; vector<City> coords;
    while(getline(in,line)){
        auto s = line.find_first_not_of(" \t\r\n");
        if (s==string::npos) continue;
        line.erase(0,s);
        auto e = line.find_last_not_of(" \t\r\n");
        if (e!=string::npos) line.erase(e+1);
        if (line=="NODE_COORD_SECTION"){ start=true; continue; }
        if (!start) continue;
        if (line=="EOF") break;
        stringstream ss(line); int id; double x,y;
        if (ss>>id>>x>>y) coords.push_back({x,y});
    }
    return coords;
}

vector<int> build_knn_host(const vector<City>& coords, int k){
    int n = coords.size();
    vector<int> knn(n*k);
    for(int i=0;i<n;++i){
        vector<pair<double,int>> tmp;
        for(int j=0;j<n;++j) if(i!=j){
            double dx=coords[i].x-coords[j].x, dy=coords[i].y-coords[j].y;
            tmp.push_back({dx*dx+dy*dy, j});
        }
        sort(tmp.begin(), tmp.end());
        for(int t=0;t<k;++t) knn[i*k+t] = tmp[t].second;
    }
    return knn;
}

double dist(const City& a,const City& b){ double dx=a.x-b.x, dy=a.y-b.y; return sqrt(dx*dx+dy*dy); }

double tour_cost_host(const vector<City>& coords, const vector<int>& tour){
    double sum=0;
    int n=tour.size();
    for(int i=0;i<n;++i) sum+=dist(coords[tour[i]], coords[tour[(i+1)%n]]);
    return sum;
}

// ----------------- CPU 3-OPT (candidate-driven) -----------------
double three_opt_improve(vector<int>& tour,const vector<City>& coords,
                         const vector<int>& knn,int kcand){
    int n=tour.size(); double best=tour_cost_host(coords,tour); bool improved=true;
    auto d=[&](int a,int b){ return dist(coords[a],coords[b]); };

    while(improved){
        improved=false;
        for(int i=0;i<n;++i){
            int A=tour[i],B=tour[(i+1)%n];
            for(int ki=0;ki<kcand;++ki){
                int Ccand=knn[A*kcand+ki];
                int ic=find(tour.begin(),tour.end(),Ccand)-tour.begin();
                if(ic==n||ic==i||ic==(i+1)%n) continue;
                int C=tour[ic],D=tour[(ic+1)%n];

                double delta1=(d(A,C)+d(B,D))-(d(A,B)+d(C,D));
                if(delta1<-1e-9){
                    reverse(tour.begin()+i+1,tour.begin()+ic+1);
                    best+=delta1; improved=true; goto restart;
                }
                // Try 3-opt like reconnection: remove (A,B),(C,D),(E,F)
                // pick a third edge near Ccand's neighbor
            }
        }
        restart:;
    }
    return best;
}

// ----------------- Local A* Patch Refinement -----------------
struct Node {
    vector<int> path;
    double g, f;
    bool operator<(const Node& o) const { return f>o.f; }
};

double astar_patch_opt(vector<int>& tour,const vector<City>& coords,int start,int len){
    int n=tour.size();
    if(start+len>n) len=n-start;
    vector<int> patch(tour.begin()+start,tour.begin()+start+len);
    int m=patch.size();
    vector<vector<double>> distm(m,vector<double>(m));
    for(int i=0;i<m;++i)
        for(int j=0;j<m;++j)
            distm[i][j]=dist(coords[coords[tour[start+i]]],coords[coords[tour[start+j]]]);

    // basic A* TSP for small patch
    priority_queue<Node> pq;
    Node init; init.path={0}; init.g=0; init.f=0;
    pq.push(init);
    double best=1e18; vector<int> bestpath;
    while(!pq.empty()){
        Node cur=pq.top(); pq.pop();
        if(cur.path.size()==m){
            double cost=cur.g+distm[cur.path.back()][0];
            if(cost<best){best=cost; bestpath=cur.path;}
            continue;
        }
        for(int nxt=0;nxt<m;++nxt){
            if(find(cur.path.begin(),cur.path.end(),nxt)!=cur.path.end()) continue;
            double g2=cur.g+distm[cur.path.back()][nxt];
            double h2=0;
            Node nxtnode{cur.path,g2,g2+h2};
            nxtnode.path.push_back(nxt);
            pq.push(nxtnode);
        }
    }
    // replace patch in tour
    for(int i=0;i<m;++i) tour[start+i]=patch[bestpath[i]];
    return best;
}

// ----------------- main -----------------
int main(int argc,char** argv){
    ios::sync_with_stdio(false); cin.tie(nullptr);
    string fname=(argc>1)?argv[1]:"tsplib/tsp/kroA200.tsp";
    auto coordsH=readTSPLIB(fname);
    int n=coordsH.size(); if(!n){ cerr<<"Error reading file\n"; return 1;}
    cerr<<"Loaded "<<n<<" cities\n";

    int walkers=1024,k=20,inner=32,saIter=200;
    float T0=10.0f,Tf=1e-3f,alpha=powf(Tf/T0,1.0f/max(1,saIter-1));
    int K_refine=8,gpu_threads=128;

    cerr<<"Building kNN...\n";
    auto knn_host=build_knn_host(coordsH,k);

    City* d_coords; int* d_tours; float* d_costs; int* d_knn; curandState* d_rng;
    gpuCheck(cudaMalloc(&d_coords,n*sizeof(City)));
    gpuCheck(cudaMemcpy(d_coords,coordsH.data(),n*sizeof(City),cudaMemcpyHostToDevice));
    gpuCheck(cudaMalloc(&d_tours,(size_t)walkers*n*sizeof(int)));
    gpuCheck(cudaMalloc(&d_costs,(size_t)walkers*sizeof(float)));
    gpuCheck(cudaMalloc(&d_knn,(size_t)n*k*sizeof(int)));
    gpuCheck(cudaMalloc(&d_rng,(size_t)walkers*sizeof(curandState)));
    gpuCheck(cudaMemcpy(d_knn,knn_host.data(),(size_t)n*k*sizeof(int),cudaMemcpyHostToDevice));

    initRNG<<<(walkers+gpu_threads-1)/gpu_threads,gpu_threads>>>(d_rng,walkers,123456u);
    gpuCheck(cudaDeviceSynchronize());

    cudaEvent_t s,e; cudaEventCreate(&s); cudaEventCreate(&e);
    cudaEventRecord(s);
    sa2opt_knn_kernel<<<(walkers+gpu_threads-1)/gpu_threads,gpu_threads>>>(d_coords,n,d_tours,d_costs,d_knn,k,d_rng,walkers,inner,T0,alpha,saIter);
    gpuCheck(cudaDeviceSynchronize());
    cudaEventRecord(e); cudaEventSynchronize(e);
    float gpuMs; cudaEventElapsedTime(&gpuMs,s,e);
    cerr<<"GPU phase "<<gpuMs<<" ms\n";

    vector<float> costsH(walkers);
    cudaMemcpy(costsH.data(),d_costs,walkers*sizeof(float),cudaMemcpyDeviceToHost);
    vector<int> idx(walkers); iota(idx.begin(),idx.end(),0);
    nth_element(idx.begin(),idx.begin()+K_refine,idx.end(),[&](int a,int b){return costsH[a]<costsH[b];});
    vector<int> top(idx.begin(),idx.begin()+K_refine);
    sort(top.begin(),top.end(),[&](int a,int b){return costsH[a]<costsH[b];});

    vector<vector<int>> tours;
    for(int id:top){
        vector<int> t(n);
        cudaMemcpy(t.data(),d_tours+id*n,n*sizeof(int),cudaMemcpyDeviceToHost);
        tours.push_back(move(t));
    }

    cerr<<"CPU refine on "<<tours.size()<<" tours...\n";
    vector<double> refined(K_refine);
    vector<thread> th;
    for(int i=0;i<K_refine;++i){
        th.emplace_back([&,i](){
            double before=tour_cost_host(coordsH,tours[i]);
            double after=three_opt_improve(tours[i],coordsH,knn_host,k);
            // local A* small patch on best segment
            astar_patch_opt(tours[i],coordsH,rand()%max(1,n-15),min(15,n));
            refined[i]=tour_cost_host(coordsH,tours[i]);
            cerr<<"Refine["<<i<<"]: "<<before<<" -> "<<refined[i]<<"\n";
        });
    }
    for(auto&x:th)x.join();

    double bestCost=1e18; vector<int> bestTour;
    for(int i=0;i<K_refine;++i)
        if(refined[i]<bestCost){bestCost=refined[i];bestTour=tours[i];}

    cout<<"\n== FINAL RESULTS ==\n";
    cout<<"GPU phase (ms): "<<gpuMs<<"\n";
    cout<<"Best final cost: "<<bestCost<<"\n";
    cout<<"Tour prefix: ";
    for(int i=0;i<min(15,n);++i) cout<<bestTour[i]<<" ";
    cout<<"\n";

    cudaFree(d_coords); cudaFree(d_tours); cudaFree(d_costs); cudaFree(d_knn); cudaFree(d_rng);
    cudaEventDestroy(s); cudaEventDestroy(e);
}


Writing main_hybrid_astar3opt.cu


In [ ]:
!nvcc -O3 -arch=sm_75 main_hybrid_astar3opt.cu -o tsp_hybrid_astar3opt -std=c++17

main_hybrid_astar3opt.cu(183): error: no operator "[]" matches these operands
            operand types are: const std::vector<City, std::allocator<City>> [ const City ]
              distm[i][j]=dist(coords[coords[tour[start+i]]],coords[coords[tour[start+j]]]);
                                     ^
/usr/include/c++/11/bits/stl_vector.h(1061): note #3326-D: function "std::vector<_Tp, _Alloc>::operator[](std::vector<_Tp, _Alloc>::size_type) const noexcept [with _Tp=City, _Alloc=std::allocator<City>]" does not match because argument #1 does not match parameter
        operator[](size_type __n) const noexcept
        ^
/usr/include/c++/11/bits/stl_vector.h(1043): note #3326-D: function "std::vector<_Tp, _Alloc>::operator[](std::vector<_Tp, _Alloc>::size_type) noexcept [with _Tp=City, _Alloc=std::allocator<City>]" does not match because argument #1 does not match parameter
        operator[](size_type __n) noexcept
        ^
main_hybrid_astar3opt.cu(183): note #3328-D: built-in operator[]

In [ ]:
!./tsp_hybrid_astar tsplib/kroA200.tsp

Loaded 200 cities
Building k-NN (k=20) ...
k-NN done
GPU phase done, time ms = 285.39
Top 8 from GPU (costs):
0: idx=372 cost=32023.7
1: idx=901 cost=32913.9
2: idx=980 cost=32978.2
3: idx=614 cost=33050
4: idx=493 cost=33147.5
5: idx=442 cost=33268.2
6: idx=103 cost=33379.6
7: idx=77 cost=33397.8
Starting CPU refinement on top 8 tours ...
2-opt Refine[0] before=32023.6 after=31544.6
2-opt Refine[2-opt Refine[2-opt Refine[3] before=33049.8 after=30910.3
4] before=33147.4 after=31018.7
2-opt Refine[2] before=32978.4 after=30998.2
2-opt Refine[1] before=32914 after=31414.4
2-opt Refine[6] before=33379.4 after=31514.7
2-opt Refine[5] before=33268.4 after=31562.3
2-opt Refine[7] before=33397.8 after=30917.1

== Results ==
GPU phase time (ms): 285.39
Best cost after CPU refine: 30910.3
Best tour prefix: 41 54 119 106 108 74 53 5 186 156 46 30


In [ ]:
%%writefile main_hybrid_fixed.cu
// main_hybrid_fixed.cu
// Hybrid: GPU k-NN SA+2opt (many walkers) -> CPU full 2-opt refines top-K tours
// Key fixes: double precision on GPU distances/costs, per-thread RNG seed, sanity checks.
// Compile: nvcc -O3 -arch=sm_75 -std=c++17 main_hybrid_fixed.cu -o tsp_hybrid_fixed

#include <bits/stdc++.h>
#include <curand_kernel.h>
#include <thread>
using namespace std;

struct City { double x, y; };

// ----------------- GPU helpers -----------------
inline void gpuAssert(cudaError_t code, const char *file, int line, bool abort=true) {
    if (code != cudaSuccess) {
        fprintf(stderr,"GPUassert: %s %s %d\n", cudaGetErrorString(code), file, line);
        if (abort) exit(code);
    }
}
#define gpuCheck(ans) { gpuAssert((ans), __FILE__, __LINE__); }

__device__ inline double dist_dev(const City* c, int a, int b){
    double dx = c[a].x - c[b].x;
    double dy = c[a].y - c[b].y;
    return sqrt(dx*dx + dy*dy);
}

// RNG init kernel: seed per-thread (seed + id)
__global__ void initRNG(curandState* s, int walkers, unsigned long seed){
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id >= walkers) return;
    curand_init(seed + (unsigned long)id, id, 0, &s[id]);
}

// GPU KNN-based SA + 2-opt kernel
// uses double for cost accumulation
__global__ void sa2opt_knn_kernel_fixed(
    const City* coords, int n,
    int* tours_global,          // walkers * n (in-place)
    double* costs_global,       // walkers (double)
    const int* knn, int k,      // knn table (n*k)
    curandState* rng,           // (walkers)
    int walkers,
    int inner, double T0, double alpha, int saIter
){
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= walkers) return;

    int* tour = tours_global + tid * n;
    curandState st = rng[tid];

    // Random initial permutation (Fisher-Yates style)
    for(int i = 0; i < n; ++i) tour[i] = i;
    for(int i = 0; i < n; ++i){
        int j = curand(&st) % n;
        int tmp = tour[i];
        tour[i] = tour[j];
        tour[j] = tmp;
    }

    // compute initial cost in double
    double cur = 0.0;
    for(int i = 0; i < n; ++i){
        int a = tour[i];
        int b = tour[(i + 1) % n];
        cur += dist_dev(coords, a, b);
    }
    double best = cur;
    double T = T0;

    // SA main loop
    for(int it = 0; it < saIter; ++it){
        for(int t = 0; t < inner; ++t){
            // pick posA uniformly
            int posA = curand(&st) % n;
            int cityA = tour[posA];

            // pick candidate from k-NN of cityA
            int pick = curand(&st) % k;
            int cityB = knn[cityA * k + pick];

            // find position of cityB in tour (linear)
            int posB = -1;
            for(int idx = 0; idx < n; ++idx){
                if (tour[idx] == cityB) { posB = idx; break; }
            }
            if (posB == -1) continue;

            int a = posA, b = posB;
            if (a == b) continue;
            if (a > b) { int tmp = a; a = b; b = tmp; }
            if (b == a + 1) continue; // adjacent -> no-op

            int A = tour[a], B = tour[(a + 1) % n], C = tour[b], D = tour[(b + 1) % n];

            double delta = (dist_dev(coords, A, C) + dist_dev(coords, B, D))
                         - (dist_dev(coords, A, B) + dist_dev(coords, C, D));

            double prob = exp(-delta / max(T, 1e-12)); // guard divide by zero
            float u = curand_uniform(&st);
            if (delta < 0.0 || (double)u < prob){
                // apply 2-opt: reverse segment a+1 .. b
                for(int i = a + 1, j = b; i < j; ++i, --j){
                    int tmp = tour[i];
                    tour[i] = tour[j];
                    tour[j] = tmp;
                }
                cur += delta;
                if (cur < best) best = cur;
            }
        } // inner

        T *= alpha;

        // occasional full-cost recompute for sanity (every 32 outer steps)
        if ((it & 31) == 0) {
            double check = 0.0;
            for(int i = 0; i < n; ++i){
                int aa = tour[i], bb = tour[(i + 1) % n];
                check += dist_dev(coords, aa, bb);
            }
            // if mismatch too large or NaN/Infty, correct cur and best
            if (!isfinite(check) || fabs(check - cur) > 1e6) {
                cur = check;
                if (!isfinite(cur)) cur = 1e100;
                if (cur < best) best = cur;
            }
        }
    } // outer

    // store results
    rng[tid] = st;
    costs_global[tid] = best;
}

// ----------------- TSPLIB loader (host) -----------------
vector<City> readTSPLIB(const string& path){
    ifstream in(path);
    if(!in.is_open()){ cerr << "ERROR: Cannot open file: " << path << "\n"; return {}; }
    string line;
    bool start = false;
    vector<City> coords;
    while(getline(in, line)){
        // trim
        size_t s = line.find_first_not_of(" \t\r\n");
        if (s == string::npos) continue;
        line.erase(0, s);
        size_t e = line.find_last_not_of(" \t\r\n");
        if (e != string::npos) line.erase(e+1);
        if (line == "NODE_COORD_SECTION" || line == "NODE_COORD_SECTION\r"){
            start = true; continue;
        }
        if (!start) continue;
        if (line == "EOF") break;
        if (line.empty()) continue;
        stringstream ss(line);
        int id; double x,y;
        if (ss >> id >> x >> y) coords.push_back({x,y});
    }
    return coords;
}

// ----------------- build k-NN on host -----------------
vector<int> build_knn_host(const vector<City>& coords, int k){
    int n = (int)coords.size();
    vector<int> knn(n * k, 0);
    vector<vector<pair<double,int>>> lists(n);
    for(int i = 0; i < n; ++i){
        lists[i].reserve(n-1);
        for(int j = 0; j < n; ++j){
            if (i == j) continue;
            double dx = coords[i].x - coords[j].x;
            double dy = coords[i].y - coords[j].y;
            lists[i].push_back({dx*dx + dy*dy, j}); // squared dist OK for ordering
        }
        sort(lists[i].begin(), lists[i].end());
        int upto = min(k, (int)lists[i].size());
        for(int t = 0; t < upto; ++t) knn[i*k + t] = lists[i][t].second;
        for(int t = upto; t < k; ++t) knn[i*k + t] = lists[i].back().second;
    }
    return knn;
}

// ----------------- host full 2-opt local search (deterministic hill-climb) -----------------
double tour_cost_host(const vector<City>& coords, const vector<int>& tour){
    int n = (int)tour.size();
    double sum = 0.0;
    for(int i = 0; i < n; ++i){
        int a = tour[i], b = tour[(i+1)%n];
        double dx = coords[a].x - coords[b].x;
        double dy = coords[a].y - coords[b].y;
        sum += sqrt(dx*dx + dy*dy);
    }
    return sum;
}

double two_opt_improve(vector<int>& tour, const vector<City>& coords){
    int n = (int)tour.size();
    bool improved = true;
    double bestCost = tour_cost_host(coords, tour);
    while(improved){
        improved = false;
        for(int i = 0; i < n - 2; ++i){
            for(int j = i + 2; j < n; ++j){
                if (i == 0 && j == n-1) continue;
                int A = tour[i], B = tour[(i+1)%n];
                int C = tour[j], D = tour[(j+1)%n];
                double oldEdges = hypot(coords[A].x - coords[B].x, coords[A].y - coords[B].y)
                                + hypot(coords[C].x - coords[D].x, coords[C].y - coords[D].y);
                double newEdges = hypot(coords[A].x - coords[C].x, coords[A].y - coords[C].y)
                                + hypot(coords[B].x - coords[D].x, coords[B].y - coords[D].y);
                double delta = newEdges - oldEdges;
                if (delta < -1e-9){
                    reverse(tour.begin() + i + 1, tour.begin() + j + 1);
                    bestCost += delta;
                    improved = true;
                }
            }
            if (improved) break;
        }
    }
    return bestCost;
}

// ----------------- utility: check tour valid permutation -----------------
bool is_valid_tour(const vector<int>& tour, int n){
    if ((int)tour.size() != n) return false;
    vector<char> seen(n, 0);
    for(int v : tour){
        if (v < 0 || v >= n) return false;
        if (seen[v]) return false;
        seen[v] = 1;
    }
    return true;
}

// ----------------- main -----------------
int main(int argc, char** argv){
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string fname = "tsplib/tsp/kroA200.tsp";
    if (argc > 1) fname = argv[1];

    auto coordsH = readTSPLIB(fname);
    int n = (int)coordsH.size();
    if (n == 0){ cerr << "No cities loaded\n"; return 1; }
    cerr << "Loaded " << n << " cities\n";

    // Hybrid params (tune)
    int walkers = 1024;
    int k = 20;
    int inner = 32;
    int saIter = 200;
    double T0 = 10.0, Tf = 1e-3;
    double alpha = pow(Tf / T0, 1.0 / max(1, saIter - 1));
    int K_refine = 8;
    int gpu_threads = 128;

    // build knn
    cerr << "Building k-NN (n=" << n << ", k=" << k << ") ...\n";
    auto knn_host = build_knn_host(coordsH, k);
    cerr << "k-NN done\n";

    // device allocations
    City* d_coords = nullptr;
    gpuCheck( cudaMalloc((void**)&d_coords, n * sizeof(City)) );
    // copy coords as double
    vector<City> coords_double = coordsH; // already double
    gpuCheck( cudaMemcpy(d_coords, coords_double.data(), n * sizeof(City), cudaMemcpyHostToDevice) );

    int *d_tours = nullptr;
    double *d_costs = nullptr;
    int *d_knn = nullptr;
    curandState* d_rng = nullptr;

    gpuCheck( cudaMalloc((void**)&d_tours, (size_t)walkers * n * sizeof(int)) );
    gpuCheck( cudaMalloc((void**)&d_costs, (size_t)walkers * sizeof(double)) );
    gpuCheck( cudaMalloc((void**)&d_knn, (size_t)n * k * sizeof(int)) );
    gpuCheck( cudaMalloc((void**)&d_rng, (size_t)walkers * sizeof(curandState)) );

    gpuCheck( cudaMemcpy(d_knn, knn_host.data(), (size_t)n * k * sizeof(int), cudaMemcpyHostToDevice) );

    // init RNG with per-thread seeds
    {
        int blocks_init = (walkers + gpu_threads - 1) / gpu_threads;
        initRNG<<<blocks_init, gpu_threads>>>(d_rng, walkers, 123456u);
        gpuCheck( cudaPeekAtLastError() );
        gpuCheck( cudaDeviceSynchronize() );
    }

    // launch GPU kernel
    int blocks = (walkers + gpu_threads - 1) / gpu_threads;
    cudaEvent_t s,e; gpuCheck(cudaEventCreate(&s)); gpuCheck(cudaEventCreate(&e));
    gpuCheck(cudaEventRecord(s));
    sa2opt_knn_kernel_fixed<<<blocks, gpu_threads>>>(d_coords, n, d_tours, d_costs, d_knn, k, d_rng,
                                                     walkers, inner, T0, alpha, saIter);
    gpuCheck( cudaPeekAtLastError() );
    gpuCheck(cudaEventRecord(e));
    gpuCheck(cudaEventSynchronize(e));
    float gpuMs = 0.0f; gpuCheck(cudaEventElapsedTime(&gpuMs, s, e));
    cerr << "GPU phase done, time ms = " << gpuMs << "\n";

    // fetch costs
    vector<double> costsH(walkers, 1e300);
    gpuCheck( cudaMemcpy(costsH.data(), d_costs, walkers * sizeof(double), cudaMemcpyDeviceToHost) );

    // select top-K_refine walkers by cost
    vector<int> idx(walkers);
    iota(idx.begin(), idx.end(), 0);
    int take = min(walkers, K_refine);
    nth_element(idx.begin(), idx.begin() + take, idx.end(),
                [&](int a, int b){ return costsH[a] < costsH[b]; });
    vector<int> topIdx(idx.begin(), idx.begin() + take);
    sort(topIdx.begin(), topIdx.end(), [&](int a, int b){ return costsH[a] < costsH[b]; });

    cout << "Top " << topIdx.size() << " from GPU (costs):\n";
    for(size_t i=0;i<topIdx.size();++i) cout << i << ": idx=" << topIdx[i] << " cost=" << costsH[topIdx[i]] << "\n";

    // copy top tours to host and sanity-check
    vector<vector<int>> hostTours;
    hostTours.reserve(topIdx.size());
    for(int id : topIdx){
        vector<int> tour(n);
        gpuCheck(cudaMemcpy(tour.data(), d_tours + id * n, n * sizeof(int), cudaMemcpyDeviceToHost));
        if (!is_valid_tour(tour, n)){
            cerr << "WARNING: invalid tour detected for walker " << id << " — skipping\n";
            continue;
        }
        hostTours.push_back(move(tour));
    }

    // CPU refinement in parallel
    cerr << "Starting CPU refinement on top " << hostTours.size() << " tours ...\n";
    vector<double> refinedCosts(hostTours.size(), 1e300);
    vector<thread> threads;
    for(size_t i=0;i<hostTours.size();++i){
        threads.emplace_back([&,i](){
            double before = tour_cost_host(coordsH, hostTours[i]);
            double after = two_opt_improve(hostTours[i], coordsH);
            refinedCosts[i] = after;
            cerr << "Refine[" << i << "] before=" << before << " after=" << after << "\n";
        });
    }
    for(auto &t : threads) if (t.joinable()) t.join();

    // choose best among refined + raw GPU
    double bestCost = 1e300;
    vector<int> bestTour;
    // check refined
    for(size_t i=0;i<hostTours.size();++i){
        if (refinedCosts[i] < bestCost){
            bestCost = refinedCosts[i];
            bestTour = hostTours[i];
        }
    }
    // also check raw GPU best (in case CPU worsen)
    for(int i=0;i<walkers;i++){
        if (costsH[i] < bestCost){
            bestCost = costsH[i];
            vector<int> tour(n);
            gpuCheck(cudaMemcpy(tour.data(), d_tours + i * n, n * sizeof(int), cudaMemcpyDeviceToHost));
            if (is_valid_tour(tour, n)) bestTour = tour;
        }
    }

    cout << "\n== Results ==\n";
    cout << "GPU phase time (ms): " << gpuMs << "\n";
    cout << "Best cost after CPU refine: " << bestCost << "\n";
    cout << "Best tour prefix: ";
    for(int i=0;i<min(12, n); ++i) cout << bestTour[i] << (i+1<min(12,n) ? " " : "\n");

    // cleanup
    cudaFree(d_coords); cudaFree(d_tours); cudaFree(d_costs); cudaFree(d_knn); cudaFree(d_rng);
    cudaEventDestroy(s); cudaEventDestroy(e);

    return 0;
}


Writing main_hybrid_fixed.cu


In [ ]:
!nvcc -O3 -arch=sm_75 -std=c++17 main_hybrid_fixed.cu -o tsp_hybrid_fixed

In [ ]:
!./tsp_hybrid_fixed tsplib/kroA200.tsp

Loaded 200 cities
Building k-NN (n=200, k=20) ...
k-NN done
GPU phase done, time ms = 256.423
Top 8 from GPU (costs):
0: idx=139 cost=33086.7
1: idx=1007 cost=33183.8
2: idx=265 cost=33189.7
3: idx=851 cost=33235.4
4: idx=484 cost=33297.5
5: idx=15 cost=33301.6
6: idx=537 cost=33320.2
7: idx=962 cost=33351.8
Starting CPU refinement on top 8 tours ...
Refine[1] before=33183.8 after=32570
Refine[0] before=33086.7 after=31606.5
Refine[3] before=33235.4 after=32165.9
Refine[4] before=33297.5 after=31334.5
Refine[7] before=33351.8 after=32106.7
Refine[2] before=33189.7 after=32387.2
Refine[5] before=33301.6 after=31305.5
Refine[6] before=33320.2 after=31666.8

== Results ==
GPU phase time (ms): 256.423
Best cost after CPU refine: 31305.5
Best tour prefix: 133 74 108 106 156 46 30 5 53 186 150 79


In [ ]:
%%writefile main_hybrid_pos.cu
// main_hybrid_pos.cu
// Hybrid: GPU k-NN SA+2opt (many walkers) -> CPU full 2-opt refines top-K tours
// Improvements: per-walker pos_of_city (O(1) lookup), double accumulation, seeded RNG, global proposals.
// Compile: nvcc -O3 -arch=sm_75 -std=c++17 main_hybrid_pos.cu -o tsp_hybrid_pos

#include <bits/stdc++.h>
#include <curand_kernel.h>
#include <thread>
using namespace std;

struct City { float x, y; };

// ---------------- GPU helpers ----------------
inline void gpuAssert(cudaError_t code, const char *file, int line, bool abort=true) {
    if (code != cudaSuccess) {
        fprintf(stderr,"GPUassert: %s %s %d\n", cudaGetErrorString(code), file, line);
        if (abort) exit(code);
    }
}
#define gpuCheck(ans) { gpuAssert((ans), __FILE__, __LINE__); }

// device Euclidean distance (accumulate in double)
__device__ inline double dist_dev_d(const City* c, int a, int b){
    double dx = (double)c[a].x - (double)c[b].x;
    double dy = (double)c[a].y - (double)c[b].y;
    return sqrt(dx*dx + dy*dy);
}

// RNG init: use seed + tid*prime to decorrelate
__global__ void initRNG(curandState* s, int walkers, unsigned long seed){
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id >= walkers) return;
    unsigned long composite = seed + (unsigned long)id * 7919u;
    curand_init(composite, id, 0, &s[id]);
}

// SA + 2-opt kernel with k-NN, pos_of_city, double costs, occasional global proposals
__global__ void sa2opt_knn_pos_kernel(
    const City* coords, int n,
    int* tours_global,        // (walkers * n) in-place
    int* pos_global,          // (walkers * n) position mapping
    double* costs_global,     // (walkers)
    const int* knn, int k,    // knn table (n*k)
    curandState* rng,         // (walkers)
    int walkers,
    int inner, double T0, double alpha, int saIter,
    float p_global, int sanity_interval
){
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= walkers) return;

    int* tour = tours_global + tid * n;
    int* pos  = pos_global + tid * n;
    curandState st = rng[tid];

    // random initial permutation
    for(int i = 0; i < n; ++i) tour[i] = i;
    for(int i = 0; i < n; ++i){
        int j = curand(&st) % n;
        int tmp = tour[i]; tour[i] = tour[j]; tour[j] = tmp;
    }
    // init pos array
    for(int i = 0; i < n; ++i) pos[tour[i]] = i;

    // compute initial cost (double)
    double cur = 0.0;
    for(int i = 0; i < n; ++i){
        int a = tour[i], b = tour[(i+1)%n];
        cur += dist_dev_d(coords, a, b);
    }
    double best = cur;
    double T = T0;

    int no_improve = 0;

    for(int it = 0; it < saIter; ++it){
        for(int t = 0; t < inner; ++t){
            // pick random position a
            int posA = curand(&st) % n;
            int cityA = tour[posA];

            // choose candidate city: mostly from knn, sometimes globally random
            int cityB;
            if (curand_uniform(&st) < p_global){
                cityB = curand(&st) % n;
            } else {
                int pick = curand(&st) % k;
                cityB = knn[cityA * k + pick];
            }

            int posB = pos[cityB]; // O(1)
            if (posB == posA) continue;
            int a = posA, b = posB;
            if (a > b){ int tmp = a; a = b; b = tmp; }
            if (b == a + 1) continue; // adjacent -> no-op

            int A = tour[a], B = tour[(a+1)%n], C = tour[b], D = tour[(b+1)%n];
            double delta = (dist_dev_d(coords, A, C) + dist_dev_d(coords, B, D))
                         - (dist_dev_d(coords, A, B) + dist_dev_d(coords, C, D));

            double accept_prob = exp(-delta / max(T, 1e-12));
            double rv = (double)curand_uniform(&st);
            if (delta < 0.0 || rv < accept_prob){
                // apply 2-opt reversal a+1..b and update pos[] accordingly
                int L = a+1, R = b;
                while(L < R){
                    int cL = tour[L], cR = tour[R];
                    // swap in tour
                    tour[L] = cR; tour[R] = cL;
                    // update positions
                    pos[cR] = L;
                    pos[cL] = R;
                    ++L; --R;
                }
                // if L==R (odd length) the middle element stays in place; pos already consistent
                cur += delta;
                if (cur < best){
                    best = cur;
                    no_improve = 0;
                } else {
                    no_improve++;
                }
            } else {
                no_improve++;
            }
        } // inner

        T *= alpha;

        // occasional full-cost recompute for sanity
        if ((it % sanity_interval) == 0){
            double check = 0.0;
            for(int i = 0; i < n; ++i){
                int aa = tour[i], bb = tour[(i+1)%n];
                check += dist_dev_d(coords, aa, bb);
            }
            if (!isfinite(check) || fabs(check - cur) > 1e6){
                cur = check;
                if (!isfinite(cur)) cur = 1e100;
                if (cur < best) best = cur;
            }
        }

        // optional mild reheating if stuck (per-thread)
        if (no_improve > (sanity_interval*4)){
            T *= 1.02;      // minor reheat
            no_improve = 0;
        }
    } // outer

    // store RNG state and result
    rng[tid] = st;
    costs_global[tid] = best;
}

// ------------- Host: TSPLIB loader -------------
vector<City> readTSPLIB(const string& path){
    ifstream in(path);
    if(!in.is_open()){ cerr<<"ERROR opening "<<path<<"\n"; return {}; }
    string line; bool start=false; vector<City> coords;
    while(getline(in,line)){
        // trim
        auto s = line.find_first_not_of(" \t\r\n");
        if (s==string::npos) continue;
        line.erase(0,s);
        auto e = line.find_last_not_of(" \t\r\n");
        if (e!=string::npos) line.erase(e+1);
        if (line=="NODE_COORD_SECTION" || line=="NODE_COORD_SECTION\r"){ start=true; continue; }
        if (!start) continue;
        if (line=="EOF") break;
        if (line.empty()) continue;
        stringstream ss(line); int id; float x,y;
        if (ss>>id>>x>>y) coords.push_back({x,y});
    }
    return coords;
}

// ------------- build k-NN on host (brute force) -------------
vector<int> build_knn_host(const vector<City>& coords, int k){
    int n = (int)coords.size();
    vector<int> knn(n * k, 0);
    vector<vector<pair<float,int>>> lists(n);
    for(int i=0;i<n;++i){
        lists[i].reserve(n-1);
        for(int j=0;j<n;++j) if (i!=j){
            float dx = coords[i].x - coords[j].x;
            float dy = coords[i].y - coords[j].y;
            lists[i].push_back({dx*dx + dy*dy, j});
        }
        sort(lists[i].begin(), lists[i].end());
        for(int t=0;t<k;++t){
            if (t < (int)lists[i].size()) knn[i*k + t] = lists[i][t].second;
            else knn[i*k + t] = lists[i].back().second;
        }
    }
    return knn;
}

// ------------- Host: tour cost and 2-opt refine -------------
double tour_cost_host(const vector<City>& coords, const vector<int>& tour){
    int n = (int)tour.size();
    double sum = 0.0;
    for(int i=0;i<n;++i){
        int a=tour[i], b=tour[(i+1)%n];
        double dx = coords[a].x - coords[b].x;
        double dy = coords[a].y - coords[b].y;
        sum += sqrt(dx*dx + dy*dy);
    }
    return sum;
}

// deterministic full 2-opt hill-climb until no improvement
double two_opt_improve(vector<int>& tour, const vector<City>& coords){
    int n = (int)tour.size();
    bool improved = true;
    double bestCost = tour_cost_host(coords, tour);
    while(improved){
        improved = false;
        for(int i=0;i<n-2;++i){
            for(int j=i+2;j<n;++j){
                if (i==0 && j==n-1) continue;
                int A = tour[i], B = tour[(i+1)%n];
                int C = tour[j], D = tour[(j+1)%n];
                double oldEdges = hypot(coords[A].x - coords[B].x, coords[A].y - coords[B].y)
                                + hypot(coords[C].x - coords[D].x, coords[C].y - coords[D].y);
                double newEdges = hypot(coords[A].x - coords[C].x, coords[A].y - coords[C].y)
                                + hypot(coords[B].x - coords[D].x, coords[B].y - coords[D].y);
                double delta = newEdges - oldEdges;
                if (delta < -1e-9){
                    reverse(tour.begin() + i + 1, tour.begin() + j + 1);
                    bestCost += delta;
                    improved = true;
                }
            }
            if (improved) break;
        }
    }
    return bestCost;
}

// validate tour is permutation 0..n-1
bool is_valid_tour(const vector<int>& tour, int n){
    if ((int)tour.size() != n) return false;
    vector<char> seen(n, 0);
    for(int v : tour){
        if (v < 0 || v >= n) return false;
        if (seen[v]) return false;
        seen[v] = 1;
    }
    return true;
}

// ------------- main -------------
int main(int argc, char** argv){
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string fname = "tsplib/tsp/kroA200.tsp";
    if (argc > 1) fname = argv[1];

    auto coordsH = readTSPLIB(fname);
    int n = (int)coordsH.size();
    if (n == 0){ cerr<<"No cities loaded\n"; return 1; }
    cerr<<"Loaded "<<n<<" cities\n";

    // Parameters (tune these)
    int walkers = 1024;   // breadth (one walker per thread)
    int k = 20;           // k-NN size
    int inner = max(32, n/8); // moves per temperature (we can increase since pos lookup is O(1))
    int saIter = 250;
    double T0 = 10.0, Tf = 1e-3;
    double alpha = pow(Tf / T0, 1.0 / max(1, saIter - 1));
    int K_refine = 8;     // how many top tours to refine on CPU
    int gpu_threads = 128;
    float p_global = 0.03f; // probability of a global random candidate
    int sanity_interval = 32;

    // Build k-NN on host
    cerr<<"Building k-NN (n="<<n<<", k="<<k<<") ...\n";
    auto knn_host = build_knn_host(coordsH, k);
    cerr<<"k-NN done\n";

    // Device allocations
    City* d_coords = nullptr;
    gpuCheck( cudaMalloc((void**)&d_coords, n * sizeof(City)) );
    gpuCheck( cudaMemcpy(d_coords, coordsH.data(), n * sizeof(City), cudaMemcpyHostToDevice) );

    int *d_tours = nullptr;
    int *d_pos = nullptr;
    double *d_costs = nullptr;
    int *d_knn = nullptr;
    curandState* d_rng = nullptr;

    gpuCheck( cudaMalloc((void**)&d_tours, (size_t)walkers * n * sizeof(int)) );
    gpuCheck( cudaMalloc((void**)&d_pos,   (size_t)walkers * n * sizeof(int)) );
    gpuCheck( cudaMalloc((void**)&d_costs, (size_t)walkers * sizeof(double)) );
    gpuCheck( cudaMalloc((void**)&d_knn,   (size_t)n * k * sizeof(int)) );
    gpuCheck( cudaMalloc((void**)&d_rng,   (size_t)walkers * sizeof(curandState)) );

    gpuCheck( cudaMemcpy(d_knn, knn_host.data(), (size_t)n * k * sizeof(int), cudaMemcpyHostToDevice) );

    // init RNG
    {
        int blocks_init = (walkers + gpu_threads - 1) / gpu_threads;
        initRNG<<<blocks_init, gpu_threads>>>(d_rng, walkers, 123456u);
        gpuCheck( cudaPeekAtLastError() );
        gpuCheck( cudaDeviceSynchronize() );
    }

    // Launch kernel (one thread per walker)
    int blocks = (walkers + gpu_threads - 1) / gpu_threads;
    cudaEvent_t startEvt, stopEvt;
    gpuCheck(cudaEventCreate(&startEvt));
    gpuCheck(cudaEventCreate(&stopEvt));
    gpuCheck(cudaEventRecord(startEvt));
    sa2opt_knn_pos_kernel<<<blocks, gpu_threads>>>(d_coords, n, d_tours, d_pos, d_costs, d_knn, k, d_rng,
                                                   walkers, inner, T0, alpha, saIter, p_global, sanity_interval);
    gpuCheck( cudaPeekAtLastError() );
    gpuCheck(cudaEventRecord(stopEvt));
    gpuCheck(cudaEventSynchronize(stopEvt));
    float gpuMs = 0.0f; gpuCheck(cudaEventElapsedTime(&gpuMs, startEvt, stopEvt));
    cerr<<"GPU phase done, time ms = "<<gpuMs<<"\n";

    // fetch costs
    vector<double> costsH(walkers, 1e300);
    gpuCheck( cudaMemcpy(costsH.data(), d_costs, walkers * sizeof(double), cudaMemcpyDeviceToHost) );

    // find top-K_refine walkers (by cost)
    vector<int> idx(walkers);
    iota(idx.begin(), idx.end(), 0);
    int take = min((int)idx.size(), K_refine);
    nth_element(idx.begin(), idx.begin() + take, idx.end(),
                [&](int a, int b){ return costsH[a] < costsH[b]; });
    vector<int> topIdx(idx.begin(), idx.begin() + take);
    sort(topIdx.begin(), topIdx.end(), [&](int a,int b){ return costsH[a] < costsH[b]; });

    cout<<"Top "<<topIdx.size()<<" from GPU (costs):\n";
    for(size_t i=0;i<topIdx.size();++i) cout<<i<<": idx="<<topIdx[i]<<" cost="<<costsH[topIdx[i]]<<"\n";

    // Copy top tours to host for CPU refinement and validate
    vector<vector<int>> hostTours;
    hostTours.reserve(topIdx.size());
    for(int id : topIdx){
        vector<int> tour(n);
        gpuCheck(cudaMemcpy(tour.data(), d_tours + id * n, n * sizeof(int), cudaMemcpyDeviceToHost));
        if (!is_valid_tour(tour, n)){
            cerr<<"WARNING: invalid tour for walker "<<id<<" — skipping\n";
            continue;
        }
        hostTours.push_back(move(tour));
    }

    // CPU refine top tours in parallel
    cerr<<"Starting CPU refinement on top "<<hostTours.size()<<" tours ...\n";
    vector<double> refinedCosts(hostTours.size(), 1e300);
    vector<thread> threads;
    for(size_t i=0;i<hostTours.size();++i){
        threads.emplace_back([&,i](){
            double before = tour_cost_host(coordsH, hostTours[i]);
            double after = two_opt_improve(hostTours[i], coordsH);
            refinedCosts[i] = after;
            cerr<<"Refine["<<i<<"] before="<<before<<" after="<<after<<"\n";
        });
    }
    for(auto &t : threads) if (t.joinable()) t.join();

    // choose best among refined + raw GPU (validate tours before accepting)
    double bestCost = 1e300;
    vector<int> bestTour;
    // check refined
    for(size_t i=0;i<hostTours.size();++i){
        if (refinedCosts[i] < bestCost){
            bestCost = refinedCosts[i];
            bestTour = hostTours[i];
        }
    }
    // also check raw GPU best (in case CPU worsen or no valid refined)
    for(int i=0;i<walkers;i++){
        if (costsH[i] < bestCost){
            vector<int> tour(n);
            gpuCheck(cudaMemcpy(tour.data(), d_tours + i * n, n * sizeof(int), cudaMemcpyDeviceToHost));
            if (is_valid_tour(tour, n)){
                bestCost = costsH[i];
                bestTour = tour;
            }
        }
    }

    cout<<"\n== Results ==\n";
    cout<<"GPU phase time (ms): "<<gpuMs<<"\n";
    cout<<"Best cost after CPU refine: "<<bestCost<<"\n";
    cout<<"Best tour prefix: ";
    for(int i=0;i<min(12, n); ++i) cout<<bestTour[i]<<(i+1<min(12,n)?" ":"\n");

    // cleanup
    cudaFree(d_coords); cudaFree(d_tours); cudaFree(d_pos); cudaFree(d_costs);
    cudaFree(d_knn); cudaFree(d_rng);
    cudaEventDestroy(startEvt); cudaEventDestroy(stopEvt);

    return 0;
}


Writing main_hybrid_pos.cu


In [ ]:
!nvcc -O3 -arch=sm_75 -std=c++17 main_hybrid_pos.cu -o tsp_hybrid_pos

In [ ]:
!./tsp_hybrid_pos tsplib/kroB200.tsp

Loaded 200 cities
Building k-NN (n=200, k=20) ...
k-NN done
GPU phase done, time ms = 166.029
Top 8 from GPU (costs):
0: idx=493 cost=31996.5
1: idx=750 cost=32027.5
2: idx=113 cost=32317.5
3: idx=252 cost=32588.1
4: idx=211 cost=32595.5
5: idx=546 cost=32611
6: idx=816 cost=32688.1
7: idx=593 cost=32695.3
Starting CPU refinement on top 8 tours ...
Refine[1] before=32027.5 after=32027.5
Refine[0] before=31996.5 after=31820.3
Refine[2] before=32317.5 after=32184.4
Refine[5] before=32611 after=32405.5
Refine[3] before=32588.1 after=31786.2
Refine[7] before=32695.3 after=31169.8
Refine[4] before=32595.5 after=31939.4
Refine[6] before=32688.1 after=31719.2

== Results ==
GPU phase time (ms): 166.029
Best cost after CPU refine: 31169.8
Best tour prefix: 87 138 172 114 71 36 183 140 46 178 80 112


In [ ]:
%%writefile main_hybrid_2_5opt.cu
// Hybrid: GPU k-NN SA + mixed 2-opt / 2.5-opt (reinsertion) -> CPU full 2-opt refine
// Improvements: pos_of_city, double accumulation, seeded RNG, occasional global proposals, 2.5-opt.
// Compile: nvcc -O3 -arch=sm_75 -std=c++17 main_hybrid_2_5opt.cu -o tsp_hybrid_2_5opt

#include <bits/stdc++.h>
#include <curand_kernel.h>
#include <thread>
using namespace std;

struct City { float x, y; };

// ---------- GPU helpers ----------
inline void gpuAssert(cudaError_t code, const char *file, int line, bool abort=true) {
    if (code != cudaSuccess) {
        fprintf(stderr,"GPUassert: %s %s %d\n", cudaGetErrorString(code), file, line);
        if (abort) exit(code);
    }
}
#define gpuCheck(ans) { gpuAssert((ans), __FILE__, __LINE__); }

// device Euclidean distance accumulated in double
__device__ inline double dist_dev_d(const City* c, int a, int b){
    double dx = (double)c[a].x - (double)c[b].x;
    double dy = (double)c[a].y - (double)c[b].y;
    return sqrt(dx*dx + dy*dy);
}

// RNG init: seed per-thread with small scrambling multiplier
__global__ void initRNG(curandState* s, int walkers, unsigned long seed){
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id >= walkers) return;
    unsigned long composite = seed + (unsigned long)id * 7919u + 1315423911u;
    curand_init(composite, id, 0, &s[id]);
}

// SA kernel with mixed 2-opt and reinsertion (2.5-opt)
__global__ void sa_mixed_2opt_reinsert_kernel(
    const City* coords, int n,
    int* tours_global,        // walkers * n (in-place)
    int* pos_global,          // walkers * n (pos of city in tour)
    double* costs_global,     // walkers (double)
    const int* knn, int k,    // knn table (n*k)
    curandState* rng,         // walkers
    int walkers,
    int inner, double T0, double alpha, int saIter,
    float p_global, float p_reinsert, int sanity_interval
){
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= walkers) return;

    int* tour = tours_global + tid * n;
    int* pos  = pos_global + tid * n;
    curandState st = rng[tid];

    // Random initial permutation (Fisher-Yates-like)
    for (int i = 0; i < n; ++i) tour[i] = i;
    for (int i = 0; i < n; ++i){
        int j = curand(&st) % n;
        int tmp = tour[i]; tour[i] = tour[j]; tour[j] = tmp;
    }
    // initialize pos[]
    for (int i = 0; i < n; ++i) pos[tour[i]] = i;

    // initial tour cost (double)
    double cur = 0.0;
    for (int i = 0; i < n; ++i){
        int a = tour[i], b = tour[(i+1)%n];
        cur += dist_dev_d(coords, a, b);
    }
    double best = cur;
    double T = T0;
    int no_improve = 0;

    for (int it = 0; it < saIter; ++it){
        for (int t = 0; t < inner; ++t){
            // decide move type: 2-opt vs reinsertion
            double mvchoice = curand_uniform(&st);

            if (mvchoice < p_reinsert) {
                // ==== Reinsertion (2.5-opt) ====
                // pick a random city position to remove
                int posI = curand(&st) % n;
                int cityI = tour[posI];
                int posPrev = (posI - 1 + n) % n;
                int posNext = (posI + 1) % n;
                int cityPrev = tour[posPrev];
                int cityNext = tour[posNext];

                // removal delta: remove edges (Prev-I) and (I-Next), add (Prev-Next)
                double delta_rem = dist_dev_d(coords, cityPrev, cityNext)
                                 - dist_dev_d(coords, cityPrev, cityI)
                                 - dist_dev_d(coords, cityI, cityNext);

                // pick insertion target: mostly kNN of cityI, occasionally global random
                int cityT;
                if (curand_uniform(&st) < p_global){
                    cityT = curand(&st) % n;
                } else {
                    int pick = curand(&st) % k;
                    // candidate city from knn of cityI
                    cityT = knn[cityI * k + pick];
                }
                int posT = pos[cityT];

                // choose insertion position: before posT or after posT randomly
                bool insertAfter = (curand_uniform(&st) < 0.5f);
                int posIns = insertAfter ? ((posT + 1) % n) : posT;

                // if insertion is adjacent to removal (no-op), skip
                if (posIns == posI || posIns == posPrev) {
                    // no-op
                } else {
                    // When we remove cityI and insert at posIns, compute insertion delta
                    int cityC = tour[(posIns + n - 1) % n];
                    int cityD = tour[posIns % n];
                    // After insertion we will have (C - I - D) replacing (C - D)
                    double delta_ins = dist_dev_d(coords, cityC, cityI)
                                     + dist_dev_d(coords, cityI, cityD)
                                     - dist_dev_d(coords, cityC, cityD);
                    double delta = delta_rem + delta_ins;

                    double rv = (double)curand_uniform(&st);
                    if (delta < 0.0 || rv < exp(-delta / max(T, 1e-12))) {
                        // perform in-place remove and insert; update pos[] accordingly
                        if (posI < posIns) {
                            // shift left posI+1 .. posIns-1 into posI .. posIns-2
                            int tmp = tour[posI];
                            for (int p = posI; p < posIns - 1; ++p){
                                tour[p] = tour[p + 1];
                                pos[tour[p]] = p;
                            }
                            tour[posIns - 1] = tmp;
                            pos[tmp] = posIns - 1;
                        } else {
                            // posI > posIns: shift right posIns .. posI-1 into posIns+1 .. posI
                            int tmp = tour[posI];
                            for (int p = posI; p > posIns; --p){
                                tour[p] = tour[p - 1];
                                pos[tour[p]] = p;
                            }
                            tour[posIns] = tmp;
                            pos[tmp] = posIns;
                        }
                        cur += delta;
                        if (cur < best){
                            best = cur;
                            no_improve = 0;
                        } else {
                            no_improve++;
                        }
                    } else {
                        no_improve++;
                    }
                }
            } else {
                // ==== 2-opt (existing) ====
                int posA = curand(&st) % n;
                int cityA = tour[posA];

                int cityB;
                if (curand_uniform(&st) < p_global) {
                    cityB = curand(&st) % n;
                } else {
                    int pick = curand(&st) % k;
                    cityB = knn[cityA * k + pick];
                }
                int posB = pos[cityB];
                if (posB == posA) { continue; }
                int a = posA, b = posB;
                if (a > b){ int tmp = a; a = b; b = tmp; }
                if (b == a + 1) continue;

                int A = tour[a], B = tour[(a+1)%n], C = tour[b], D = tour[(b+1)%n];
                double delta = (dist_dev_d(coords, A, C) + dist_dev_d(coords, B, D))
                             - (dist_dev_d(coords, A, B) + dist_dev_d(coords, C, D));
                double rv = (double)curand_uniform(&st);
                if (delta < 0.0 || rv < exp(-delta / max(T, 1e-12))) {
                    // reverse a+1 .. b and update pos[]
                    int L = a + 1, R = b;
                    while (L < R){
                        int cL = tour[L], cR = tour[R];
                        tour[L] = cR; tour[R] = cL;
                        pos[cR] = L; pos[cL] = R;
                        ++L; --R;
                    }
                    cur += delta;
                    if (cur < best){
                        best = cur;
                        no_improve = 0;
                    } else {
                        no_improve++;
                    }
                } else {
                    no_improve++;
                }
            } // end move choice
        } // inner

        T *= alpha;

        // occasional sanity recompute
        if ((it % sanity_interval) == 0){
            double check = 0.0;
            for (int i = 0; i < n; ++i){
                int aa = tour[i], bb = tour[(i+1)%n];
                check += dist_dev_d(coords, aa, bb);
            }
            if (!isfinite(check) || fabs(check - cur) > 1e6){
                cur = check;
                if (!isfinite(cur)) cur = 1e100;
                if (cur < best) best = cur;
            }
        }

        // mild reheating if stuck for too long
        if (no_improve > (sanity_interval * 4)) {
            T *= 1.02; // small reheat
            no_improve = 0;
        }
    } // outer

    rng[tid] = st;
    costs_global[tid] = best;
}

// ---------- Host utilities ----------
vector<City> readTSPLIB(const string& path){
    ifstream in(path);
    if(!in.is_open()){ cerr<<"ERROR opening "<<path<<"\n"; return {}; }
    string line; bool start=false; vector<City> coords;
    while(getline(in, line)){
        size_t s = line.find_first_not_of(" \t\r\n");
        if (s == string::npos) continue;
        line.erase(0, s);
        size_t e = line.find_last_not_of(" \t\r\n");
        if (e != string::npos) line.erase(e + 1);
        if (line == "NODE_COORD_SECTION" || line == "NODE_COORD_SECTION\r") { start = true; continue; }
        if (!start) continue;
        if (line == "EOF") break;
        if (line.empty()) continue;
        stringstream ss(line);
        int id; float x, y;
        if (ss >> id >> x >> y) coords.push_back({x, y});
    }
    return coords;
}

vector<int> build_knn_host(const vector<City>& coords, int k){
    int n = (int)coords.size();
    vector<int> knn(n * k, 0);
    vector<vector<pair<float,int>>> lists(n);
    for (int i = 0; i < n; ++i){
        lists[i].reserve(n - 1);
        for (int j = 0; j < n; ++j) if (i != j){
            float dx = coords[i].x - coords[j].x, dy = coords[i].y - coords[j].y;
            lists[i].push_back({dx*dx + dy*dy, j});
        }
        sort(lists[i].begin(), lists[i].end());
        int upto = min(k, (int)lists[i].size());
        for (int t = 0; t < upto; ++t) knn[i*k + t] = lists[i][t].second;
        for (int t = upto; t < k; ++t) knn[i*k + t] = lists[i].back().second;
    }
    return knn;
}

double tour_cost_host(const vector<City>& coords, const vector<int>& tour){
    int n = (int)tour.size(); double sum = 0.0;
    for (int i = 0; i < n; ++i){
        int a = tour[i], b = tour[(i+1)%n];
        double dx = coords[a].x - coords[b].x, dy = coords[a].y - coords[b].y;
        sum += sqrt(dx*dx + dy*dy);
    }
    return sum;
}

// deterministic full 2-opt until no improvement
double two_opt_improve(vector<int>& tour, const vector<City>& coords){
    int n = (int)tour.size();
    bool improved = true;
    double bestCost = tour_cost_host(coords, tour);
    while (improved){
        improved = false;
        for (int i = 0; i < n - 2; ++i){
            for (int j = i + 2; j < n; ++j){
                if (i == 0 && j == n - 1) continue;
                int A = tour[i], B = tour[(i+1)%n];
                int C = tour[j], D = tour[(j+1)%n];
                double oldEdges = hypot(coords[A].x - coords[B].x, coords[A].y - coords[B].y)
                                + hypot(coords[C].x - coords[D].x, coords[C].y - coords[D].y);
                double newEdges = hypot(coords[A].x - coords[C].x, coords[A].y - coords[C].y)
                                + hypot(coords[B].x - coords[D].x, coords[B].y - coords[D].y);
                double delta = newEdges - oldEdges;
                if (delta < -1e-9){
                    reverse(tour.begin() + i + 1, tour.begin() + j + 1);
                    bestCost += delta;
                    improved = true;
                }
            }
            if (improved) break;
        }
    }
    return bestCost;
}

bool is_valid_tour(const vector<int>& tour, int n){
    if ((int)tour.size() != n) return false;
    vector<char> seen(n, 0);
    for (int v : tour){
        if (v < 0 || v >= n) return false;
        if (seen[v]) return false;
        seen[v] = 1;
    }
    return true;
}

// ---------- main ----------
int main(int argc, char** argv){
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string fname = "tsplib/tsp/kroA200.tsp";
    if (argc > 1) fname = argv[1];

    auto coordsH = readTSPLIB(fname);
    int n = (int)coordsH.size();
    if (n == 0){ cerr<<"No cities loaded\n"; return 1; }
    cerr<<"Loaded "<<n<<" cities\n";

    // Tuning parameters (feel free to tweak)
    int walkers = 1024;
    int k = 20;
    int inner = max(32, n / 8);
    int saIter = 250;
    double T0 = 10.0, Tf = 1e-3;
    double alpha = pow(Tf / T0, 1.0 / max(1, saIter - 1));
    int K_refine = 8;
    int gpu_threads = 128;
    float p_global = 0.03f;   // chance to pick global random candidate
    float p_reinsert = 0.45f; // probability that move is reinsertion; rest 2-opt
    int sanity_interval = 32;

    cerr << "Parameters: walkers=" << walkers << " k=" << k << " inner=" << inner
         << " saIter=" << saIter << " p_global=" << p_global << " p_reinsert=" << p_reinsert << "\n";

    // build k-NN on host
    cerr << "Building k-NN (n=" << n << ", k=" << k << ") ...\n";
    auto knn_host = build_knn_host(coordsH, k);
    cerr << "k-NN done\n";

    // device allocations
    City* d_coords = nullptr;
    gpuCheck( cudaMalloc((void**)&d_coords, n * sizeof(City)) );
    gpuCheck( cudaMemcpy(d_coords, coordsH.data(), n * sizeof(City), cudaMemcpyHostToDevice) );

    int *d_tours = nullptr, *d_pos = nullptr, *d_knn = nullptr;
    double *d_costs = nullptr;
    curandState* d_rng = nullptr;

    gpuCheck( cudaMalloc((void**)&d_tours, (size_t)walkers * n * sizeof(int)) );
    gpuCheck( cudaMalloc((void**)&d_pos,   (size_t)walkers * n * sizeof(int)) );
    gpuCheck( cudaMalloc((void**)&d_costs, (size_t)walkers * sizeof(double)) );
    gpuCheck( cudaMalloc((void**)&d_knn,   (size_t)n * k * sizeof(int)) );
    gpuCheck( cudaMalloc((void**)&d_rng,   (size_t)walkers * sizeof(curandState)) );

    gpuCheck( cudaMemcpy(d_knn, knn_host.data(), (size_t)n * k * sizeof(int), cudaMemcpyHostToDevice) );

    // init RNG
    {
        int blocks_init = (walkers + gpu_threads - 1) / gpu_threads;
        initRNG<<<blocks_init, gpu_threads>>>(d_rng, walkers, 123456789u);
        gpuCheck(cudaPeekAtLastError());
        gpuCheck(cudaDeviceSynchronize());
    }

    // run kernel
    int blocks = (walkers + gpu_threads - 1) / gpu_threads;
    cudaEvent_t s,e; gpuCheck(cudaEventCreate(&s)); gpuCheck(cudaEventCreate(&e));
    gpuCheck(cudaEventRecord(s));
    sa_mixed_2opt_reinsert_kernel<<<blocks, gpu_threads>>>(
        d_coords, n, d_tours, d_pos, d_costs, d_knn, k, d_rng,
        walkers, inner, T0, alpha, saIter, p_global, p_reinsert, sanity_interval
    );
    gpuCheck(cudaPeekAtLastError());
    gpuCheck(cudaEventRecord(e));
    gpuCheck(cudaEventSynchronize(e));
    float gpuMs = 0.0f; gpuCheck(cudaEventElapsedTime(&gpuMs, s, e));
    cerr << "GPU phase done, time ms = " << gpuMs << "\n";

    // fetch costs
    vector<double> costsH(walkers, 1e300);
    gpuCheck( cudaMemcpy(costsH.data(), d_costs, walkers * sizeof(double), cudaMemcpyDeviceToHost) );

    // pick top K_refine walkers
    vector<int> idx(walkers);
    iota(idx.begin(), idx.end(), 0);
    int take = min((int)idx.size(), K_refine);
    nth_element(idx.begin(), idx.begin() + take, idx.end(),
                [&](int a, int b){ return costsH[a] < costsH[b]; });
    vector<int> topIdx(idx.begin(), idx.begin() + take);
    sort(topIdx.begin(), topIdx.end(), [&](int a, int b){ return costsH[a] < costsH[b]; });

    cout << "Top " << topIdx.size() << " from GPU (costs):\n";
    for (size_t i = 0; i < topIdx.size(); ++i) cout << i << ": idx=" << topIdx[i] << " cost=" << costsH[topIdx[i]] << "\n";

    // copy top tours to host and validate
    vector<vector<int>> hostTours;
    hostTours.reserve(topIdx.size());
    for (int id : topIdx){
        vector<int> tour(n);
        gpuCheck(cudaMemcpy(tour.data(), d_tours + id * n, n * sizeof(int), cudaMemcpyDeviceToHost));
        if (!is_valid_tour(tour, n)){
            cerr << "WARNING: invalid tour for walker " << id << " — skipping\n";
            continue;
        }
        hostTours.push_back(move(tour));
    }

    // CPU refine in parallel
    cerr << "Starting CPU refinement on top " << hostTours.size() << " tours ...\n";
    vector<double> refinedCosts(hostTours.size(), 1e300);
    vector<thread> threads;
    for (size_t i = 0; i < hostTours.size(); ++i){
        threads.emplace_back([&, i](){
            double before = tour_cost_host(coordsH, hostTours[i]);
            double after = two_opt_improve(hostTours[i], coordsH);
            refinedCosts[i] = after;
            cerr << "Refine[" << i << "] before=" << before << " after=" << after << "\n";
        });
    }
    for (auto &t : threads) if (t.joinable()) t.join();

    // choose best among refined and raw GPU
    double bestCost = 1e300;
    vector<int> bestTour;
    for (size_t i = 0; i < hostTours.size(); ++i){
        if (refinedCosts[i] < bestCost){
            bestCost = refinedCosts[i];
            bestTour = hostTours[i];
        }
    }
    // also check raw GPU results (copy best raw if valid)
    for (int i = 0; i < walkers; ++i){
        if (costsH[i] < bestCost){
            vector<int> tour(n);
            gpuCheck(cudaMemcpy(tour.data(), d_tours + i * n, n * sizeof(int), cudaMemcpyDeviceToHost));
            if (is_valid_tour(tour, n)){
                bestCost = costsH[i];
                bestTour = tour;
            }
        }
    }

    cout << "\n== Results ==\n";
    cout << "GPU phase time (ms): " << gpuMs << "\n";
    cout << "Best cost after CPU refine: " << bestCost << "\n";
    cout << "Best tour prefix: ";
    for (int i = 0; i < min(12, n); ++i) cout << bestTour[i] << (i+1<min(12,n) ? " " : "\n");

    // cleanup
    cudaFree(d_coords); cudaFree(d_tours); cudaFree(d_pos); cudaFree(d_costs);
    cudaFree(d_knn); cudaFree(d_rng);
    cudaEventDestroy(s); cudaEventDestroy(e);

    return 0;
}


Writing main_hybrid_2_5opt.cu


In [ ]:
!nvcc -O3 -arch=sm_75 -std=c++17 main_hybrid_2_5opt.cu -o tsp_hybrid_2_5opt

In [ ]:
!./tsp_hybrid_2_5opt tsplib/kroA200.tsp

Loaded 200 cities
Parameters: walkers=1024 k=20 inner=32 saIter=250 p_global=0.03 p_reinsert=0.45
Building k-NN (n=200, k=20) ...
k-NN done
GPU phase done, time ms = 269.126
Top 8 from GPU (costs):
0: idx=977 cost=21546.4
1: idx=424 cost=21747.4
2: idx=415 cost=22105.8
3: idx=776 cost=22288
4: idx=903 cost=22318.3
5: idx=188 cost=22497.9
6: idx=51 cost=22532.2
7: idx=1009 cost=22543.6
Starting CPU refinement on top 8 tours ...
Refine[5] before=33012.1 after=31431.5
Refine[7] before=33037.1 after=31382.7
Refine[4] before=33828.3 after=31308.7
Refine[1] before=34701.4 after=32200.3
Refine[3] before=33460.6 after=32052
Refine[6] before=34107.9 after=32404
Refine[0] before=33890.2 after=31087.5
Refine[2] before=34328.6 after=30973.7

== Results ==
GPU phase time (ms): 269.126
Best cost after CPU refine: 21546.4
Best tour prefix: 168 34 67 29 166 127 192 64 156 53 5 108


In [ ]:
%%writefile main_hybrid_2_5opt_dist.cu
// Hybrid: GPU k-NN SA + mixed 2-opt / 2.5-opt (reinsertion) -> CPU full 2-opt refine
// Uses TSPLIB EUC_2D integer distance matrix (host-built) uploaded to device.
// Compile: nvcc -O3 -arch=sm_75 -std=c++17 main_hybrid_2_5opt_dist.cu -o tsp_hybrid_2_5opt_dist

#include <bits/stdc++.h>
#include <curand_kernel.h>
#include <thread>
using namespace std;

struct City { float x, y; };

// ---------------- GPU helpers ----------------
inline void gpuAssert(cudaError_t code, const char *file, int line, bool abort=true) {
    if (code != cudaSuccess) {
        fprintf(stderr,"GPUassert: %s %s %d\n", cudaGetErrorString(code), file, line);
        if (abort) exit(code);
    }
}
#define gpuCheck(ans) { gpuAssert((ans), __FILE__, __LINE__); }

// Lookup TSPLIB integer distance matrix (device) -> returns double for accumulation
__device__ inline double dist_lookup_int(const int* distMat, int n, int a, int b){
    return (double)distMat[a * n + b];
}

// RNG init: seed per-thread with small scrambling multiplier
__global__ void initRNG(curandState* s, int walkers, unsigned long seed){
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id >= walkers) return;
    unsigned long composite = seed + (unsigned long)id * 7919u + 1315423911u;
    curand_init(composite, id, 0, &s[id]);
}

// SA kernel with mixed 2-opt and reinsertion (2.5-opt) using distMat
__global__ void sa_mixed_2opt_reinsert_kernel_dist(
    const City* coords, int n,
    int* tours_global,        // walkers * n (in-place)
    int* pos_global,          // walkers * n (pos of city in tour)
    double* costs_global,     // walkers (double)
    const int* knn, int k,    // knn table (n*k)
    const int* distMat,       // n*n TSPLIB integer distances
    curandState* rng,         // walkers
    int walkers,
    int inner, double T0, double alpha, int saIter,
    float p_global, float p_reinsert, int sanity_interval
){
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= walkers) return;

    int* tour = tours_global + tid * n;
    int* pos  = pos_global + tid * n;
    curandState st = rng[tid];

    // Random initial permutation (Fisher-Yates like)
    for (int i = 0; i < n; ++i) tour[i] = i;
    for (int i = 0; i < n; ++i){
        int j = curand(&st) % n;
        int tmp = tour[i]; tour[i] = tour[j]; tour[j] = tmp;
    }
    // init pos[]
    for (int i = 0; i < n; ++i) pos[tour[i]] = i;

    // compute initial cost using distMat
    double cur = 0.0;
    for (int i = 0; i < n; ++i){
        int a = tour[i], b = tour[(i+1)%n];
        cur += dist_lookup_int(distMat, n, a, b);
    }
    double best = cur;
    double T = T0;
    int no_improve = 0;

    for (int it = 0; it < saIter; ++it){
        for (int t = 0; t < inner; ++t){
            double mvchoice = curand_uniform(&st);

            if (mvchoice < p_reinsert){
                // Reinsertion (2.5-opt)
                int posI = curand(&st) % n;
                int cityI = tour[posI];
                int posPrev = (posI - 1 + n) % n;
                int posNext = (posI + 1) % n;
                int cityPrev = tour[posPrev];
                int cityNext = tour[posNext];

                double delta_rem = dist_lookup_int(distMat,n,cityPrev,cityNext)
                                 - dist_lookup_int(distMat,n,cityPrev,cityI)
                                 - dist_lookup_int(distMat,n,cityI,cityNext);

                int cityT;
                if (curand_uniform(&st) < p_global){
                    cityT = curand(&st) % n;
                } else {
                    int pick = curand(&st) % k;
                    cityT = knn[cityI * k + pick];
                }
                int posT = pos[cityT];
                bool insertAfter = (curand_uniform(&st) < 0.5f);
                int posIns = insertAfter ? ((posT + 1) % n) : posT;

                if (posIns == posI || posIns == posPrev) {
                    // no-op
                } else {
                    int cityC = tour[(posIns + n - 1) % n];
                    int cityD = tour[posIns % n];

                    double delta_ins = dist_lookup_int(distMat,n,cityC,cityI)
                                     + dist_lookup_int(distMat,n,cityI,cityD)
                                     - dist_lookup_int(distMat,n,cityC,cityD);
                    double delta = delta_rem + delta_ins;

                    double rv = (double)curand_uniform(&st);
                    if (delta < 0.0 || rv < exp(-delta / max(T, 1e-12))){
                        if (posI < posIns){
                            int tmp = tour[posI];
                            for (int p = posI; p < posIns - 1; ++p){
                                tour[p] = tour[p + 1];
                                pos[tour[p]] = p;
                            }
                            tour[posIns - 1] = tmp;
                            pos[tmp] = posIns - 1;
                        } else {
                            int tmp = tour[posI];
                            for (int p = posI; p > posIns; --p){
                                tour[p] = tour[p - 1];
                                pos[tour[p]] = p;
                            }
                            tour[posIns] = tmp;
                            pos[tmp] = posIns;
                        }
                        cur += delta;
                        if (cur < best){
                            best = cur;
                            no_improve = 0;
                        } else no_improve++;
                    } else no_improve++;
                }
            } else {
                // 2-opt
                int posA = curand(&st) % n;
                int cityA = tour[posA];

                int cityB;
                if (curand_uniform(&st) < p_global){
                    cityB = curand(&st) % n;
                } else {
                    int pick = curand(&st) % k;
                    cityB = knn[cityA * k + pick];
                }
                int posB = pos[cityB];
                if (posB == posA) continue;
                int a = posA, b = posB;
                if (a > b){ int tmp = a; a = b; b = tmp; }
                if (b == a + 1) continue;

                int A = tour[a], B = tour[(a+1)%n], C = tour[b], D = tour[(b+1)%n];
                double delta = (dist_lookup_int(distMat,n,A,C) + dist_lookup_int(distMat,n,B,D))
                             - (dist_lookup_int(distMat,n,A,B) + dist_lookup_int(distMat,n,C,D));
                double rv = (double)curand_uniform(&st);
                if (delta < 0.0 || rv < exp(-delta / max(T, 1e-12))){
                    int L = a + 1, R = b;
                    while (L < R){
                        int cL = tour[L], cR = tour[R];
                        tour[L] = cR; tour[R] = cL;
                        pos[cR] = L; pos[cL] = R;
                        ++L; --R;
                    }
                    cur += delta;
                    if (cur < best){
                        best = cur;
                        no_improve = 0;
                    } else no_improve++;
                } else no_improve++;
            }
        } // inner

        T *= alpha;

        // sanity recompute occasionally
        if ((it % sanity_interval) == 0){
            double check = 0.0;
            for (int i = 0; i < n; ++i){
                int aa = tour[i], bb = tour[(i+1)%n];
                check += dist_lookup_int(distMat,n,aa,bb);
            }
            if (!isfinite(check) || fabs(check - cur) > 1e6){
                cur = check;
                if (!isfinite(cur)) cur = 1e100;
                if (cur < best) best = cur;
            }
        }

        if (no_improve > (sanity_interval * 4)){
            T *= 1.02;
            no_improve = 0;
        }
    } // outer

    rng[tid] = st;
    costs_global[tid] = best;
}

// ---------------- Host helpers ----------------
vector<City> readTSPLIB(const string& path){
    ifstream in(path);
    if(!in.is_open()){ cerr<<"ERROR opening "<<path<<"\n"; return {}; }
    string line; bool start=false; vector<City> coords;
    while(getline(in, line)){
        size_t s = line.find_first_not_of(" \t\r\n");
        if (s == string::npos) continue;
        line.erase(0, s);
        size_t e = line.find_last_not_of(" \t\r\n");
        if (e != string::npos) line.erase(e + 1);
        if (line == "NODE_COORD_SECTION" || line == "NODE_COORD_SECTION\r") { start = true; continue; }
        if (!start) continue;
        if (line == "EOF") break;
        if (line.empty()) continue;
        stringstream ss(line);
        int id; float x,y;
        if (ss >> id >> x >> y) coords.push_back({x,y});
    }
    return coords;
}

// Build TSPLIB integer distance matrix (EUC_2D rounding)
vector<int> build_distMat_tsplib(const vector<City>& coords){
    int n = (int)coords.size();
    vector<int> distMat((size_t)n * n);
    for (int i = 0; i < n; ++i){
        for (int j = 0; j < n; ++j){
            if (i == j) { distMat[i*n + j] = 0; continue; }
            double dx = (double)coords[i].x - (double)coords[j].x;
            double dy = (double)coords[i].y - (double)coords[j].y;
            double d = sqrt(dx*dx + dy*dy);
            distMat[i*n + j] = (int)floor(d + 0.5);
        }
    }
    return distMat;
}

vector<int> build_knn_host(const vector<City>& coords, int k){
    int n = (int)coords.size();
    vector<int> knn(n * k, 0);
    vector<vector<pair<float,int>>> lists(n);
    for (int i = 0; i < n; ++i){
        lists[i].reserve(n-1);
        for (int j = 0; j < n; ++j) if (i != j){
            float dx = coords[i].x - coords[j].x, dy = coords[i].y - coords[j].y;
            lists[i].push_back({dx*dx + dy*dy, j});
        }
        sort(lists[i].begin(), lists[i].end());
        int upto = min(k, (int)lists[i].size());
        for (int t = 0; t < upto; ++t) knn[i*k + t] = lists[i][t].second;
        for (int t = upto; t < k; ++t) knn[i*k + t] = lists[i].back().second;
    }
    return knn;
}

double tour_cost_host_float(const vector<City>& coords, const vector<int>& tour){
    int n = (int)tour.size(); double sum = 0.0;
    for (int i = 0; i < n; ++i){
        int a = tour[i], b = tour[(i+1)%n];
        double dx = coords[a].x - coords[b].x, dy = coords[a].y - coords[b].y;
        sum += sqrt(dx*dx + dy*dy);
    }
    return sum;
}

long long tour_cost_host_tsplib_int(const vector<int>& distMat, int n, const vector<int>& tour){
    long long sum = 0;
    for (int i = 0; i < n; ++i){
        int a = tour[i], b = tour[(i+1)%n];
        sum += distMat[a*n + b];
    }
    return sum;
}

double two_opt_improve(vector<int>& tour, const vector<City>& coords){
    int n = (int)tour.size(); bool improved = true;
    double bestCost = tour_cost_host_float(coords, tour);
    while (improved){
        improved = false;
        for (int i = 0; i < n - 2; ++i){
            for (int j = i + 2; j < n; ++j){
                if (i == 0 && j == n - 1) continue;
                int A = tour[i], B = tour[(i+1)%n];
                int C = tour[j], D = tour[(j+1)%n];
                double oldEdges = hypot(coords[A].x - coords[B].x, coords[A].y - coords[B].y)
                                + hypot(coords[C].x - coords[D].x, coords[C].y - coords[D].y);
                double newEdges = hypot(coords[A].x - coords[C].x, coords[A].y - coords[C].y)
                                + hypot(coords[B].x - coords[D].x, coords[B].y - coords[D].y);
                double delta = newEdges - oldEdges;
                if (delta < -1e-9){
                    reverse(tour.begin() + i + 1, tour.begin() + j + 1);
                    bestCost += delta;
                    improved = true;
                }
            }
            if (improved) break;
        }
    }
    return bestCost;
}

bool is_valid_tour(const vector<int>& tour, int n){
    if ((int)tour.size() != n) return false;
    vector<char> seen(n, 0);
    for (int v : tour){
        if (v < 0 || v >= n) return false;
        if (seen[v]) return false;
        seen[v] = 1;
    }
    return true;
}

// ---------------- main ----------------
int main(int argc, char** argv){
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string fname = "tsplib/tsp/kroA200.tsp";
    if (argc > 1) fname = argv[1];

    auto coordsH = readTSPLIB(fname);
    int n = (int)coordsH.size();
    if (n == 0){ cerr<<"No cities loaded\n"; return 1; }
    cerr<<"Loaded "<<n<<" cities\n";

    // params (tune)
    int walkers = 1024;
    int k = 20;
    int inner = max(32, n / 8);
    int saIter = 250;
    double T0 = 10.0, Tf = 1e-3;
    double alpha = pow(Tf / T0, 1.0 / max(1, saIter - 1));
    int K_refine = 8;
    int gpu_threads = 128;
    float p_global = 0.03f;
    float p_reinsert = 0.45f;
    int sanity_interval = 32;

    cerr << "Parameters: walkers="<<walkers<<" k="<<k<<" inner="<<inner
         <<" saIter="<<saIter<<" p_global="<<p_global<<" p_reinsert="<<p_reinsert<<"\n";

    // build distance matrix (TSPLIB rounding) and k-NN
    cerr<<"Building TSPLIB distance matrix and k-NN...\n";
    auto distMat = build_distMat_tsplib(coordsH);
    auto knn_host = build_knn_host(coordsH, k);
    cerr<<"Finished building distMat and knn\n";

    // device allocations
    City* d_coords = nullptr;
    int *d_dist = nullptr;
    int *d_tours = nullptr, *d_pos = nullptr, *d_knn = nullptr;
    double *d_costs = nullptr;
    curandState* d_rng = nullptr;

    gpuCheck( cudaMalloc((void**)&d_coords, n * sizeof(City)) );
    gpuCheck( cudaMemcpy(d_coords, coordsH.data(), n * sizeof(City), cudaMemcpyHostToDevice) );

    gpuCheck( cudaMalloc((void**)&d_dist, (size_t)n * n * sizeof(int)) );
    gpuCheck( cudaMemcpy(d_dist, distMat.data(), (size_t)n * n * sizeof(int), cudaMemcpyHostToDevice) );

    gpuCheck( cudaMalloc((void**)&d_tours, (size_t)walkers * n * sizeof(int)) );
    gpuCheck( cudaMalloc((void**)&d_pos,   (size_t)walkers * n * sizeof(int)) );
    gpuCheck( cudaMalloc((void**)&d_costs, (size_t)walkers * sizeof(double)) );
    gpuCheck( cudaMalloc((void**)&d_knn,   (size_t)n * k * sizeof(int)) );
    gpuCheck( cudaMalloc((void**)&d_rng,   (size_t)walkers * sizeof(curandState)) );

    gpuCheck( cudaMemcpy(d_knn, knn_host.data(), (size_t)n * k * sizeof(int), cudaMemcpyHostToDevice) );

    // init RNG
    {
        int blocks_init = (walkers + gpu_threads - 1) / gpu_threads;
        initRNG<<<blocks_init, gpu_threads>>>(d_rng, walkers, 123456789u);
        gpuCheck(cudaPeekAtLastError());
        gpuCheck(cudaDeviceSynchronize());
    }

    // run kernel
    int blocks = (walkers + gpu_threads - 1) / gpu_threads;
    cudaEvent_t s,e; gpuCheck(cudaEventCreate(&s)); gpuCheck(cudaEventCreate(&e));
    gpuCheck(cudaEventRecord(s));
    sa_mixed_2opt_reinsert_kernel_dist<<<blocks, gpu_threads>>>(
        d_coords, n, d_tours, d_pos, d_costs, d_knn, k, d_dist, d_rng,
        walkers, inner, T0, alpha, saIter, p_global, p_reinsert, sanity_interval
    );
    gpuCheck(cudaPeekAtLastError());
    gpuCheck(cudaEventRecord(e));
    gpuCheck(cudaEventSynchronize(e));
    float gpuMs = 0.0f; gpuCheck(cudaEventElapsedTime(&gpuMs, s, e));
    cerr<<"GPU phase done, time ms = "<<gpuMs<<"\n";

    // fetch costs
    vector<double> costsH(walkers, 1e300);
    gpuCheck( cudaMemcpy(costsH.data(), d_costs, walkers * sizeof(double), cudaMemcpyDeviceToHost) );

    // pick top-K_refine
    vector<int> idx(walkers);
    iota(idx.begin(), idx.end(), 0);
    int take = min((int)idx.size(), K_refine);
    nth_element(idx.begin(), idx.begin() + take, idx.end(),
                [&](int a, int b){ return costsH[a] < costsH[b]; });
    vector<int> topIdx(idx.begin(), idx.begin() + take);
    sort(topIdx.begin(), topIdx.end(), [&](int a, int b){ return costsH[a] < costsH[b]; });

    cout<<"Top "<<topIdx.size()<<" from GPU (costs):\n";
    for(size_t i=0;i<topIdx.size();++i) cout<<i<<": idx="<<topIdx[i]<<" cost="<<costsH[topIdx[i]]<<"\n";

    // copy top tours to host and validate
    vector<vector<int>> hostTours;
    hostTours.reserve(topIdx.size());
    for (int id : topIdx){
        vector<int> tour(n);
        gpuCheck(cudaMemcpy(tour.data(), d_tours + id * n, n * sizeof(int), cudaMemcpyDeviceToHost));
        if (!is_valid_tour(tour, n)){
            cerr<<"WARNING: invalid tour for walker "<<id<<" — skipping\n";
            continue;
        }
        hostTours.push_back(move(tour));
    }

    // CPU refine in parallel
    cerr<<"Starting CPU refinement on top "<<hostTours.size()<<" tours ...\n";
    vector<double> refinedCosts(hostTours.size(), 1e300);
    vector<thread> threads;
    for(size_t i=0;i<hostTours.size();++i){
        threads.emplace_back([&,i](){
            double before = tour_cost_host_float(coordsH, hostTours[i]);
            double after = two_opt_improve(hostTours[i], coordsH);
            refinedCosts[i] = after;
            cerr<<"Refine["<<i<<"] before="<<before<<" after="<<after<<"\n";
        });
    }
    for(auto &t : threads) if (t.joinable()) t.join();

    // choose best among refined and raw GPU (validate)
    double bestCostFloat = 1e300;
    long long bestCostTSPLIB = (long long)9e18;
    vector<int> bestTour;
    // check refined
    for(size_t i=0;i<hostTours.size();++i){
        if (refinedCosts[i] < bestCostFloat){
            bestCostFloat = refinedCosts[i];
            bestTour = hostTours[i];
        }
    }
    // also check raw GPU best (if valid)
    for(int i=0;i<walkers;i++){
        if (costsH[i] < bestCostFloat){
            vector<int> tour(n);
            gpuCheck(cudaMemcpy(tour.data(), d_tours + i * n, n * sizeof(int), cudaMemcpyDeviceToHost));
            if (is_valid_tour(tour, n)){
                bestCostFloat = costsH[i];
                bestTour = tour;
            }
        }
    }

    // Validate bestTour using TSPLIB distMat (integer)
    if (!bestTour.empty()){
        long long cost_tsplib = tour_cost_host_tsplib_int(distMat, n, bestTour);
        double cost_float_raw = tour_cost_host_float(coordsH, bestTour);
        cout << "\n== Final Validation ==\n";
        cout << "Best host float-cost (sum euclidean): " << cost_float_raw << "\n";
        cout << "Best TSPLIB integer-cost (rounded): " << cost_tsplib << "\n";
        // print first 20 edges for quick inspection
        cout << "First edges (idx : a->b dist):\n";
        for(int i=0;i<min(20,n);++i){
            int a = bestTour[i], b = bestTour[(i+1)%n];
            cout << i << " : " << a << " -> " << b << " " << distMat[a*n + b] << "\n";
        }
        // Compare to known optimum if you want (not hard-coded)
        bestCostTSPLIB = cost_tsplib;
    } else {
        cerr << "No valid best tour found!\n";
    }

    cout << "\n== Results ==\n";
    cout << "GPU phase time (ms): " << gpuMs << "\n";
    if (bestTour.size()){
        cout << "Best TSPLIB integer-cost: " << bestCostTSPLIB << "\n";
    } else {
        cout << "No valid tour to report.\n";
    }

    // cleanup
    cudaFree(d_coords); cudaFree(d_dist);
    cudaFree(d_tours); cudaFree(d_pos); cudaFree(d_costs);
    cudaFree(d_knn); cudaFree(d_rng);
    cudaEventDestroy(s); cudaEventDestroy(e);

    return 0;
}


Writing main_hybrid_2_5opt_dist.cu


In [ ]:
!nvcc -O3 -arch=sm_75 -std=c++17 main_hybrid_2_5opt_dist.cu -o tsp_hybrid_2_5opt_dist

In [ ]:
!./tsp_hybrid_2_5opt_dist tsplib/kroA200.tsp

Loaded 200 cities
Parameters: walkers=1024 k=20 inner=32 saIter=250 p_global=0.03 p_reinsert=0.45
Building TSPLIB distance matrix and k-NN...
Finished building distMat and knn
GPU phase done, time ms = 219.544
Top 8 from GPU (costs):
0: idx=931 cost=21051
1: idx=536 cost=22267
2: idx=60 cost=22276
3: idx=298 cost=22292
4: idx=51 cost=22300
5: idx=567 cost=22377
6: idx=188 cost=22501
7: idx=433 cost=22651
Starting CPU refinement on top 8 tours ...
Refine[1] before=33022.3 after=31508.6
Refine[0] before=34646.2 after=31613.3
Refine[7] before=32792.5 after=31418.2
Refine[Refine[Refine[6] before=33012.1 after=31431.5
3] before=34112.3 after=32336.2
Refine[4] before=33375.7 after=30811.3
Refine[5] before=35148.6 after=33617.1
Refine[2] before=35058.6 after=31508.7

== Final Validation ==
Best host float-cost (sum euclidean): 34646.2
Best TSPLIB integer-cost (rounded): 34645
First edges (idx : a->b dist):
0 : 58 -> 40 285
1 : 40 -> 88 213
2 : 88 -> 163 367
3 : 163 -> 139 80
4 : 139 -> 153 10